# A3 Step 4 -- recall@k probe: 39 gold TEST pages, dense vs. reranked

First honest retrieval number for A3 (`plan_a3.md` Step 4) -- measured **before** any
`tasks.jsonl` question has been written, so it can't be tuned to flatter the task suite
later. One query per gold page (the page's own first line), dense search against the
real A2 index (`mathscholar-index` dataset), then the same candidates reranked with
`BAAI/bge-reranker-v2-m3`. Reports recall@1/5/10 both ways.

Self-contained: doesn't clone the repo, mounts only the `mathscholar-index` dataset, and
the 39 gold labels are embedded directly in the code cell below.

In [1]:
import json
import time
import zipfile
from pathlib import Path

import numpy as np
import torch

# --- The 39 gold TEST-page labels, embedded directly (114 KB) -- no second dataset needed.
LABELS_JSON = "[{\"page_id\": \"as_p0243\", \"text\": \"EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.243)\\nTable 5.1  SINE, COSINE AND EXPONENTIAL INTEGRALS\\nColumns: x | \\\\mathrm{Si}(x) | \\\\mathrm{Ci}(x) | xe^{-x}\\\\mathrm{Ei}(x) | xe^{x}E_1(x)\\nSample rows:\\n7.0  1.45459\\\\,66142  0.07669\\\\,52785  1.22240\\\\,8053  0.88648\\\\,7675\\n7.5  1.51068\\\\,15309  0.11563\\\\,32032  1.20042\\\\,1500  0.89268\\\\,7854\\n8.0  1.57418\\\\,68217  0.12244\\\\,38825  1.18184\\\\,7987  0.89823\\\\,7113\\n9.0  1.66504\\\\,00758  0.05534\\\\,75313  1.15275\\\\,9209  0.90775\\\\,7602\\n10.0 1.65834\\\\,75942 -0.04545\\\\,64330  1.13147\\\\,0205  0.91563\\\\,3339\\nTable 5.2  SINE, COSINE AND EXPONENTIAL INTEGRALS FOR LARGE ARGUMENTS\\nColumns: x^{-1} | x f(x) | x^2 g(x) | xe^{-x}\\\\mathrm{Ei}(x) | xe^{x}E_1(x) | x\\nSample rows:\\n0.100  0.98191\\\\,0357  0.94885\\\\,39  1.13147\\\\,021  0.91563\\\\,33394  10\\n0.050  0.99514\\\\,0052  0.98568\\\\,24  1.05595\\\\,591  0.95437\\\\,09099  20\\n0.000  1.00000\\\\,0000  1.00000\\\\,00  1.00000\\\\,000  1.00000\\\\,00000  \\\\infty\\nDefining relations (foot of page):\\n\\\\mathrm{Si}(x)=\\\\tfrac{\\\\pi}{2}-f(x)\\\\cos x-g(x)\\\\sin x, \\\\quad \\\\mathrm{Ci}(x)=f(x)\\\\sin x-g(x)\\\\cos x, \\\\quad \\\\tfrac{\\\\pi}{2}=1.57079\\\\,63268.\\n\\\\langle x\\\\rangle = nearest integer to x.\", \"chapter_id\": \"ch05_expint\", \"printed_page\": 243, \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0243.png\", \"note\": \"Dense numeric-table page (Ch.5, Tables 5.1-5.2). Hard OCR case included on purpose. Gold = structural headers + defining formulas + sampled rows (not every cell).\"}, {\"page_id\": \"as_p0255\", \"text\": \"6. Gamma Function and Related Functions. Mathematical Properties.\\n6.1. Gamma (Factorial) Function.\\nEuler's Integral\\n6.1.1  \\\\Gamma(z)=\\\\int_0^\\\\infty t^{z-1}e^{-t}\\\\,dt \\\\quad (\\\\Re z>0)\\n       =k^z\\\\int_0^\\\\infty t^{z-1}e^{-kt}\\\\,dt \\\\quad (\\\\Re z>0,\\\\ \\\\Re k>0)\\nEuler's Formula\\n6.1.2  \\\\Gamma(z)=\\\\lim_{n\\\\to\\\\infty}\\\\frac{n!\\\\,n^{z}}{z(z+1)\\\\cdots(z+n)} \\\\quad (z\\\\neq 0,-1,-2,\\\\dots)\\nEuler's Infinite Product\\n6.1.3  \\\\frac{1}{\\\\Gamma(z)}=z e^{\\\\gamma z}\\\\prod_{n=1}^{\\\\infty}\\\\left[\\\\left(1+\\\\frac{z}{n}\\\\right)e^{-z/n}\\\\right] \\\\quad (|z|<\\\\infty)\\n       \\\\gamma=\\\\lim_{m\\\\to\\\\infty}\\\\left[1+\\\\frac12+\\\\frac13+\\\\frac14+\\\\cdots+\\\\frac1m-\\\\ln m\\\\right]=.57721\\\\,56649\\\\dots\\n\\\\gamma is known as Euler's constant and is given to 25 decimal places in chapter 1. \\\\Gamma(z) is single valued and analytic over the entire complex plane, save for the points z=-n\\\\ (n=0,1,2,\\\\dots) where it possesses simple poles with residue (-1)^n/n!. Its reciprocal 1/\\\\Gamma(z) is an entire function possessing simple zeros at the points z=-n\\\\ (n=0,1,2,\\\\dots).\\nHankel's Contour Integral\\n6.1.4  \\\\frac{1}{\\\\Gamma(z)}=\\\\frac{i}{2\\\\pi}\\\\int_C (-t)^{-z}e^{-t}\\\\,dt \\\\quad (|z|<\\\\infty)\\nThe path of integration C starts at +\\\\infty on the real axis, circles the origin in the counterclockwise direction and returns to the starting point.\\nFactorial and \\\\Pi Notations\\n6.1.5  \\\\Pi(z)=z!=\\\\Gamma(z+1)\\nInteger Values\\n6.1.6  \\\\Gamma(n+1)=1\\\\cdot 2\\\\cdot 3\\\\cdots(n-1)n=n!\\n6.1.7  \\\\lim_{z\\\\to n}\\\\frac{1}{\\\\Gamma(-z)}=0=\\\\frac{1}{(-n-1)!} \\\\quad (n=0,1,2,\\\\dots)\\nFractional Values\\n6.1.8  \\\\Gamma(\\\\tfrac12)=2\\\\int_0^\\\\infty e^{-t^2}\\\\,dt=\\\\pi^{1/2}=1.77245\\\\,38509\\\\dots=(-\\\\tfrac12)!\\n6.1.9  \\\\Gamma(\\\\tfrac32)=\\\\tfrac12\\\\pi^{1/2}=.88622\\\\,69254\\\\dots=(\\\\tfrac12)!\\n6.1.10 \\\\Gamma(n+\\\\tfrac14)=\\\\frac{1\\\\cdot 5\\\\cdot 9\\\\cdot 13\\\\cdots(4n-3)}{4^n}\\\\Gamma(\\\\tfrac14),\\\\quad \\\\Gamma(\\\\tfrac14)=3.62560\\\\,99082\\\\dots\\n6.1.11 \\\\Gamma(n+\\\\tfrac13)=\\\\frac{1\\\\cdot 4\\\\cdot 7\\\\cdot 10\\\\cdots(3n-2)}{3^n}\\\\Gamma(\\\\tfrac13),\\\\quad \\\\Gamma(\\\\tfrac13)=2.67893\\\\,85347\\\\dots\\n6.1.12 \\\\Gamma(n+\\\\tfrac12)=\\\\frac{1\\\\cdot 3\\\\cdot 5\\\\cdot 7\\\\cdots(2n-1)}{2^n}\\\\Gamma(\\\\tfrac12)\\n6.1.13 \\\\Gamma(n+\\\\tfrac23)=\\\\frac{2\\\\cdot 5\\\\cdot 8\\\\cdot 11\\\\cdots(3n-1)}{3^n}\\\\Gamma(\\\\tfrac23),\\\\quad \\\\Gamma(\\\\tfrac23)=1.35411\\\\,79394\\\\dots\\n6.1.14 \\\\Gamma(n+\\\\tfrac34)=\\\\frac{3\\\\cdot 7\\\\cdot 11\\\\cdot 15\\\\cdots(4n-1)}{4^n}\\\\Gamma(\\\\tfrac34),\\\\quad \\\\Gamma(\\\\tfrac34)=1.22541\\\\,67024\\\\dots\\nFigure 6.1. Gamma function. ---, y=\\\\Gamma(x); ----, y=1/\\\\Gamma(x).\", \"chapter_id\": \"ch06_gamma\", \"printed_page\": 255, \"region_type\": \"prose+formula\", \"image\": \"grading_kit/heldout_pages/as_p0255.png\", \"note\": \"Clean prose+formula page: start of Ch.6 Gamma function, formulas 6.1.1-6.1.14. Includes Gamma(1/2)=pi^{1/2} (6.1.8).\"}, {\"page_id\": \"as_p0360\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.360)\\nLimiting Forms for Small Arguments. When \\\\nu is fixed and z\\\\to 0\\n9.1.7  J_\\\\nu(z)\\\\sim(\\\\tfrac12 z)^\\\\nu/\\\\Gamma(\\\\nu+1) \\\\quad (\\\\nu\\\\neq -1,-2,-3,\\\\dots)\\n9.1.8  Y_0(z)\\\\sim -iH_0^{(1)}(z)\\\\sim iH_0^{(2)}(z)\\\\sim(2/\\\\pi)\\\\ln z\\n9.1.9  Y_\\\\nu(z)\\\\sim -iH_\\\\nu^{(1)}(z)\\\\sim iH_\\\\nu^{(2)}(z)\\\\sim -(1/\\\\pi)\\\\Gamma(\\\\nu)(\\\\tfrac12 z)^{-\\\\nu} \\\\quad (\\\\Re \\\\nu>0)\\nAscending Series\\n9.1.10 J_\\\\nu(z)=(\\\\tfrac12 z)^\\\\nu\\\\sum_{k=0}^{\\\\infty}\\\\frac{(-\\\\tfrac14 z^2)^k}{k!\\\\,\\\\Gamma(\\\\nu+k+1)}\\n9.1.11 Y_n(z)=-\\\\frac{(\\\\tfrac12 z)^{-n}}{\\\\pi}\\\\sum_{k=0}^{n-1}\\\\frac{(n-k-1)!}{k!}(\\\\tfrac14 z^2)^k+\\\\frac{2}{\\\\pi}\\\\ln(\\\\tfrac12 z)J_n(z)-\\\\frac{(\\\\tfrac12 z)^n}{\\\\pi}\\\\sum_{k=0}^{\\\\infty}\\\\{\\\\psi(k+1)+\\\\psi(n+k+1)\\\\}\\\\frac{(-\\\\tfrac14 z^2)^k}{k!\\\\,(n+k)!}\\nwhere \\\\psi(n) is given by 6.3.2.\\n9.1.12 J_0(z)=1-\\\\frac{\\\\tfrac14 z^2}{(1!)^2}+\\\\frac{(\\\\tfrac14 z^2)^2}{(2!)^2}-\\\\frac{(\\\\tfrac14 z^2)^3}{(3!)^2}+\\\\cdots\\n9.1.13 Y_0(z)=\\\\frac{2}{\\\\pi}\\\\{\\\\ln(\\\\tfrac12 z)+\\\\gamma\\\\}J_0(z)+\\\\frac{2}{\\\\pi}\\\\{\\\\frac{\\\\tfrac14 z^2}{(1!)^2}-(1+\\\\tfrac12)\\\\frac{(\\\\tfrac14 z^2)^2}{(2!)^2}+(1+\\\\tfrac12+\\\\tfrac13)\\\\frac{(\\\\tfrac14 z^2)^3}{(3!)^2}-\\\\cdots\\\\}\\n9.1.15 W\\\\{J_\\\\nu(z),J_{-\\\\nu}(z)\\\\}=J_{\\\\nu+1}(z)J_{-\\\\nu}(z)+J_\\\\nu(z)J_{-(\\\\nu+1)}(z)=-2\\\\sin(\\\\nu\\\\pi)/(\\\\pi z)\\n9.1.16 W\\\\{J_\\\\nu(z),Y_\\\\nu(z)\\\\}=J_{\\\\nu+1}(z)Y_\\\\nu(z)-J_\\\\nu(z)Y_{\\\\nu+1}(z)=2/(\\\\pi z)\\nIntegral Representations\\n9.1.18 J_0(z)=\\\\frac{1}{\\\\pi}\\\\int_0^\\\\pi \\\\cos(z\\\\sin\\\\theta)\\\\,d\\\\theta=\\\\frac{1}{\\\\pi}\\\\int_0^\\\\pi \\\\cos(z\\\\cos\\\\theta)\\\\,d\\\\theta\\n9.1.20 J_\\\\nu(z)=\\\\frac{(\\\\tfrac12 z)^\\\\nu}{\\\\pi^{1/2}\\\\Gamma(\\\\nu+\\\\tfrac12)}\\\\int_0^\\\\pi \\\\cos(z\\\\cos\\\\theta)\\\\sin^{2\\\\nu}\\\\theta\\\\,d\\\\theta \\\\quad (\\\\Re\\\\nu>-\\\\tfrac12)\\n9.1.21 J_n(z)=\\\\frac{1}{\\\\pi}\\\\int_0^\\\\pi \\\\cos(z\\\\sin\\\\theta-n\\\\theta)\\\\,d\\\\theta\\n9.1.23 J_0(x)=\\\\frac{2}{\\\\pi}\\\\int_0^\\\\infty \\\\sin(x\\\\cosh t)\\\\,dt \\\\quad (x>0)\\n(This page also carries 9.1.14, 9.1.17, 9.1.19, 9.1.22, 9.1.24, 9.1.25, 9.1.26 in the same two-column dense-formula layout.)\", \"chapter_id\": \"ch09_bessel\", \"printed_page\": 360, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0360.png\", \"note\": \"Dense two-column formula page: Ch.9 Bessel functions, 9.1.7-9.1.26, incl. the J_nu ascending series (9.1.10) and J_0 series (9.1.12).\"}, {\"page_id\": \"as_p0229\", \"text\": \"EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.229)\\nFIGURE 5.3. y=\\\\alpha_n(x), n=0(1)6\\nFIGURE 5.4. y=\\\\beta_n(x), n=0,1,2,5,10,15\\nSeries Expansions\\n5.1.10  \\\\mathrm{Ei}(x)=\\\\gamma+\\\\ln x+\\\\sum_{n=1}^{\\\\infty}\\\\frac{x^n}{nn!} \\\\quad (x>0)\\n5.1.11  E_1(z)=-\\\\gamma-\\\\ln z-\\\\sum_{n=1}^{\\\\infty}\\\\frac{(-1)^n z^n}{nn!} \\\\quad (|\\\\arg z|<\\\\pi)\\n5.1.12  E_n(z)=\\\\frac{(-z)^{n-1}}{(n-1)!}[-\\\\ln z+\\\\psi(n)]-\\\\sum_{m=0,m\\\\neq n-1}^{\\\\infty}\\\\frac{(-z)^m}{(m-n+1)m!} \\\\quad (|\\\\arg z|<\\\\pi)\\n\\\\psi(1)=-\\\\gamma,\\\\ \\\\psi(n)=-\\\\gamma+\\\\sum_{m=1}^{n-1}\\\\frac1m \\\\quad (n>1)\\n\\\\gamma=.57721\\\\,56649\\\\dots is Euler's constant.\\nSymmetry Relation\\n5.1.13  E_n(\\\\bar z)=\\\\overline{E_n(z)}\\nRecurrence Relations\\n5.1.14  E_{n+1}(z)=\\\\frac1n[e^{-z}-zE_n(z)] \\\\quad (n=1,2,3,\\\\dots)\\n5.1.15  z\\\\alpha_n(z)=e^{-z}+n\\\\alpha_{n-1}(z) \\\\quad (n=1,2,3,\\\\dots)\\n5.1.16  z\\\\beta_n(z)=(-1)^n e^z-e^{-z}+n\\\\beta_{n-1}(z) \\\\quad (n=1,2,3,\\\\dots)\\nInequalities [5.8], [5.4]\\n5.1.17  \\\\frac{n-1}{n}E_n(x)<E_{n+1}(x)<E_n(x) \\\\quad (x>0;n=1,2,3,\\\\dots)\\n5.1.18  E_n^2(x)<E_{n-1}(x)E_{n+1}(x) \\\\quad (x>0;n=1,2,3,\\\\dots)\\n5.1.19  \\\\frac{1}{x+n}<e^xE_n(x)\\\\le\\\\frac{1}{x+n-1} \\\\quad (x>0;n=1,2,3,\\\\dots)\\n5.1.20  \\\\tfrac12\\\\ln(1+\\\\tfrac2x)<e^xE_1(x)<\\\\ln(1+\\\\tfrac1x) \\\\quad (x>0)\\n5.1.21  \\\\frac{d}{dx}\\\\left[\\\\frac{E_n(x)}{E_{n-1}(x)}\\\\right]>0 \\\\quad (x>0;n=1,2,3,\\\\dots)\\nContinued Fraction\\n5.1.22  E_n(z)=e^{-z}\\\\left(\\\\cfrac{1}{z+}\\\\cfrac{n}{1+}\\\\cfrac{1}{z+}\\\\cfrac{n+1}{1+}\\\\cfrac{2}{z+}\\\\cdots\\\\right) \\\\quad (|\\\\arg z|<\\\\pi)\\nSpecial Values\\n5.1.23  E_n(0)=\\\\frac{1}{n-1} \\\\quad (n>1)\\n5.1.24  E_0(z)=\\\\frac{e^{-z}}{z}\\n5.1.25  \\\\alpha_0(z)=\\\\frac{e^{-z}}{z},\\\\ \\\\beta_0(z)=\\\\frac2z\\\\sinh z\", \"chapter_id\": \"ch05_expint\", \"printed_page\": 229, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0229.png\", \"note\": \"Two-column formula page (Ch.5 exponential integral), 5.1.10-5.1.25, plus two auxiliary function plots (Figs 5.3-5.4, not individually transcribed -- captions only).\"}, {\"page_id\": \"as_p0230\", \"text\": \"EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.230)\\nDerivatives\\n5.1.26  \\\\frac{dE_n(z)}{dz}=-E_{n-1}(z) \\\\quad (n=1,2,3,\\\\dots)\\n5.1.27  \\\\frac{d^n}{dz^n}[e^zE_1(z)]=\\\\frac{d^{n-1}}{dz^{n-1}}[e^zE_1(z)]+\\\\frac{(-1)^n(n-1)!}{z^n} \\\\quad (n=1,2,3,\\\\dots)\\nDefinite and Indefinite Integrals\\n(For more extensive tables of integrals see [5.3], [5.6], [5.11], [5.12], [5.13]. For integrals involving E_n(x) see [5.9].)\\n5.1.28  \\\\int_0^\\\\infty \\\\frac{e^{-at}}{b+t}dt=e^{ab}E_1(ab)\\n5.1.29  \\\\int_0^\\\\infty \\\\frac{e^{iat}}{b+t}dt=e^{-iab}E_1(-iab) \\\\quad (a>0,b>0)\\n5.1.30  \\\\int_0^\\\\infty \\\\frac{t-ib}{t^2+b^2}e^{iat}dt=e^{ab}E_1(ab) \\\\quad (a>0,b>0)\\n5.1.31  \\\\int_0^\\\\infty \\\\frac{t+ib}{t^2+b^2}e^{iat}dt=e^{-ab}(-\\\\mathrm{Ei}(ab)+i\\\\pi) \\\\quad (a>0,b>0)\\n5.1.32  \\\\int_0^\\\\infty \\\\frac{e^{-at}-e^{-bt}}{t}dt=\\\\ln\\\\frac{b}{a}\\n5.1.33  \\\\int_0^\\\\infty E_1^2(t)dt=2\\\\ln 2\\n5.1.34  \\\\int_0^\\\\infty e^{-at}E_n(t)dt=\\\\frac{(-1)^{n-1}}{a^n}\\\\left[\\\\ln(1+a)+\\\\sum_{k=1}^{n-1}\\\\frac{(-1)^ka^k}{k}\\\\right] \\\\quad (a>-1)\\n5.1.35  \\\\int_0^1 \\\\frac{e^{at}\\\\sin bt}{t}dt=\\\\pi-\\\\arctan\\\\frac{b}{a}+\\\\mathscr{I}E_1(-a+ib) \\\\quad (a>0,b>0)\\n5.1.36  \\\\int_0^1 \\\\frac{e^{-at}\\\\sin bt}{t}dt=\\\\arctan\\\\frac{b}{a}+\\\\mathscr{I}E_1(a+ib) \\\\quad (a>0,b\\\\ \\\\mathrm{real})\\n5.1.37  \\\\int_0^1 \\\\frac{e^{at}(1-\\\\cos bt)}{t}dt=\\\\tfrac12\\\\ln(1+\\\\tfrac{b^2}{a^2})+\\\\mathrm{Ei}(a)+\\\\mathscr{R}E_1(-a+ib) \\\\quad (a>0,b\\\\ \\\\mathrm{real})\\n5.1.38  \\\\int_0^1 \\\\frac{e^{-at}(1-\\\\cos bt)}{t}dt=\\\\tfrac12\\\\ln(1+\\\\tfrac{b^2}{a^2})-E_1(a)+\\\\mathscr{R}E_1(a+ib) \\\\quad (a>0,b\\\\ \\\\mathrm{real})\\n5.1.39  \\\\int_0^z \\\\frac{1-e^{-t}}{t}dt=E_1(z)+\\\\ln z+\\\\gamma\\n5.1.40  \\\\int_0^x \\\\frac{e^t-1}{t}dt=\\\\mathrm{Ei}(x)-\\\\ln x-\\\\gamma \\\\quad (x>0)\\n5.1.41  \\\\int \\\\frac{e^{ix}}{a^2+x^2}dx=\\\\frac{i}{2a}[e^{-a}E_1(-a-ix)-e^aE_1(a-ix)]+\\\\mathrm{const.}\\n5.1.42  \\\\int \\\\frac{xe^{ix}}{a^2+x^2}dx=-\\\\frac12[e^{-a}E_1(-a-ix)+e^aE_1(a-ix)]+\\\\mathrm{const.}\\n5.1.43  \\\\int \\\\frac{e^x}{a^2+x^2}dx=-\\\\frac1a\\\\mathscr{I}(e^{ia}E_1(-x+ia))+\\\\mathrm{const.} \\\\quad (a>0)\\n5.1.44  \\\\int \\\\frac{xe^x}{a^2+x^2}dx=-\\\\mathscr{R}(e^{ia}E_1(-x+ia))+\\\\mathrm{const.} \\\\quad (a>0)\\nRelation to Incomplete Gamma Function (see 6.5)\\n5.1.45  E_n(z)=z^{n-1}\\\\Gamma(1-n,z)\\n5.1.46  \\\\alpha_n(z)=z^{-n-1}\\\\Gamma(n+1,z)\\n5.1.47  \\\\beta_n(z)=z^{-n-1}[\\\\Gamma(n+1,-z)-\\\\Gamma(n+1,z)]\\nRelation to Spherical Bessel Functions (see 10.2)\\n5.1.48  \\\\alpha_0(z)=\\\\sqrt{\\\\frac{2}{\\\\pi z}}K_{1/2}(z),\\\\ \\\\beta_0(z)=\\\\sqrt{\\\\frac{2\\\\pi}{z}}I_{1/2}(z)\\n5.1.49  \\\\alpha_1(z)=\\\\sqrt{\\\\frac{2}{\\\\pi z}}K_{3/2}(z),\\\\ \\\\beta_1(z)=-\\\\sqrt{\\\\frac{2\\\\pi}{z}}I_{3/2}(z)\", \"chapter_id\": \"ch05_expint\", \"printed_page\": 230, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0230.png\", \"note\": \"Dense two-column formula/integral-table page (Ch.5), 5.1.26-5.1.49, transcribed in full (each formula individually citable, not a sampled numeric table).\"}, {\"page_id\": \"as_p0232\", \"text\": \"EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.232)\\n5.2.5  \\\\mathrm{si}(z)=\\\\mathrm{Si}(z)-\\\\frac{\\\\pi}{2}\\nAuxiliary Functions\\n5.2.6  f(z)=\\\\mathrm{Ci}(z)\\\\sin z-\\\\mathrm{si}(z)\\\\cos z\\n5.2.7  g(z)=-\\\\mathrm{Ci}(z)\\\\cos z-\\\\mathrm{si}(z)\\\\sin z\\nSine and Cosine Integrals in Terms of Auxiliary Functions\\n5.2.8  \\\\mathrm{Si}(z)=\\\\frac{\\\\pi}{2}-f(z)\\\\cos z-g(z)\\\\sin z\\n5.2.9  \\\\mathrm{Ci}(z)=f(z)\\\\sin z-g(z)\\\\cos z\\nIntegral Representations\\n5.2.10  \\\\mathrm{si}(z)=-\\\\int_0^{\\\\pi/2} e^{-z\\\\cos t}\\\\cos(z\\\\sin t)dt\\n5.2.11  \\\\mathrm{Ci}(z)+E_1(z)=\\\\int_0^{\\\\pi/2} e^{-z\\\\cos t}\\\\sin(z\\\\sin t)dt\\n5.2.12  f(z)=\\\\int_0^\\\\infty \\\\frac{\\\\sin t}{t+z}dt=\\\\int_0^\\\\infty \\\\frac{e^{-zt}}{t^2+1}dt \\\\quad (\\\\Re z>0)\\n5.2.13  g(z)=\\\\int_0^\\\\infty \\\\frac{\\\\cos t}{t+z}dt=\\\\int_0^\\\\infty \\\\frac{te^{-zt}}{t^2+1}dt \\\\quad (\\\\Re z>0)\\nFIGURE 5.6. y=Si(x) and y=Ci(x)\\nSeries Expansions\\n5.2.14  \\\\mathrm{Si}(z)=\\\\sum_{n=0}^{\\\\infty} \\\\frac{(-1)^nz^{2n+1}}{(2n+1)(2n+1)!}\\n5.2.15  \\\\mathrm{Si}(z)=\\\\pi\\\\sum_{n=0}^{\\\\infty} J_{n+\\\\frac12}^2\\\\left(\\\\frac{z}{2}\\\\right)\\n5.2.16  \\\\mathrm{Ci}(z)=\\\\gamma+\\\\ln z+\\\\sum_{n=1}^{\\\\infty} \\\\frac{(-1)^nz^{2n}}{2n(2n)!}\\n5.2.17  \\\\mathrm{Shi}(z)=\\\\sum_{n=0}^{\\\\infty} \\\\frac{z^{2n+1}}{(2n+1)(2n+1)!}\\n5.2.18  \\\\mathrm{Chi}(z)=\\\\gamma+\\\\ln z+\\\\sum_{n=1}^{\\\\infty} \\\\frac{z^{2n}}{2n(2n)!}\\nSymmetry Relations\\n5.2.19  \\\\mathrm{Si}(-z)=-\\\\mathrm{Si}(z),\\\\ \\\\mathrm{Si}(\\\\bar z)=\\\\overline{\\\\mathrm{Si}(z)}\\n5.2.20  \\\\mathrm{Ci}(-z)=\\\\mathrm{Ci}(z)-i\\\\pi \\\\quad (0<\\\\arg z<\\\\pi)\\n        \\\\mathrm{Ci}(\\\\bar z)=\\\\overline{\\\\mathrm{Ci}(z)}\\nRelation to Exponential Integral\\n5.2.21  \\\\mathrm{Si}(z)=\\\\frac{1}{2i}[E_1(iz)-E_1(-iz)]+\\\\frac{\\\\pi}{2} \\\\quad (|\\\\arg z|<\\\\tfrac{\\\\pi}{2})\\n5.2.22  \\\\mathrm{Si}(ix)=\\\\frac{i}{2}[\\\\mathrm{Ei}(x)+E_1(x)] \\\\quad (x>0)\\n5.2.23  \\\\mathrm{Ci}(z)=-\\\\frac12[E_1(iz)+E_1(-iz)] \\\\quad (|\\\\arg z|<\\\\tfrac{\\\\pi}{2})\\n5.2.24  \\\\mathrm{Ci}(ix)=\\\\frac12[\\\\mathrm{Ei}(x)-E_1(x)]+i\\\\frac{\\\\pi}{2} \\\\quad (x>0)\\nValue at Infinity\\n5.2.25  \\\\lim_{x\\\\to\\\\infty}\\\\mathrm{Si}(x)=\\\\frac{\\\\pi}{2}\\nIntegrals\\n(For more extensive tables of integrals see [5.3], [5.6], [5.11], [5.12], [5.13].)\\n5.2.26  \\\\int_z^\\\\infty \\\\frac{\\\\sin t}{t}dt=-\\\\mathrm{si}(z) \\\\quad (|\\\\arg z|<\\\\pi)\\n5.2.27  \\\\int_z^\\\\infty \\\\frac{\\\\cos t}{t}dt=-\\\\mathrm{Ci}(z) \\\\quad (|\\\\arg z|<\\\\pi)\\n5.2.28  \\\\int_0^\\\\infty e^{-at}\\\\mathrm{Ci}(t)dt=\\\\frac{1}{2a}\\\\ln(1+a^2) \\\\quad (\\\\Re a>0)\\n5.2.29  \\\\int_0^\\\\infty e^{-at}\\\\mathrm{si}(t)dt=-\\\\frac1a\\\\arctan a \\\\quad (\\\\Re a>0)\\n5.2.30  \\\\int_0^\\\\infty \\\\cos t\\\\,\\\\mathrm{Ci}(t)dt=\\\\int_0^\\\\infty \\\\sin t\\\\,\\\\mathrm{si}(t)dt=-\\\\frac{\\\\pi}{4}\", \"chapter_id\": \"ch05_expint\", \"printed_page\": 232, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0232.png\", \"note\": \"Dense two-column formula page (Ch.5 sine/cosine integrals), 5.2.5-5.2.30, plus Figure 5.6 (Si/Ci plot, caption only). Transcribed in full.\"}, {\"page_id\": \"as_p0234\", \"text\": \"EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.234)\\nColumns: n | E_n(1.275)\\n1  .1408099   6  .0430168\\n2  .0998984   7  .0374307\\n3  .0760303   8  .0331009\\n4  .0608307   9  .0296534\\n5  .0504679   10 .0268469\\nInterpolating directly in Table 5.4 for n=10 we get E_10(1.275)=.0268470 as a check.\\nExample 5. Compute E_n(x), n=1(1)N, to 5S for x=10, N=10.\\nIf, as in this example, x is appreciably larger than five and N\\\\le x, then the recurrence relation 5.1.14 may be safely used in decreasing order of n ([5.5]). From Table 5.5 for x^{-1}=.1 we get (x+10)e^xE_{10}(x)=1.02436 so that E_{10}(10)=2.32529\\\\times10^{-6}. Using this as the initial value we obtain column (2).\\nColumns: n | 10^5 E_n(10) (1) | 10^5 E_n(10) (2)\\n1  .41570   .41570\\n2  .38300   .38302\\n3  .35500   .35488\\n4  .33000   .33041\\n5  .31000   .30898\\n6  .28800   .29005\\n7  .27667   .27325\\n8  .25333   .25822\\n9  .25084   .24472\\n10 .22573   .23253\\n(Digits shown underlined in the source are flagged there as in error.)\\nFrom Table 5.2 we get xe^xE_1(x)=.915633 so that E_1(10)=4.15697\\\\times10^{-6} as a check. Forward recurrence starting with E_1(10)=4.1570\\\\times10^{-6} yields the values in column (1). The underlined figures are in error.\\nExample 6. Compute E_n(x), n=1(1)N, to 5S for x=12.3, N=20.\\nIf N is appreciably larger than x, and x appreciably larger than five, then the recurrence relation 5.1.14 should be used in the backward direction to generate E_n(x) for n<n_0, and in the forward direction to generate E_n(x) for n>n_0, where n_0=\\\\langle x\\\\rangle.\\nFrom 5.1.52, with n_0=12, x=12.3, we have\\nE_{n_0}(x)=\\\\frac{e^{-12.3}}{24.3}(1+.02032-.00043-.00001)=1.91038\\\\times10^{-7}.\\nUsing the recurrence relation 5.1.14, as indicated, we get\\nColumns: n | 10^6 E_n(12.3) | 10^6 E_n(12.3) | n\\n12  .191038   .191038   12\\n11  .199213   .183498   13\\n10  .208098   .176516   14\\n9   .217793   .170042   15\\n8   .228406   .164015   16\\n7   .240073   .158397   17\\n6   .252951   .153144   18\\n5   .267234   .148226   19\\n4   .283155   .143608   20\\n3   .300998\\n2   .321117\\n1   .343953\\nFrom Tables 5.2 and 5.5 we find E_1(12.3)=.343953\\\\times10^{-6}, E_{20}(12.3)=.143609\\\\times10^{-6} as a check.\\nExample 7. Compute \\\\alpha_n(2) to 6S for n=1(1)5.\\nThe recurrence formula 5.1.15 can be used for all x>0 in increasing order of n without loss of accuracy. From 5.1.25 we have \\\\alpha_0(2)=\\\\frac12e^{-2}=.0676676, so we get\\nColumns: n | \\\\alpha_n(2)\\n0  .0676676\\n1  .101501\\n2  .169169\\n3  .321421\\n4  .710510\\n5  1.84394\\nIndependent calculation with 5.1.8 yields the same result for \\\\alpha_5(2).\\nThe functions \\\\alpha_0(x) and \\\\alpha_1(x) can be obtained from Table 10.8 using 5.1.48, 5.1.49.\\nExample 8. Compute \\\\beta_n(x), n=0(1)N to 6S for x=1, N=5.\\nUse the recurrence relation 5.1.16 in increasing order of n if\\nx>.368N+.184\\\\ln N+.821\\nand in decreasing order of n otherwise [5.5].\\nFrom 5.1.9 with n=5 we get \\\\beta_5(1)=-.324297 correctly rounded to 6D. Using the recurrence formula 5.1.16 in decreasing order of n and carrying 9D we get the values in column (2).\\nColumns: n | \\\\beta_n(1) (1) | \\\\beta_n(1) (2)\\n0  2.35040\\\\,2   2.35040\\\\,2389\\n1  -.73575\\\\,9269   -.73575\\\\,8880\\n2  .87888\\\\,3849   .87888\\\\,4629\\n3  -.44950\\\\,9722   -.44950\\\\,7383\\n4  .55236\\\\,3499   .55237\\\\,2854\\n5  -.32434\\\\,3774   -.32429\\\\,7\\nUsing forward recurrence instead, starting with [continues on next page].\", \"chapter_id\": \"ch05_expint\", \"printed_page\": 234, \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0234.png\", \"note\": \"Worked-examples page (Ch.5), not a formula listing: prose explanation (Examples 5-8) with several short embedded numeric tables, transcribed in full (not sampled -- each table is the pedagogical point of its example, unlike a large reference table).\"}, {\"page_id\": \"as_p0242\", \"text\": \"EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.242)\\nTable 5.1  SINE, COSINE AND EXPONENTIAL INTEGRALS\\nColumns: x | Si(x) | Ci(x) | xe^{-x}\\\\mathrm{Ei}(x) | xe^xE_1(x)\\nSample rows:\\n2.0  1.60541\\\\,29768  0.42298\\\\,08288  1.34096\\\\,5420  0.72265\\\\,7234\\n4.0  1.75820\\\\,31389  -0.14098\\\\,16979  1.43820\\\\,8032  0.82538\\\\,2600\\n5.5  1.46872\\\\,40727  -0.14205\\\\,29476  1.31414\\\\,3566  0.86256\\\\,1885\\n7.0  1.45459\\\\,66142  0.07669\\\\,52785  1.22240\\\\,8053  0.88648\\\\,7675\\n(Table continues on p.243 from x=7.0; a bracketed interpolation-difference scale factor is printed under each column at the foot of the page, not transcribed here.)\", \"chapter_id\": \"ch05_expint\", \"printed_page\": 242, \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0242.png\", \"note\": \"Dense numeric-table page (Ch.5 Table 5.1, continues directly into the existing gold page as_p0243's copy of the same table). Gold = structural headers + sampled rows, same convention as as_p0243.\"}, {\"page_id\": \"as_p0247\", \"text\": \"EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.247)\\nEXPONENTIAL INTEGRALS E_n(x)  Table 5.4\\nColumns: x | E_2(x) | E_3(x) | E_4(x) | E_{10}(x) | E_{20}(x)\\nSample rows:\\n1.00  0.14849\\\\,55  0.10969\\\\,20  0.08606\\\\,25  0.03639\\\\,40  0.01834\\\\,60\\n1.20  0.11110\\\\,41  0.08393\\\\,47  0.06682\\\\,42  0.02916\\\\,68  0.01486\\\\,49\\n1.40  0.08388\\\\,99  0.06457\\\\,55  0.05206\\\\,37  0.02338\\\\,72  0.01204\\\\,58\\n1.60  0.06380\\\\,32  0.04990\\\\,57  0.04068\\\\,25  0.01876\\\\,22  0.00976\\\\,24\\n(A bracketed interpolation-difference scale factor is printed under each column at the foot of the page, not transcribed here.)\", \"chapter_id\": \"ch05_expint\", \"printed_page\": 247, \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0247.png\", \"note\": \"Dense numeric-table page (Ch.5 Table 5.4), rows x=1.00 to 1.60 step 0.01. Gold = structural headers + sampled rows (first/last + 2 middle), not exhaustive.\"}, {\"page_id\": \"as_p0256\", \"text\": \"GAMMA FUNCTION AND RELATED FUNCTIONS (p.256)\\nRecurrence Formulas\\n6.1.15  \\\\Gamma(z+1)=z\\\\Gamma(z)=z!=z(z-1)!\\n6.1.16  \\\\Gamma(n+z)=(n-1+z)(n-2+z)\\\\cdots(1+z)\\\\Gamma(1+z)=(n-1+z)!=(n-1+z)(n-2+z)\\\\cdots(1+z)z!\\nReflection Formula\\n6.1.17  \\\\Gamma(z)\\\\Gamma(1-z)=-z\\\\Gamma(-z)\\\\Gamma(z)=\\\\pi\\\\csc\\\\pi z=\\\\int_0^\\\\infty \\\\frac{t^{z-1}}{1+t}dt \\\\quad (0<\\\\Re z<1)\\nDuplication Formula\\n6.1.18  \\\\Gamma(2z)=(2\\\\pi)^{-1/2}2^{2z-1/2}\\\\Gamma(z)\\\\Gamma(z+\\\\tfrac12)\\nTriplication Formula\\n6.1.19  \\\\Gamma(3z)=(2\\\\pi)^{-1}3^{3z-1/2}\\\\Gamma(z)\\\\Gamma(z+\\\\tfrac13)\\\\Gamma(z+\\\\tfrac23)\\nGauss' Multiplication Formula\\n6.1.20  \\\\Gamma(nz)=(2\\\\pi)^{\\\\frac12(1-n)}n^{nz-\\\\frac12}\\\\prod_{k=0}^{n-1}\\\\Gamma\\\\left(z+\\\\frac{k}{n}\\\\right)\\nBinomial Coefficient\\n6.1.21  \\\\binom{z}{w}=\\\\frac{z!}{w!(z-w)!}=\\\\frac{\\\\Gamma(z+1)}{\\\\Gamma(w+1)\\\\Gamma(z-w+1)}\\nPochhammer's Symbol\\n6.1.22  (z)_0=1,\\\\quad (z)_n=z(z+1)(z+2)\\\\cdots(z+n-1)=\\\\frac{\\\\Gamma(z+n)}{\\\\Gamma(z)}\\nGamma Function in the Complex Plane\\n6.1.23  \\\\Gamma(\\\\bar z)=\\\\overline{\\\\Gamma(z)};\\\\ \\\\ln\\\\Gamma(\\\\bar z)=\\\\overline{\\\\ln\\\\Gamma(z)}\\n6.1.24  \\\\arg\\\\Gamma(z+1)=\\\\arg\\\\Gamma(z)+\\\\arctan\\\\frac{y}{x}\\n6.1.25  \\\\left|\\\\frac{\\\\Gamma(x+iy)}{\\\\Gamma(x)}\\\\right|^2=\\\\prod_{n=0}^{\\\\infty}\\\\left[1+\\\\frac{y^2}{(x+n)^2}\\\\right]^{-1}\\n6.1.26  |\\\\Gamma(x+iy)|\\\\le|\\\\Gamma(x)|\\n6.1.27  \\\\arg\\\\Gamma(x+iy)=y\\\\psi(x)+\\\\sum_{n=0}^{\\\\infty}\\\\left(\\\\frac{y}{x+n}-\\\\arctan\\\\frac{y}{x+n}\\\\right) \\\\quad (x+iy\\\\neq0,-1,-2,\\\\dots)\\nwhere \\\\psi(z)=\\\\Gamma'(z)/\\\\Gamma(z)\\n6.1.28  \\\\Gamma(1+iy)=iy\\\\,\\\\Gamma(iy)\\n6.1.29  \\\\Gamma(iy)\\\\Gamma(-iy)=|\\\\Gamma(iy)|^2=\\\\frac{\\\\pi}{y\\\\sinh\\\\pi y}\\n6.1.30  \\\\Gamma(\\\\tfrac12+iy)\\\\Gamma(\\\\tfrac12-iy)=|\\\\Gamma(\\\\tfrac12+iy)|^2=\\\\frac{\\\\pi}{\\\\cosh\\\\pi y}\\n6.1.31  \\\\Gamma(1+iy)\\\\Gamma(1-iy)=|\\\\Gamma(1+iy)|^2=\\\\frac{\\\\pi y}{\\\\sinh\\\\pi y}\\n6.1.32  \\\\Gamma(\\\\tfrac14+iy)\\\\Gamma(\\\\tfrac34-iy)=\\\\frac{\\\\pi\\\\sqrt2}{\\\\cosh\\\\pi y+i\\\\sinh\\\\pi y}\\nPower Series\\n6.1.33  \\\\ln\\\\Gamma(1+z)=-\\\\ln(1+z)+z(1-\\\\gamma)+\\\\sum_{n=2}^{\\\\infty}(-1)^n[\\\\zeta(n)-1]z^n/n \\\\quad (|z|<2)\\n\\\\zeta(n) is the Riemann Zeta Function (see chapter 23).\\nSeries Expansion for 1/\\\\Gamma(z)\\n6.1.34  \\\\frac{1}{\\\\Gamma(z)}=\\\\sum_{k=1}^{\\\\infty}c_kz^k \\\\quad (|z|<\\\\infty)\\nSample coefficients (k, c_k):\\n1  1.00000\\\\,00000\\\\,000000\\n2  0.57721\\\\,56649\\\\,015329\\n3  -0.65587\\\\,80715\\\\,202538\\n13  -0.00000\\\\,12504\\\\,934821\\n26  0.00000\\\\,00000\\\\,000001\\n(Coefficients from H. T. Davis, Tables of higher mathematical functions, Principia Press, 1933/1935, with corrections due to H. E. Salzer.)\", \"chapter_id\": \"ch06_gamma\", \"printed_page\": 256, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0256.png\", \"note\": \"Dense two-column formula page (Ch.6 Gamma function), 6.1.15-6.1.34, formulas transcribed in full; the 26-row c_k coefficient table under 6.1.34 sampled (first 3 + a couple more), same convention as a dense numeric table.\"}, {\"page_id\": \"as_p0258\", \"text\": \"GAMMA FUNCTION AND RELATED FUNCTIONS (p.258)\\nContinued Fraction\\n6.1.48  \\\\ln\\\\Gamma(z)+z-(z-\\\\tfrac12)\\\\ln z-\\\\tfrac12\\\\ln(2\\\\pi)=\\\\cfrac{a_0}{z+}\\\\cfrac{a_1}{z+}\\\\cfrac{a_2}{z+}\\\\cfrac{a_3}{z+}\\\\cfrac{a_4}{z+}\\\\cfrac{a_5}{z+}\\\\cdots \\\\quad (\\\\Re z>0)\\na_0=\\\\frac{1}{12},\\\\ a_1=\\\\frac{1}{30},\\\\ a_2=\\\\frac{53}{210},\\\\ a_3=\\\\frac{195}{371},\\\\ a_4=\\\\frac{22999}{22737},\\\\ a_5=\\\\frac{29944523}{19733142},\\\\ a_6=\\\\frac{109535241009}{48264275462}\\nWallis' Formula\\n6.1.49  \\\\frac{2}{\\\\pi}\\\\int_0^{\\\\pi/2}\\\\binom{\\\\sin}{\\\\cos}^{2n}x\\\\,dx=\\\\frac{1\\\\cdot3\\\\cdot5\\\\cdots(2n-1)}{2\\\\cdot4\\\\cdot6\\\\cdots(2n)}=\\\\frac{(2n)!}{2^{2n}(n!)^2}=\\\\frac{1}{2^{2n}}\\\\binom{2n}{n}=\\\\frac{\\\\Gamma(n+\\\\tfrac12)}{\\\\pi^{1/2}\\\\Gamma(n+1)} \\\\sim \\\\frac{1}{\\\\pi^{1/2}n^{1/2}}\\\\left[1-\\\\frac{1}{8n}+\\\\frac{1}{128n^2}-\\\\cdots\\\\right] \\\\quad (n\\\\to\\\\infty)\\nSome Definite Integrals\\n6.1.50  \\\\ln\\\\Gamma(z)=\\\\int_0^\\\\infty \\\\left[(z-1)e^{-t}-\\\\frac{e^{-t}-e^{-zt}}{1-e^{-t}}\\\\right]\\\\frac{dt}{t} \\\\quad (\\\\Re z>0)\\n        =(z-\\\\tfrac12)\\\\ln z-z+\\\\tfrac12\\\\ln 2\\\\pi+2\\\\int_0^\\\\infty \\\\frac{\\\\arctan(t/z)}{e^{2\\\\pi t}-1}dt \\\\quad (\\\\Re z>0)\\n6.2. Beta Function\\n6.2.1  B(z,w)=\\\\int_0^1 t^{z-1}(1-t)^{w-1}dt=\\\\int_0^\\\\infty \\\\frac{t^{z-1}}{(1+t)^{z+w}}dt=2\\\\int_0^{\\\\pi/2}(\\\\sin t)^{2z-1}(\\\\cos t)^{2w-1}dt \\\\quad (\\\\Re z>0,\\\\Re w>0)\\n6.2.2  B(z,w)=\\\\frac{\\\\Gamma(z)\\\\Gamma(w)}{\\\\Gamma(z+w)}=B(w,z)\\n6.3. Psi (Digamma) Function\\n6.3.1  \\\\psi(z)=d[\\\\ln\\\\Gamma(z)]/dz=\\\\Gamma'(z)/\\\\Gamma(z)\\nFIGURE 6.2. Psi function. y=\\\\psi(x)=d\\\\ln\\\\Gamma(x)/dx\\nInteger Values\\n6.3.2  \\\\psi(1)=-\\\\gamma,\\\\ \\\\psi(n)=-\\\\gamma+\\\\sum_{k=1}^{n-1}k^{-1} \\\\quad (n\\\\ge2)\\nFractional Values\\n6.3.3  \\\\psi(\\\\tfrac12)=-\\\\gamma-2\\\\ln2=-1.96351\\\\,00260\\\\,21423\\\\dots\\n6.3.4  \\\\psi(n+\\\\tfrac12)=-\\\\gamma-2\\\\ln2+2\\\\left(1+\\\\tfrac13+\\\\cdots+\\\\frac{1}{2n-1}\\\\right) \\\\quad (n\\\\ge1)\\nRecurrence Formulas\\n6.3.5  \\\\psi(z+1)=\\\\psi(z)+\\\\frac1z\\n6.3.6  \\\\psi(n+z)=\\\\frac{1}{(n-1)+z}+\\\\frac{1}{(n-2)+z}+\\\\cdots+\\\\frac{1}{2+z}+\\\\frac{1}{1+z}+\\\\psi(1+z)\", \"chapter_id\": \"ch06_gamma\", \"printed_page\": 258, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0258.png\", \"note\": \"Two-column formula page (Ch.6 Gamma function, Beta function intro, Psi function intro), 6.1.48-6.3.6, plus Figure 6.2 (Psi function plot, caption only) and two footnotes on double-factorial notation (not transcribed, cross-referenced elsewhere in the chapter).\"}, {\"page_id\": \"as_p0259\", \"text\": \"GAMMA FUNCTION AND RELATED FUNCTIONS (p.259)\\nReflection Formula\\n6.3.7  \\\\psi(1-z)=\\\\psi(z)+\\\\pi\\\\cot\\\\pi z\\nDuplication Formula\\n6.3.8  \\\\psi(2z)=\\\\tfrac12\\\\psi(z)+\\\\tfrac12\\\\psi(z+\\\\tfrac12)+\\\\ln2\\nPsi Function in the Complex Plane\\n6.3.9  \\\\psi(\\\\bar z)=\\\\overline{\\\\psi(z)}\\n6.3.10  \\\\mathscr{R}\\\\psi(iy)=\\\\mathscr{R}\\\\psi(-iy)=\\\\mathscr{R}\\\\psi(1+iy)=\\\\mathscr{R}\\\\psi(1-iy)\\n6.3.11  \\\\mathscr{I}\\\\psi(iy)=\\\\tfrac12y^{-1}+\\\\tfrac12\\\\pi\\\\coth\\\\pi y\\n6.3.12  \\\\mathscr{I}\\\\psi(\\\\tfrac12+iy)=\\\\tfrac12\\\\pi\\\\tanh\\\\pi y\\n6.3.13  \\\\mathscr{I}\\\\psi(1+iy)=-\\\\frac{1}{2y}+\\\\tfrac12\\\\pi\\\\coth\\\\pi y=y\\\\sum_{n=1}^{\\\\infty}(n^2+y^2)^{-1}\\nSeries Expansions\\n6.3.14  \\\\psi(1+z)=-\\\\gamma+\\\\sum_{n=2}^{\\\\infty}(-1)^n\\\\zeta(n)z^{n-1} \\\\quad (|z|<1)\\n6.3.15  \\\\psi(1+z)=\\\\tfrac12z^{-1}-\\\\tfrac12\\\\pi\\\\cot\\\\pi z-(1-z^2)^{-1}+1-\\\\gamma-\\\\sum_{n=1}^{\\\\infty}[\\\\zeta(2n+1)-1]z^{2n} \\\\quad (|z|<2)\\n6.3.16  \\\\psi(1+z)=-\\\\gamma+\\\\sum_{n=1}^{\\\\infty}\\\\frac{z}{n(n+z)} \\\\quad (z\\\\neq-1,-2,-3,\\\\dots)\\n6.3.17  \\\\mathscr{R}\\\\psi(1+iy)=1-\\\\gamma-\\\\frac{1}{1+y^2}+\\\\sum_{n=1}^{\\\\infty}(-1)^{n+1}[\\\\zeta(2n+1)-1]y^{2n} \\\\quad (|y|<2)\\n        =-\\\\gamma+y^2\\\\sum_{n=1}^{\\\\infty}n^{-1}(n^2+y^2)^{-1} \\\\quad (-\\\\infty<y<\\\\infty)\\nAsymptotic Formulas\\n6.3.18  \\\\psi(z)\\\\sim\\\\ln z-\\\\frac{1}{2z}-\\\\sum_{n=1}^{\\\\infty}\\\\frac{B_{2n}}{2nz^{2n}}=\\\\ln z-\\\\frac{1}{2z}-\\\\frac{1}{12z^2}+\\\\frac{1}{120z^4}-\\\\frac{1}{252z^6}+\\\\cdots \\\\quad (z\\\\to\\\\infty\\\\ \\\\mathrm{in}\\\\ |\\\\arg z|<\\\\pi)\\n6.3.19  \\\\mathscr{R}\\\\psi(1+iy)\\\\sim\\\\ln y+\\\\sum_{n=1}^{\\\\infty}\\\\frac{(-1)^{n-1}B_{2n}}{2ny^{2n}}=\\\\ln y+\\\\frac{1}{12y^2}+\\\\frac{1}{120y^4}+\\\\frac{1}{252y^6}+\\\\cdots \\\\quad (y\\\\to\\\\infty)\\nExtrema of \\\\Gamma(x) -- Zeros of \\\\psi(x): \\\\Gamma'(x_n)=\\\\psi(x_n)=0\\nColumns: n | x_n | \\\\Gamma(x_n)\\n0  +1.462  +0.886\\n1  -0.504  -3.545\\n2  -1.573  +2.302\\n3  -2.611  -0.888\\n4  -3.635  +0.245\\n5  -4.653  -0.053\\n6  -5.667  +0.009\\n7  -6.678  -0.001\\nx_0=1.46163\\\\,21449\\\\,68362,\\\\quad \\\\Gamma(x_0)=.88560\\\\,31944\\\\,10889\\n6.3.20  x_n=-n+(\\\\ln n)^{-1}+o[(\\\\ln n)^{-2}]\\nDefinite Integrals\\n6.3.21  \\\\psi(z)=\\\\int_0^\\\\infty \\\\left[\\\\frac{e^{-t}}{t}-\\\\frac{e^{-zt}}{1-e^{-t}}\\\\right]dt=\\\\int_0^\\\\infty \\\\left[e^{-t}-\\\\frac{1}{(1+t)^z}\\\\right]\\\\frac{dt}{t}=\\\\ln z-\\\\frac{1}{2z}-2\\\\int_0^\\\\infty \\\\frac{t\\\\,dt}{(t^2+z^2)(e^{2\\\\pi t}-1)} \\\\quad (|\\\\arg z|<\\\\tfrac{\\\\pi}{2})\\n6.3.22  \\\\psi(z)+\\\\gamma=\\\\int_0^\\\\infty \\\\frac{e^{-t}-e^{-zt}}{1-e^{-t}}dt=\\\\int_0^1 \\\\frac{1-t^{z-1}}{1-t}dt\\n        \\\\gamma=\\\\int_0^\\\\infty \\\\left(\\\\frac{1}{e^t-1}-\\\\frac{1}{te^t}\\\\right)dt=\\\\int_0^\\\\infty \\\\left(\\\\frac{1}{1+t}-e^{-t}\\\\right)\\\\frac{dt}{t}\\n(Table of x_n, \\\\Gamma(x_n) from W. Sibagaki, Theory and applications of the gamma function, Iwanami Syoten, Tokyo, 1952, with permission.)\", \"chapter_id\": \"ch06_gamma\", \"printed_page\": 259, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0259.png\", \"note\": \"Dense two-column formula page (Ch.6 Psi function), 6.3.7-6.3.22, plus a small 8-row extrema table transcribed in full. Government-document identifier '716-654 O - 64 - 18' at the page foot omitted (printing artifact, not content).\"}, {\"page_id\": \"as_p0260\", \"text\": \"GAMMA FUNCTION AND RELATED FUNCTIONS (p.260)\\n6.4. Polygamma Functions\\n6.4.1  \\\\psi^{(n)}(z)=\\\\frac{d^n}{dz^n}\\\\psi(z)=\\\\frac{d^{n+1}}{dz^{n+1}}\\\\ln\\\\Gamma(z) \\\\quad (n=1,2,3,\\\\dots) =(-1)^{n+1}\\\\int_0^\\\\infty \\\\frac{t^ne^{-zt}}{1-t}dt \\\\quad (\\\\Re z>0)\\n\\\\psi^{(n)}(z), (n=0,1,\\\\dots), is a single valued analytic function over the entire complex plane save at the points z=-m\\\\ (m=0,1,2,\\\\dots) where it possesses poles of order (n+1).\\nInteger Values\\n6.4.2  \\\\psi^{(n)}(1)=(-1)^{n+1}n!\\\\zeta(n+1) \\\\quad (n=1,2,3,\\\\dots)\\n6.4.3  \\\\psi^{(m)}(n+1)=(-1)^mm!\\\\left[-\\\\zeta(m+1)+1+\\\\frac{1}{2^{m+1}}+\\\\cdots+\\\\frac{1}{n^{m+1}}\\\\right]\\nFractional Values\\n6.4.4  \\\\psi^{(n)}(\\\\tfrac12)=(-1)^{n+1}n!(2^{n+1}-1)\\\\zeta(n+1) \\\\quad (n=1,2,\\\\dots)\\n6.4.5  \\\\psi'(n+\\\\tfrac12)=\\\\tfrac12\\\\pi^2-4\\\\sum_{k=1}^{n}(2k-1)^{-2}\\nRecurrence Formula\\n6.4.6  \\\\psi^{(n)}(z+1)=\\\\psi^{(n)}(z)+(-1)^nn!z^{-n-1}\\nReflection Formula\\n6.4.7  \\\\psi^{(n)}(1-z)+(-1)^{n+1}\\\\psi^{(n)}(z)=(-1)^n\\\\pi\\\\frac{d^n}{dz^n}\\\\cot\\\\pi z\\nMultiplication Formula\\n6.4.8  \\\\psi^{(n)}(mz)=\\\\frac{1}{m^{n+1}}\\\\left[\\\\psi^{(n)}(z)+\\\\psi^{(n)}\\\\left(z+\\\\frac1m\\\\right)+\\\\psi^{(n)}\\\\left(z+\\\\frac2m\\\\right)+\\\\cdots+\\\\psi^{(n)}\\\\left(z+1-\\\\frac1m\\\\right)\\\\right]\\nSeries Expansions\\n6.4.9  \\\\psi^{(n)}(1+z)=(-1)^{n+1}\\\\left[n!\\\\zeta(n+1)-\\\\frac{(n+1)!}{1!}\\\\zeta(n+2)z+\\\\frac{(n+2)!}{2!}\\\\zeta(n+3)z^2-\\\\cdots\\\\right] \\\\quad (|z|<1)\\n6.4.10  \\\\psi^{(n)}(z)=(-1)^{n+1}n!\\\\sum_{k=0}^{\\\\infty}(z+k)^{-n-1} \\\\quad (z\\\\neq0,-1,-2,\\\\dots)\\nAsymptotic Formulas\\n6.4.11  \\\\psi^{(n)}(z)\\\\sim(-1)^{n-1}\\\\left[\\\\frac{(n-1)!}{z^n}+\\\\frac{n!}{2z^{n+1}}+\\\\sum_{k=1}^{\\\\infty}B_{2k}\\\\frac{(2k+n-1)!}{(2k)!z^{2k+n}}\\\\right] \\\\quad (z\\\\to\\\\infty\\\\ \\\\mathrm{in}\\\\ |\\\\arg z|<\\\\pi)\\n6.4.12  \\\\psi'(z)\\\\sim\\\\frac1z+\\\\frac{1}{2z^2}+\\\\frac{1}{6z^3}-\\\\frac{1}{30z^5}+\\\\frac{1}{42z^7}-\\\\frac{1}{30z^9}+\\\\cdots \\\\quad (z\\\\to\\\\infty\\\\ \\\\mathrm{in}\\\\ |\\\\arg z|<\\\\pi)\\n6.4.13  \\\\psi''(z)\\\\sim-\\\\frac{1}{z^2}-\\\\frac{1}{z^3}-\\\\frac{1}{2z^4}+\\\\frac{1}{6z^6}-\\\\frac{1}{6z^8}+\\\\frac{3}{10z^{10}}-\\\\frac{5}{6z^{12}}+\\\\cdots \\\\quad (z\\\\to\\\\infty\\\\ \\\\mathrm{in}\\\\ |\\\\arg z|<\\\\pi)\\n6.4.14  \\\\psi^{(2)}(z)\\\\sim-\\\\frac{2}{z^3}+\\\\frac{3}{z^4}+\\\\frac{2}{z^5}-\\\\frac{1}{z^7}+\\\\frac{4}{3z^9}-\\\\frac{3}{z^{11}}+\\\\frac{10}{z^{13}}-\\\\cdots \\\\quad (z\\\\to\\\\infty\\\\ \\\\mathrm{in}\\\\ |\\\\arg z|<\\\\pi)\\n6.5. Incomplete Gamma Function (see also 26.4)\\n6.5.1  P(a,x)=\\\\frac{1}{\\\\Gamma(a)}\\\\int_0^x e^{-t}t^{a-1}dt \\\\quad (\\\\Re a>0)\\n6.5.2  \\\\gamma(a,x)=P(a,x)\\\\Gamma(a)=\\\\int_0^x e^{-t}t^{a-1}dt \\\\quad (\\\\Re a>0)\\n6.5.3  \\\\Gamma(a,x)=\\\\Gamma(a)-\\\\gamma(a,x)=\\\\int_x^\\\\infty e^{-t}t^{a-1}dt\\n6.5.4  \\\\gamma^*(a,x)=x^{-a}P(a,x)=\\\\frac{x^{-a}}{\\\\Gamma(a)}\\\\gamma(a,x)\\n\\\\gamma^* is a single valued analytic function of a and x possessing no finite singularities.\", \"chapter_id\": \"ch06_gamma\", \"printed_page\": 260, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0260.png\", \"note\": \"Dense two-column formula page: Ch.6 \\u00a76.4 Polygamma Functions in full (6.4.1-6.4.14, with prose definition) plus the opening of \\u00a76.5 Incomplete Gamma Function (6.5.1-6.5.4).\"}, {\"page_id\": \"as_p0262\", \"text\": \"GAMMA FUNCTION AND RELATED FUNCTIONS (p.262)\\n6.5.5  Probability Integral of the \\\\chi^2-Distribution\\nP(\\\\chi^2|\\\\nu)=\\\\frac{1}{2^{\\\\frac12\\\\nu}\\\\Gamma(\\\\nu/2)}\\\\int_0^{\\\\chi^2} t^{\\\\frac12\\\\nu-1}e^{-t/2}dt\\n6.5.6  (Pearson's Form of the Incomplete Gamma Function)\\nI(u,p)=\\\\frac{1}{\\\\Gamma(p+1)}\\\\int_0^{u\\\\sqrt{p+1}} e^{-t}t^pdt=P(p+1,u\\\\sqrt{p+1})\\n6.5.7  C(x,a)=\\\\int_x^\\\\infty t^{a-1}\\\\cos t\\\\,dt \\\\quad (\\\\Re a<1)\\n6.5.8  S(x,a)=\\\\int_x^\\\\infty t^{a-1}\\\\sin t\\\\,dt \\\\quad (\\\\Re a<1)\\n6.5.9  E_n(x)=\\\\int_1^\\\\infty e^{-xt}t^{-n}dt=x^{n-1}\\\\Gamma(1-n,x)\\n6.5.10  \\\\alpha_n(x)=\\\\int_1^\\\\infty e^{-xt}t^ndt=x^{-n-1}\\\\Gamma(1+n,x)\\n6.5.11  e_n(x)=\\\\sum_{j=0}^{n} \\\\frac{x^j}{j!}\\nIncomplete Gamma Function as a Confluent Hypergeometric Function (see chapter 13)\\n6.5.12  \\\\gamma(a,x)=a^{-1}x^ae^{-x}M(1,1+a,x)=a^{-1}x^aM(a,1+a,-x)\\nSpecial Values\\n6.5.13  P(n,x)=1-\\\\left(1+x+\\\\frac{x^2}{2!}+\\\\cdots+\\\\frac{x^{n-1}}{(n-1)!}\\\\right)e^{-x}=1-e_{n-1}(x)e^{-x}\\nFor relation to the Poisson distribution, see 26.4.\\n6.5.14  \\\\gamma^*(-n,x)=x^n\\n6.5.15  \\\\Gamma(0,x)=\\\\int_x^\\\\infty e^{-t}t^{-1}dt=E_1(x)\\n6.5.16  \\\\gamma(\\\\tfrac12,x^2)=2\\\\int_0^x e^{-t^2}dt=\\\\sqrt\\\\pi\\\\,\\\\mathrm{erf}\\\\,x\\n6.5.17  \\\\Gamma(\\\\tfrac12,x^2)=2\\\\int_x^\\\\infty e^{-t^2}dt=\\\\sqrt\\\\pi\\\\,\\\\mathrm{erfc}\\\\,x\\n6.5.18  \\\\tfrac12\\\\sqrt\\\\pi\\\\,x\\\\gamma^*(\\\\tfrac12,-x^2)=\\\\int_0^x e^{t^2}dt\\n6.5.19  \\\\Gamma(-n,x)=\\\\frac{(-1)^n}{n!}\\\\left[E_1(x)-e^{-x}\\\\sum_{j=0}^{n-1}\\\\frac{(-1)^jj!}{x^{j+1}}\\\\right]\\n6.5.20  \\\\Gamma(a,ix)=e^{\\\\frac12\\\\pi ia}[C(x,a)-iS(x,a)]\\nRecurrence Formulas\\n6.5.21  P(a+1,x)=P(a,x)-\\\\frac{x^ae^{-x}}{\\\\Gamma(a+1)}\\n6.5.22  \\\\gamma(a+1,x)=a\\\\gamma(a,x)-x^ae^{-x}\\n6.5.23  \\\\gamma^*(a-1,x)=x\\\\gamma^*(a,x)+\\\\frac{e^{-x}}{\\\\Gamma(a)}\\nDerivatives and Differential Equations\\n6.5.24  \\\\left(\\\\frac{\\\\partial\\\\gamma^*}{\\\\partial\\\\alpha}\\\\right)_{\\\\alpha=0}=-\\\\int_x^\\\\infty \\\\frac{e^{-t}dt}{t}-\\\\ln x=-E_1(x)-\\\\ln x\\n6.5.25  \\\\frac{\\\\partial\\\\gamma(a,x)}{\\\\partial x}=-\\\\frac{\\\\partial\\\\Gamma(a,x)}{\\\\partial x}=x^{a-1}e^{-x}\\n6.5.26  \\\\frac{\\\\partial^n}{\\\\partial x^n}[x^{-a}\\\\Gamma(a,x)]=(-1)^nx^{-a-n}\\\\Gamma(a+n,x) \\\\quad (n=0,1,2,\\\\dots)\\n6.5.27  \\\\frac{\\\\partial^n}{\\\\partial x^n}[e^xx^a\\\\gamma^*(a,x)]=e^xx^{a-n}\\\\gamma^*(a-n,x) \\\\quad (n=0,1,2,\\\\dots)\\n6.5.28  x\\\\frac{\\\\partial^2\\\\gamma^*}{\\\\partial x^2}+(a+1+x)\\\\frac{\\\\partial\\\\gamma^*}{\\\\partial x}+a\\\\gamma^*=0\\nSeries Developments\\n6.5.29  \\\\gamma^*(a,z)=e^{-z}\\\\sum_{n=0}^{\\\\infty} \\\\frac{z^n}{\\\\Gamma(a+n+1)}=\\\\frac{1}{\\\\Gamma(a)}\\\\sum_{n=0}^{\\\\infty} \\\\frac{(-z)^n}{(a+n)n!} \\\\quad (|z|<\\\\infty)\", \"chapter_id\": \"ch06_gamma\", \"printed_page\": 262, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0262.png\", \"note\": \"Dense two-column formula page (Ch.6 Incomplete Gamma Function), 6.5.5-6.5.29, transcribed in full.\"}, {\"page_id\": \"as_p0280\", \"text\": \"GAMMA FUNCTION AND RELATED FUNCTIONS (p.280)\\nTable 6.7  GAMMA FUNCTION FOR COMPLEX ARGUMENTS, x=1.3\\nColumns: y | \\\\mathscr{R}\\\\ln\\\\Gamma(z) | \\\\mathscr{I}\\\\ln\\\\Gamma(z)\\nSample rows:\\n0.0   -0.10817\\\\,48095\\\\,08   0.00000\\\\,00000\\\\,00\\n2.5   -2.26671\\\\,88222\\\\,04   0.93666\\\\,21049\\\\,03\\n5.0   -5.64541\\\\,41381\\\\,33   4.24823\\\\,90621\\\\,27\\n7.5   -9.24918\\\\,73322\\\\,19   8.83132\\\\,20546\\\\,97\\n10.0  -12.94643\\\\,67480\\\\,34   14.25466\\\\,45529\\\\,28\\n(y runs 0.0 to 10.0 in steps of 0.1, printed as two side-by-side blocks of columns on the page; only a sampled subset of rows is given here.)\", \"chapter_id\": \"ch06_gamma\", \"printed_page\": 280, \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0280.png\", \"note\": \"Dense numeric-table page (Ch.6 Table 6.7, complex-argument ln Gamma at fixed x=1.3, 101 rows). Gold = structural headers + sampled rows, same convention as as_p0243.\"}, {\"page_id\": \"as_p0281\", \"text\": \"GAMMA FUNCTION AND RELATED FUNCTIONS (p.281)\\nGAMMA FUNCTION FOR COMPLEX ARGUMENTS  Table 6.7\\nx=1.4\\nColumns: y | \\\\mathscr{R}\\\\ln\\\\Gamma(z) | \\\\mathscr{I}\\\\ln\\\\Gamma(z)\\nSample rows:\\n0.0   -0.11961\\\\,29141\\\\,72   0.00000\\\\,00000\\\\,00\\n2.5   -2.17009\\\\,23032\\\\,73   1.06059\\\\,19035\\\\,92\\n5.0   -5.48319\\\\,80511\\\\,50   4.38842\\\\,59888\\\\,87\\n7.5   -9.04712\\\\,96653\\\\,17   8.97710\\\\,02057\\\\,23\\n10.0  -12.71585\\\\,87212\\\\,03   14.40325\\\\,76321\\\\,42\\n(y runs 0.0 to 10.0 in steps of 0.1, printed as two side-by-side blocks of columns on the page; only a sampled subset of rows is given here. Directly continues as_p0280's copy of the same table at x=1.3.)\", \"chapter_id\": \"ch06_gamma\", \"printed_page\": 281, \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0281.png\", \"note\": \"Dense numeric-table page (Ch.6 Table 6.7, complex-argument ln Gamma at fixed x=1.4, 101 rows). Gold = structural headers + sampled rows, same convention as as_p0243.\"}, {\"page_id\": \"as_p0298\", \"text\": \"ERROR FUNCTION AND FRESNEL INTEGRALS (p.298)\\nFIGURE 7.3. Altitude Chart of w(z).\\nInequalities [7.11], [7.17]\\n7.1.13  \\\\frac{1}{x+\\\\sqrt{x^2+2}}<e^{x^2}\\\\int_x^\\\\infty e^{-t^2}dt\\\\le\\\\frac{1}{x+\\\\sqrt{x^2+\\\\frac4\\\\pi}} \\\\quad (x\\\\ge0)\\n(For other inequalities see [7.2].)\\nContinued Fractions\\n7.1.14  2e^{z^2}\\\\int_z^\\\\infty e^{-t^2}dt=\\\\cfrac{1}{z+}\\\\cfrac{1/2}{z+}\\\\cfrac{1}{z+}\\\\cfrac{3/2}{z+}\\\\cfrac{2}{z+}\\\\cdots \\\\quad (\\\\Re z>0)\\n7.1.15  \\\\frac{1}{\\\\sqrt\\\\pi}\\\\int_{-\\\\infty}^{\\\\infty} \\\\frac{e^{-t^2}dt}{z-t}=\\\\cfrac{1}{z-}\\\\cfrac{1/2}{z-}\\\\cfrac{1}{z-}\\\\cfrac{3/2}{z-}\\\\cfrac{2}{z-}\\\\cdots=\\\\frac{1}{\\\\sqrt\\\\pi}\\\\lim_{n\\\\to\\\\infty}\\\\sum_{k=1}^{n} \\\\frac{H_k^{(n)}}{z-x_k^{(n)}} \\\\quad (\\\\mathscr{I}z\\\\neq0)\\nx_k^{(n)} and H_k^{(n)} are the zeros and weight factors of the Hermite polynomials. For numerical values see chapter 25.\\nValue at Infinity\\n7.1.16  \\\\mathrm{erf}\\\\,z\\\\to1 \\\\quad (z\\\\to\\\\infty\\\\ \\\\mathrm{in}\\\\ |\\\\arg z|<\\\\tfrac{\\\\pi}{4})\\nMaximum and Inflection Points for Dawson's Integral [7.31]\\nF(x)=e^{-x^2}\\\\int_0^x e^{t^2}dt\\n7.1.17  F(.92413\\\\,88730\\\\dots)=.54104\\\\,42246\\\\dots\\n7.1.18  F(1.50197\\\\,52682\\\\dots)=.42768\\\\,66160\\\\dots\\nDerivatives\\n7.1.19  \\\\frac{d^{n+1}}{dz^{n+1}}\\\\mathrm{erf}\\\\,z=(-1)^n\\\\frac{2}{\\\\sqrt\\\\pi}H_n(z)e^{-z^2} \\\\quad (n=0,1,2,\\\\dots)\\n7.1.20  w^{(n+2)}(z)+2zw^{(n+1)}(z)+2(n+1)w^{(n)}(z)=0 \\\\quad (n=0,1,2,\\\\dots)\\n        w^{(0)}(z)=w(z),\\\\quad w'(z)=-2zw(z)+\\\\frac{2i}{\\\\sqrt\\\\pi}\\n(For the Hermite polynomials H_n(z) see chapter 22.)\\nRelation to Confluent Hypergeometric Function (see chapter 13)\\n7.1.21  \\\\mathrm{erf}\\\\,z=\\\\frac{2z}{\\\\sqrt\\\\pi}M(\\\\tfrac12,\\\\tfrac32,-z^2)=\\\\frac{2z}{\\\\sqrt\\\\pi}e^{-z^2}M(1,\\\\tfrac32,z^2)\\nThe Normal Distribution Function With Mean m and Standard Deviation \\\\sigma (see chapter 26)\\n7.1.22  \\\\frac{1}{\\\\sigma\\\\sqrt{2\\\\pi}}\\\\int_{-\\\\infty}^{x} e^{-\\\\frac{(t-m)^2}{2\\\\sigma^2}}dt=\\\\frac12\\\\left(1+\\\\mathrm{erf}\\\\left(\\\\frac{x-m}{\\\\sigma\\\\sqrt2}\\\\right)\\\\right)\\nAsymptotic Expansion\\n7.1.23  \\\\sqrt\\\\pi\\\\,ze^{z^2}\\\\mathrm{erfc}\\\\,z\\\\sim1+\\\\sum_{m=1}^{\\\\infty} (-1)^m\\\\frac{1\\\\cdot3\\\\cdots(2m-1)}{(2z^2)^m} \\\\quad (z\\\\to\\\\infty,|\\\\arg z|<\\\\tfrac{3\\\\pi}{4})\", \"chapter_id\": \"ch07_error_fresnel\", \"printed_page\": 298, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0298.png\", \"note\": \"Two-column formula page (Ch.7 error function), 7.1.13-7.1.23, plus Figure 7.3 (a large conformal 'altitude chart' of w(z), caption only -- the plot's contour data is visual, not transcribable text).\"}, {\"page_id\": \"as_p0301\", \"text\": \"ERROR FUNCTION AND FRESNEL INTEGRALS (p.301)\\n7.3.8  S(x)=S_1\\\\left(x\\\\sqrt{\\\\tfrac{\\\\pi}{2}}\\\\right)=S_2\\\\left(\\\\tfrac{\\\\pi}{2}x^2\\\\right)\\n7.3.9  C(z)=\\\\tfrac12+f(z)\\\\sin(\\\\tfrac{\\\\pi}{2}z^2)-g(z)\\\\cos(\\\\tfrac{\\\\pi}{2}z^2)\\n7.3.10  S(z)=\\\\tfrac12-f(z)\\\\cos(\\\\tfrac{\\\\pi}{2}z^2)-g(z)\\\\sin(\\\\tfrac{\\\\pi}{2}z^2)\\nSeries Expansions\\n7.3.11  C(z)=\\\\sum_{n=0}^{\\\\infty} \\\\frac{(-1)^n(\\\\pi/2)^{2n}}{(2n)!(4n+1)}z^{4n+1}\\n7.3.12  C(z)=\\\\cos(\\\\tfrac{\\\\pi}{2}z^2)\\\\sum_{n=0}^{\\\\infty} \\\\frac{(-1)^n\\\\pi^{2n}}{1\\\\cdot3\\\\cdots(4n+1)}z^{4n+1}+\\\\sin(\\\\tfrac{\\\\pi}{2}z^2)\\\\sum_{n=0}^{\\\\infty} \\\\frac{(-1)^n\\\\pi^{2n+1}}{1\\\\cdot3\\\\cdots(4n+3)}z^{4n+3}\\n7.3.13  S(z)=\\\\sum_{n=0}^{\\\\infty} \\\\frac{(-1)^n(\\\\pi/2)^{2n+1}}{(2n+1)!(4n+3)}z^{4n+3}\\nFIGURE 7.5. Fresnel Integrals. y=C(x), y=S(x)\\n7.3.14  S(z)=-\\\\cos(\\\\tfrac{\\\\pi}{2}z^2)\\\\sum_{n=0}^{\\\\infty} \\\\frac{(-1)^n\\\\pi^{2n+1}}{1\\\\cdot3\\\\cdots(4n+3)}z^{4n+3}+\\\\sin(\\\\tfrac{\\\\pi}{2}z^2)\\\\sum_{n=0}^{\\\\infty} \\\\frac{(-1)^n\\\\pi^{2n}}{1\\\\cdot3\\\\cdots(4n+1)}z^{4n+1}\\n7.3.15  C_2(z)=J_{1/2}(z)+J_{5/2}(z)+J_{9/2}(z)+\\\\cdots\\n7.3.16  S_2(z)=J_{3/2}(z)+J_{7/2}(z)+J_{11/2}(z)+\\\\cdots\\nFor Bessel functions J_{n+1/2}(z) see chapter 10.\\nSymmetry Relations\\n7.3.17  C(-z)=-C(z),\\\\ S(-z)=-S(z)\\n7.3.18  C(iz)=iC(z),\\\\ S(iz)=-iS(z)\\n7.3.19  C(\\\\bar z)=\\\\overline{C(z)},\\\\ S(\\\\bar z)=\\\\overline{S(z)}\\nValue at Infinity\\n7.3.20  C(x)\\\\to\\\\tfrac12,\\\\ S(x)\\\\to\\\\tfrac12 \\\\quad (x\\\\to\\\\infty)\\nDerivatives\\n7.3.21  \\\\frac{df(x)}{dx}=-\\\\pi xg(x),\\\\quad \\\\frac{dg(x)}{dx}=\\\\pi xf(x)-1\\nRelation to Error Function (see 7.1.1, 7.1.3)\\n7.3.22  C(z)+iS(z)=\\\\frac{1+i}{2}\\\\mathrm{erf}\\\\left[\\\\frac{\\\\sqrt\\\\pi}{2}(1-i)z\\\\right]=\\\\frac{1+i}{2}\\\\left\\\\{1-e^{i\\\\frac{\\\\pi}{2}z^2}w\\\\left[\\\\frac{\\\\sqrt\\\\pi}{2}(1+i)z\\\\right]\\\\right\\\\}\\n7.3.23  g(x)=\\\\mathscr{R}\\\\left\\\\{\\\\frac{1+i}{2}w\\\\left[\\\\frac{\\\\sqrt\\\\pi}{2}(1+i)x\\\\right]\\\\right\\\\}\\n7.3.24  f(x)=\\\\mathscr{I}\\\\left\\\\{\\\\frac{1+i}{2}w\\\\left[\\\\frac{\\\\sqrt\\\\pi}{2}(1+i)x\\\\right]\\\\right\\\\}\\nRelation to Confluent Hypergeometric Function (see chapter 13)\\n7.3.25  C(z)+iS(z)=zM\\\\left(\\\\tfrac12,\\\\tfrac32,i\\\\tfrac{\\\\pi}{2}z^2\\\\right)=ze^{i\\\\frac{\\\\pi}{2}z^2}M\\\\left(1,\\\\tfrac32,-i\\\\tfrac{\\\\pi}{2}z^2\\\\right)\\nRelation to Spherical Bessel Functions (see chapter 10)\\n7.3.26  C_2(z)=\\\\tfrac12\\\\int_0^z J_{-\\\\frac12}(t)dt,\\\\ S_2(z)=\\\\tfrac12\\\\int_0^z J_{\\\\frac12}(t)dt\", \"chapter_id\": \"ch07_error_fresnel\", \"printed_page\": 301, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0301.png\", \"note\": \"Two-column formula page (Ch.7 Fresnel integrals), 7.3.8-7.3.26, plus Figure 7.5 (Fresnel integral plot, caption only).\"}, {\"page_id\": \"as_p0302\", \"text\": \"ERROR FUNCTION AND FRESNEL INTEGRALS (p.302)\\nAsymptotic Expansions\\n7.3.27  \\\\pi zf(z)\\\\sim1+\\\\sum_{m=1}^{\\\\infty} (-1)^m\\\\frac{1\\\\cdot3\\\\cdots(4m-1)}{(\\\\pi z^2)^{2m}} \\\\quad (z\\\\to\\\\infty,|\\\\arg z|<\\\\tfrac{\\\\pi}{2})\\n7.3.28  \\\\pi zg(z)\\\\sim\\\\sum_{m=0}^{\\\\infty} (-1)^m\\\\frac{1\\\\cdot3\\\\cdots(4m+1)}{(\\\\pi z^2)^{2m+1}} \\\\quad (z\\\\to\\\\infty,|\\\\arg z|<\\\\tfrac{\\\\pi}{2})\\nIf R_n^{(f)}(z), R_n^{(g)}(z) are the remainders after n terms in 7.3.27, 7.3.28, respectively, then\\n7.3.29  R_n^{(f)}(z)=(-1)^n\\\\frac{1\\\\cdot3\\\\cdots(4n-1)}{(\\\\pi z^2)^{2n}}\\\\theta^{(f)},\\\\quad \\\\theta^{(f)}=\\\\frac{1}{\\\\Gamma(2n+\\\\frac12)}\\\\int_0^\\\\infty \\\\frac{e^{-t}t^{2n-\\\\frac12}}{1+\\\\left(\\\\frac{2t}{\\\\pi z^2}\\\\right)^2}dt \\\\quad (|\\\\arg z|<\\\\tfrac{\\\\pi}{4})\\n7.3.30  R_n^{(g)}(z)=(-1)^n\\\\frac{1\\\\cdot3\\\\cdots(4n+1)}{(\\\\pi z^2)^{2n}}\\\\theta^{(g)},\\\\quad \\\\theta^{(g)}=\\\\frac{1}{\\\\Gamma(2n+\\\\frac32)}\\\\int_0^\\\\infty \\\\frac{e^{-t}t^{2n+\\\\frac12}}{1+\\\\left(\\\\frac{2t}{\\\\pi z^2}\\\\right)^2}dt \\\\quad (|\\\\arg z|<\\\\tfrac{\\\\pi}{4})\\n7.3.31  |\\\\theta^{(f)}|<1,\\\\ |\\\\theta^{(g)}|<1 \\\\quad (|\\\\arg z|\\\\le\\\\tfrac{\\\\pi}{8})\\nFor x real, R_n^{(f)}(x) and R_n^{(g)}(x) are less in absolute value than the first neglected term and of the same sign.\\nRational Approximations (0\\\\le x\\\\le\\\\infty)\\n7.3.32  f(x)=\\\\frac{1+.926x}{2+1.792x+3.104x^2}+\\\\epsilon(x) \\\\quad |\\\\epsilon(x)|\\\\le2\\\\times10^{-3}\\n7.3.33  g(x)=\\\\frac{1}{2+4.142x+3.492x^2+6.670x^3}+\\\\epsilon(x) \\\\quad |\\\\epsilon(x)|\\\\le2\\\\times10^{-3}\\n(For more accurate approximations see [7.1].)\\n7.4. Definite and Indefinite Integrals\\nFor a more extensive list of integrals see [7.5], [7.8], [7.15].\\n7.4.1  \\\\int_0^\\\\infty e^{-t^2}dt=\\\\frac{\\\\sqrt\\\\pi}{2}\\n7.4.2  \\\\int_0^\\\\infty e^{-(at^2+2bt+c)}dt=\\\\tfrac12\\\\sqrt{\\\\frac{\\\\pi}{a}}e^{\\\\frac{b^2-ac}{a}}\\\\mathrm{erfc}\\\\frac{b}{\\\\sqrt a} \\\\quad (\\\\Re a>0)\\n7.4.3  \\\\int_0^\\\\infty e^{-at^2-b/t^2}dt=\\\\tfrac12\\\\sqrt{\\\\frac{\\\\pi}{a}}e^{-2\\\\sqrt{ab}} \\\\quad (\\\\Re a>0,\\\\Re b>0)\\n7.4.4  \\\\int_0^\\\\infty t^{2n}e^{-at^2}dt=\\\\frac{1\\\\cdot3\\\\cdots(2n-1)}{2^{n+1}a^n}\\\\sqrt{\\\\frac{\\\\pi}{a}}=\\\\frac{\\\\Gamma(n+\\\\frac12)}{2a^{n+\\\\frac12}} \\\\quad (\\\\Re a>0;n=0,1,2,\\\\dots)\\n7.4.5  \\\\int_0^\\\\infty t^{2n+1}e^{-at^2}dt=\\\\frac{n!}{2a^{n+1}} \\\\quad (\\\\Re a>0;n=0,1,2,\\\\dots)\\n7.4.6  \\\\int_0^\\\\infty e^{-at^2}\\\\cos(2xt)dt=\\\\tfrac12\\\\sqrt{\\\\frac{\\\\pi}{a}}e^{-x^2/a} \\\\quad (\\\\Re a>0)\\n7.4.7  \\\\int_0^\\\\infty e^{-at^2}\\\\sin(2xt)dt=\\\\frac{1}{\\\\sqrt a}e^{-x^2/a}\\\\int_0^{x/\\\\sqrt a} e^{t^2}dt \\\\quad (\\\\Re a>0)\\n7.4.8  \\\\int_0^\\\\infty \\\\frac{e^{-at}dt}{\\\\sqrt{t+z^2}}=\\\\sqrt{\\\\frac{\\\\pi}{a}}e^{az^2}\\\\mathrm{erfc}\\\\sqrt{az} \\\\quad (\\\\Re a>0,\\\\Re z>0)\\n7.4.9  \\\\int_0^\\\\infty \\\\frac{e^{-at}dt}{\\\\sqrt t(t+z)}=\\\\frac{\\\\pi}{\\\\sqrt z}e^{az}\\\\mathrm{erfc}\\\\sqrt{az} \\\\quad (\\\\Re a>0,z\\\\neq0,|\\\\arg z|<\\\\pi)\\n7.4.10  \\\\int_0^\\\\infty \\\\frac{e^{-at^2}dt}{t+x}=e^{-ax^2}\\\\left[\\\\sqrt\\\\pi \\\\int_0^{ax} e^{t^2}dt-\\\\tfrac12\\\\mathrm{Ei}(ax^2)\\\\right] \\\\quad (a>0,x>0)\\n7.4.11  \\\\int_0^\\\\infty \\\\frac{e^{-at^2}dt}{t^2+x^2}=\\\\frac{\\\\pi}{2x}e^{ax^2}\\\\mathrm{erfc}\\\\sqrt{ax} \\\\quad (a>0,x>0)\\n7.4.12  \\\\int_0^1 \\\\frac{e^{-at^2}dt}{t^2+1}=\\\\frac{\\\\pi}{4}e^a[1-(\\\\mathrm{erf}\\\\sqrt a)^2] \\\\quad (a>0)\\n7.4.13  \\\\int_{-\\\\infty}^{\\\\infty} \\\\frac{ye^{-t^2}dt}{(x-t)^2+y^2}=\\\\pi\\\\mathscr{R}w(x+iy) \\\\quad (x\\\\ \\\\mathrm{real},y>0)\", \"chapter_id\": \"ch07_error_fresnel\", \"printed_page\": 302, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0302.png\", \"note\": \"Dense two-column formula page (Ch.7 error function asymptotics + integrals), 7.3.27-7.4.13, plus a prose remainder-bound paragraph, transcribed in full.\"}, {\"page_id\": \"as_p0303\", \"text\": \"ERROR FUNCTION AND FRESNEL INTEGRALS (p.303)\\n7.4.14  \\\\int_{-\\\\infty}^{\\\\infty} \\\\frac{(x-t)e^{-t^2}dt}{(x-t)^2+y^2}=\\\\pi\\\\mathscr{I}w(x+iy) \\\\quad (x\\\\ \\\\mathrm{real},y>0)\\n7.4.15  \\\\int_0^\\\\infty \\\\frac{[t^2-(x^2-y^2)]e^{-t^2}dt}{t^4-2(x^2-y^2)t^2+(x^2+y^2)^2}=\\\\frac{\\\\pi}{2}\\\\mathscr{R}\\\\frac{w(x+iy)}{y-ix} \\\\quad (x\\\\ \\\\mathrm{real},y>0)\\n7.4.16  \\\\int_0^\\\\infty \\\\frac{2xye^{-t^2}dt}{t^4-2(x^2-y^2)t^2+(x^2+y^2)^2}=\\\\frac{\\\\pi}{2}\\\\mathscr{I}\\\\frac{w(x+iy)}{y-ix} \\\\quad (x\\\\ \\\\mathrm{real},y>0)\\n7.4.17  \\\\int_0^\\\\infty e^{-at}\\\\,\\\\mathrm{erf}\\\\,bt\\\\,dt=\\\\frac1a e^{\\\\frac{a^2}{4b^2}}\\\\mathrm{erfc}\\\\frac{a}{2b} \\\\quad (\\\\Re a>0,|\\\\arg b|<\\\\tfrac{\\\\pi}{4})\\n7.4.18  \\\\int_0^\\\\infty \\\\sin(2at)\\\\,\\\\mathrm{erfc}\\\\,bt\\\\,dt=\\\\frac{1}{2a}[1-e^{-(a/b)^2}] \\\\quad (a>0,\\\\Re b>0)\\n7.4.19  \\\\int_0^\\\\infty e^{-at}\\\\,\\\\mathrm{erf}\\\\sqrt{bt}\\\\,dt=\\\\frac1a\\\\sqrt{\\\\frac{b}{a+b}} \\\\quad (\\\\Re(a+b)>0)\\n7.4.20  \\\\int_0^\\\\infty e^{-at}\\\\,\\\\mathrm{erfc}\\\\sqrt{\\\\frac{b}{t}}\\\\,dt=\\\\frac1a e^{-2\\\\sqrt{ab}} \\\\quad (\\\\Re a>0,\\\\Re b>0)\\n7.4.21  \\\\int_0^\\\\infty e^{(a-b)t}\\\\,\\\\mathrm{erfc}\\\\left(\\\\sqrt{at}+\\\\sqrt{\\\\frac{c}{t}}\\\\right)dt=\\\\frac{e^{-2(\\\\sqrt{ac}+\\\\sqrt{bc})}}{\\\\sqrt b(\\\\sqrt a+\\\\sqrt b)} \\\\quad (\\\\Re b>0,\\\\Re c>0)\\n7.4.22  \\\\int_0^\\\\infty e^{-at}\\\\cos(t^2)dt=\\\\sqrt{\\\\frac{\\\\pi}{2}}\\\\left\\\\{\\\\left[\\\\tfrac12-S\\\\left(\\\\tfrac{a}{2}\\\\sqrt{\\\\tfrac2\\\\pi}\\\\right)\\\\right]\\\\cos\\\\left(\\\\tfrac{a^2}{4}\\\\right)-\\\\left[\\\\tfrac12-C\\\\left(\\\\tfrac{a}{2}\\\\sqrt{\\\\tfrac2\\\\pi}\\\\right)\\\\right]\\\\sin\\\\left(\\\\tfrac{a^2}{4}\\\\right)\\\\right\\\\} \\\\quad (\\\\Re a>0)\\n7.4.23  \\\\int_0^\\\\infty e^{-at}\\\\sin(t^2)dt=\\\\sqrt{\\\\frac{\\\\pi}{2}}\\\\left\\\\{\\\\left[\\\\tfrac12-C\\\\left(\\\\tfrac{a}{2}\\\\sqrt{\\\\tfrac2\\\\pi}\\\\right)\\\\right]\\\\cos\\\\left(\\\\tfrac{a^2}{4}\\\\right)+\\\\left[\\\\tfrac12-S\\\\left(\\\\tfrac{a}{2}\\\\sqrt{\\\\tfrac2\\\\pi}\\\\right)\\\\right]\\\\sin\\\\left(\\\\tfrac{a^2}{4}\\\\right)\\\\right\\\\} \\\\quad (\\\\Re a>0)\\n7.4.24  \\\\int_0^\\\\infty e^{-at}\\\\frac{\\\\sin(t^2)}{t}dt=\\\\frac{\\\\pi}{2}\\\\left[\\\\tfrac12-C\\\\left(\\\\tfrac{a}{2}\\\\sqrt{\\\\tfrac2\\\\pi}\\\\right)\\\\right]^2+\\\\frac{\\\\pi}{2}\\\\left[\\\\tfrac12-S\\\\left(\\\\tfrac{a}{2}\\\\sqrt{\\\\tfrac2\\\\pi}\\\\right)\\\\right]^2 \\\\quad (\\\\Re a>0)\\n7.4.25  \\\\int_0^\\\\infty \\\\frac{e^{-a\\\\sqrt t}}{t^2+b^2}dt=\\\\pi\\\\sqrt{\\\\frac2b}\\\\left\\\\{\\\\left[\\\\tfrac12-C\\\\left(\\\\sqrt{\\\\tfrac{2ab}{\\\\pi}}\\\\right)\\\\right]\\\\cos(ab)+\\\\left[\\\\tfrac12-S\\\\left(\\\\sqrt{\\\\tfrac{2ab}{\\\\pi}}\\\\right)\\\\right]\\\\sin(ab)\\\\right\\\\} \\\\quad (\\\\Re a>0,\\\\Re b>0)\\n7.4.26  \\\\int_0^\\\\infty \\\\frac{e^{-at}dt}{\\\\sqrt t(t^2+b^2)}=\\\\frac{\\\\pi}{b}\\\\sqrt{\\\\frac2b}\\\\left\\\\{\\\\left[\\\\tfrac12-S\\\\left(\\\\sqrt{\\\\tfrac{2ab}{\\\\pi}}\\\\right)\\\\right]\\\\cos(ab)-\\\\left[\\\\tfrac12-C\\\\left(\\\\sqrt{\\\\tfrac{2ab}{\\\\pi}}\\\\right)\\\\right]\\\\sin(ab)\\\\right\\\\} \\\\quad (\\\\Re a>0,\\\\Re b>0)\\n7.4.27  \\\\int_0^\\\\infty e^{-at}C(t)dt=\\\\frac1a\\\\left\\\\{\\\\left[\\\\tfrac12-S\\\\left(\\\\tfrac{a}{\\\\pi}\\\\right)\\\\right]\\\\cos\\\\left(\\\\tfrac{a^2}{2\\\\pi}\\\\right)-\\\\left[\\\\tfrac12-C\\\\left(\\\\tfrac{a}{\\\\pi}\\\\right)\\\\right]\\\\sin\\\\left(\\\\tfrac{a^2}{2\\\\pi}\\\\right)\\\\right\\\\} \\\\quad (\\\\Re a>0)\\n7.4.28  \\\\int_0^\\\\infty e^{-at}S(t)dt=\\\\frac1a\\\\left\\\\{\\\\left[\\\\tfrac12-C\\\\left(\\\\tfrac{a}{\\\\pi}\\\\right)\\\\right]\\\\cos\\\\left(\\\\tfrac{a^2}{2\\\\pi}\\\\right)+\\\\left[\\\\tfrac12-S\\\\left(\\\\tfrac{a}{\\\\pi}\\\\right)\\\\right]\\\\sin\\\\left(\\\\tfrac{a^2}{2\\\\pi}\\\\right)\\\\right\\\\} \\\\quad (\\\\Re a>0)\\n7.4.29  \\\\int_0^\\\\infty e^{-at}C\\\\left(\\\\sqrt{\\\\tfrac{2t}{\\\\pi}}\\\\right)dt=\\\\frac{1}{2a(\\\\sqrt{a^2+1}-a)^{1/2}\\\\sqrt{a^2+1}} \\\\quad (\\\\Re a>0)\\n7.4.30  \\\\int_0^\\\\infty e^{-at}S\\\\left(\\\\sqrt{\\\\tfrac{2t}{\\\\pi}}\\\\right)dt=\\\\frac{1}{2a(\\\\sqrt{a^2+1}+a)^{1/2}\\\\sqrt{a^2+1}} \\\\quad (\\\\Re a>0)\\n7.4.31  \\\\int_0^\\\\infty \\\\left\\\\{\\\\left[\\\\tfrac12-C(t)\\\\right]^2+\\\\left[\\\\tfrac12-S(t)\\\\right]^2\\\\right\\\\}dt=\\\\frac1\\\\pi\\n7.4.32  \\\\int e^{-(ax^2+2bx+c)}dx=\\\\tfrac12\\\\sqrt{\\\\frac{\\\\pi}{a}}e^{\\\\frac{b^2-ac}{a}}\\\\mathrm{erf}\\\\left(\\\\sqrt a\\\\,x+\\\\frac{b}{\\\\sqrt a}\\\\right)+\\\\mathrm{const.} \\\\quad (a\\\\neq0)\", \"chapter_id\": \"ch07_error_fresnel\", \"printed_page\": 303, \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0303.png\", \"note\": \"Dense two-column integral-table page (Ch.7 error/Fresnel integrals), 7.4.14-7.4.32, transcribed in full.\"}, {\"page_id\": \"as_p0312\", \"text\": \"ERROR FUNCTION AND FRESNEL INTEGRALS (p.312)\\nTable 7.2  DERIVATIVE OF THE ERROR FUNCTION\\nColumns: x | (2/\\\\sqrt\\\\pi)e^{-x^2}, printed as a mantissa with a bracketed power-of-ten exponent, e.g. \\\"(-2)2.0666985\\\" meaning 2.0666985\\\\times10^{-2}.\\nSample rows:\\n2.00  2.0666\\\\,985\\\\times10^{-2}\\n2.50  2.1782\\\\,842\\\\times10^{-3}\\n3.00  1.3925\\\\,305\\\\times10^{-4}\\n3.50  5.3994\\\\,268\\\\times10^{-6}\\n4.00  1.2698\\\\,235\\\\times10^{-7}\\n(Table runs x=2.00 to 4.00 in steps of 0.01, printed as four side-by-side blocks; only a sampled subset of rows is given here.)\\n\\\\frac{\\\\sqrt\\\\pi}{2}=0.88622\\\\,69255\", \"chapter_id\": \"ch07_error_fresnel\", \"printed_page\": 312, \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0312.png\", \"note\": \"Dense numeric-table page (Ch.7 Table 7.2, derivative of the error function, 201 rows). Gold = structural headers + sampled rows, same convention as as_p0243.\"}, {\"page_id\": \"as_p0318\", \"printed_page\": 318, \"chapter_id\": \"ch07_error_fresnel\", \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0318.png\", \"note\": \"BLANK-SLATE page: the repaired reader emitted Nougat's [MISSING_PAGE] marker, so there was no draft to correct -- transcribed directly from the scan. Dense numeric-table page (Ch.7, Table 7.4). Gold = structural headers + the defining expression + the normalisation row at the foot + sampled rows, per the same convention as gold pages as_p0242/as_p0243. Entries use A&S's bracketed power-of-ten notation, e.g. (-1)7.13475 means 7.13475e-1.\", \"text\": \"ERROR FUNCTION AND FRESNEL INTEGRALS (p.318)\\nTable 7.4  REPEATED INTEGRALS OF THE ERROR FUNCTION\\nDefining expression: 2^n\\\\Gamma\\\\left(\\\\frac{n}{2}+1\\\\right)i^n\\\\mathrm{erfc}\\\\,x\\nColumns: x | n=5 | n=6 | n=10 | n=11\\nSample rows:\\n0.0  1.00000       1.00000       1.00000       1.00000\\n0.1  (-1)7.13475   (-1)6.93283   (-1)6.28971   (-1)6.15727\\n0.5  (-1)1.65569   (-1)1.43588   (-2)8.84744   (-2)7.95749\\n1.0  (-2)2.03707   (-2)1.53850   (-3)5.90062   (-3)4.78106\\n1.5  (-3)1.80252   (-3)1.19278   (-4)2.89186   (-4)2.11641\\n2.0  (-4)1.11492   (-5)6.51088   (-5)1.01722   (-6)6.74666\\n2.5  (-6)4.70641   (-6)2.44418   (-7)2.51397   (-7)1.51693\\n3.0  (-7)1.32935   (-8)6.18684   (-9)4.28380   (-9)2.36143\\n3.5  (-9)2.47236   (-9)1.03880   (-11)4.95086  (-11)2.50393\\n4.0  (-11)2.98854  (-11)1.14149  (-13)3.82601  (-13)1.78294\\n4.5  (-13)2.32332  (-14)8.11851  (-15)1.95316  (-16)8.42124\\n5.0  (-15)1.15173  (-16)3.70336  (-18)6.51829  (-18)2.61062\\nNormalisation row at the foot of the page, \\\\left[2^n\\\\Gamma\\\\left(\\\\frac{n}{2}+1\\\\right)\\\\right]^{-1}:\\n(-3)9.40315\\\\,97258   (-3)2.60416\\\\,66667   (-6)8.13802\\\\,08333   (-6)1.69609\\\\,66316\\n(The full table runs x=0.0(0.1)5.0 in 51 rows across the four columns above; rows were sampled at 0.5 intervals, not transcribed exhaustively.)\"}, {\"page_id\": \"as_p0324\", \"printed_page\": 324, \"chapter_id\": \"ch07_error_fresnel\", \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0324.png\", \"note\": \"BLANK-SLATE page: the repaired reader produced empty output, so there was no draft to correct -- transcribed directly from the scan. Dense numeric-table page (Ch.7, Table 7.8, auxiliary functions for the Fresnel integrals). Gold = structural headers + the defining relations at the foot + sampled rows, per the as_p0242/as_p0243 convention.\", \"text\": \"ERROR FUNCTION AND FRESNEL INTEGRALS (p.324)\\nTable 7.8  AUXILIARY FUNCTIONS\\nColumns: x^{-1} | u^{-1}=\\\\frac{2}{\\\\pi x^2} | f(x)=f_2(u) | g(x)=g_2(u) | \\\\langle x\\\\rangle | \\\\langle u\\\\rangle\\nSample rows:\\n1.00  0.63661\\\\,97723\\\\,67581  0.27989\\\\,34003\\\\,76823  0.06174\\\\,08526\\\\,09645  1  2\\n0.90  0.51566\\\\,20156\\\\,17741  0.25946\\\\,14023\\\\,65674  0.04986\\\\,06317\\\\,93636  1  2\\n0.80  0.40743\\\\,66543\\\\,15252  0.23689\\\\,07256\\\\,57089  0.03856\\\\,20343\\\\,27312  1  2\\n0.70  0.31194\\\\,36884\\\\,60115  0.21214\\\\,23821\\\\,60229  0.02819\\\\,34743\\\\,19381  1  3\\n0.60  0.22918\\\\,31180\\\\,52329  0.18529\\\\,53067\\\\,79209  0.01913\\\\,61240\\\\,35536  2  4\\n0.50  0.15915\\\\,49430\\\\,91895  0.15658\\\\,43216\\\\,36302  0.01174\\\\,65939\\\\,24659  2  6\\n0.40  0.10185\\\\,91635\\\\,78813  0.12640\\\\,69204\\\\,94864  0.00626\\\\,36346\\\\,49122  3  10\\n0.30  0.05729\\\\,57795\\\\,13082  0.09526\\\\,41276\\\\,74844  0.00270\\\\,35642\\\\,68526  3  17\\n0.20  0.02546\\\\,47908\\\\,94703  0.06363\\\\,11887\\\\,04012  0.00080\\\\,86180\\\\,82883  5  39\\n0.10  0.00636\\\\,61977\\\\,23676  0.03183\\\\,00214\\\\,15118  0.00010\\\\,13057\\\\,94484  10  157\\n0.02  0.00025\\\\,46479\\\\,08947  0.00636\\\\,61974\\\\,14061  0.00000\\\\,08105\\\\,69272  50  3927\\n0.00  0.00000\\\\,00000\\\\,00000  0.00000\\\\,00000\\\\,00000  0.00000\\\\,00000\\\\,00000  \\\\infty  \\\\infty\\nInterpolation-difference scale factors printed under the last row: [(-5)6/3], [(-5)1/12], [(-5)1/12]\\nDefining relations (foot of page):\\nC(x)=\\\\frac{1}{2}+f(x)\\\\sin\\\\left(\\\\frac{\\\\pi}{2}x^2\\\\right)-g(x)\\\\cos\\\\left(\\\\frac{\\\\pi}{2}x^2\\\\right) \\\\qquad C_2(u)=\\\\frac{1}{2}+f_2(u)\\\\sin u-g_2(u)\\\\cos u\\nS(x)=\\\\frac{1}{2}-f(x)\\\\cos\\\\left(\\\\frac{\\\\pi}{2}x^2\\\\right)-g(x)\\\\sin\\\\left(\\\\frac{\\\\pi}{2}x^2\\\\right) \\\\qquad S_2(u)=\\\\frac{1}{2}-f_2(u)\\\\cos u-g_2(u)\\\\sin u\\n\\\\langle x\\\\rangle = nearest integer to x.\\n(The full table runs x^{-1}=1.00(0.02)0.00 in 51 rows; rows were sampled at 0.10 intervals, not transcribed exhaustively.)\"}, {\"page_id\": \"as_p0361\", \"printed_page\": 361, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0361.png\", \"note\": \"Two-column dense-formula page (Ch.9 Bessel), 9.1.27-9.1.48: recurrence relations, derivative formulas, cross-product recurrences, analytic continuation, and the generating function with its associated series. Draft existed (5792 chars) and was corrected against the scan.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.361)\\nRecurrence Relations\\n9.1.27  \\\\mathscr{C}_{\\\\nu-1}(z)+\\\\mathscr{C}_{\\\\nu+1}(z)=\\\\frac{2\\\\nu}{z}\\\\mathscr{C}_\\\\nu(z)\\n        \\\\mathscr{C}_{\\\\nu-1}(z)-\\\\mathscr{C}_{\\\\nu+1}(z)=2\\\\mathscr{C}'_\\\\nu(z)\\n        \\\\mathscr{C}'_\\\\nu(z)=\\\\mathscr{C}_{\\\\nu-1}(z)-\\\\frac{\\\\nu}{z}\\\\mathscr{C}_\\\\nu(z)\\n        \\\\mathscr{C}'_\\\\nu(z)=-\\\\mathscr{C}_{\\\\nu+1}(z)+\\\\frac{\\\\nu}{z}\\\\mathscr{C}_\\\\nu(z)\\n\\\\mathscr{C} denotes J, Y, H^{(1)}, H^{(2)} or any linear combination of these functions, the coefficients in which are independent of z and \\\\nu.\\n9.1.28  J'_0(z)=-J_1(z) \\\\qquad Y'_0(z)=-Y_1(z)\\nIf f_\\\\nu(z)=z^p\\\\mathscr{C}_\\\\nu(\\\\lambda z^q) where p, q, \\\\lambda are independent of \\\\nu, then\\n9.1.29  f_{\\\\nu-1}(z)+f_{\\\\nu+1}(z)=(2\\\\nu/\\\\lambda)z^{-q}f_\\\\nu(z)\\n        (p+\\\\nu q)f_{\\\\nu-1}(z)+(p-\\\\nu q)f_{\\\\nu+1}(z)=(2\\\\nu/\\\\lambda)z^{1-q}f'_\\\\nu(z)\\n        zf'_\\\\nu(z)=\\\\lambda qz^q f_{\\\\nu-1}(z)+(p-\\\\nu q)f_\\\\nu(z)\\n        zf'_\\\\nu(z)=-\\\\lambda qz^q f_{\\\\nu+1}(z)+(p+\\\\nu q)f_\\\\nu(z)\\nFormulas for Derivatives\\n9.1.30  \\\\left(\\\\frac{1}{z}\\\\frac{d}{dz}\\\\right)^k\\\\{z^\\\\nu\\\\mathscr{C}_\\\\nu(z)\\\\}=z^{\\\\nu-k}\\\\mathscr{C}_{\\\\nu-k}(z)\\n        \\\\left(\\\\frac{1}{z}\\\\frac{d}{dz}\\\\right)^k\\\\{z^{-\\\\nu}\\\\mathscr{C}_\\\\nu(z)\\\\}=(-)^k z^{-\\\\nu-k}\\\\mathscr{C}_{\\\\nu+k}(z) \\\\quad (k=0,1,2,\\\\dots)\\n9.1.31  \\\\mathscr{C}^{(k)}_\\\\nu(z)=\\\\frac{1}{2^k}\\\\{\\\\mathscr{C}_{\\\\nu-k}(z)-\\\\binom{k}{1}\\\\mathscr{C}_{\\\\nu-k+2}(z)+\\\\binom{k}{2}\\\\mathscr{C}_{\\\\nu-k+4}(z)-\\\\dots+(-)^k\\\\mathscr{C}_{\\\\nu+k}(z)\\\\} \\\\quad (k=0,1,2,\\\\dots)\\nRecurrence Relations for Cross-Products\\nIf\\n9.1.32  p_\\\\nu=J_\\\\nu(a)Y_\\\\nu(b)-J_\\\\nu(b)Y_\\\\nu(a)\\n        q_\\\\nu=J_\\\\nu(a)Y'_\\\\nu(b)-J'_\\\\nu(b)Y_\\\\nu(a)\\n        r_\\\\nu=J'_\\\\nu(a)Y_\\\\nu(b)-J_\\\\nu(b)Y'_\\\\nu(a)\\n        s_\\\\nu=J'_\\\\nu(a)Y'_\\\\nu(b)-J'_\\\\nu(b)Y'_\\\\nu(a)\\nthen\\n9.1.33  p_{\\\\nu+1}-p_{\\\\nu-1}=-\\\\frac{2\\\\nu}{a}q_\\\\nu-\\\\frac{2\\\\nu}{b}r_\\\\nu\\n        q_{\\\\nu+1}+r_\\\\nu=\\\\frac{\\\\nu}{a}p_\\\\nu-\\\\frac{\\\\nu+1}{b}p_{\\\\nu+1}\\n        r_{\\\\nu+1}+q_\\\\nu=\\\\frac{\\\\nu}{b}p_\\\\nu-\\\\frac{\\\\nu+1}{a}p_{\\\\nu+1}\\n        s_\\\\nu=\\\\frac{1}{2}p_{\\\\nu+1}+\\\\frac{1}{2}p_{\\\\nu-1}-\\\\frac{\\\\nu^2}{ab}p_\\\\nu\\nand\\n9.1.34  p_\\\\nu s_\\\\nu-q_\\\\nu r_\\\\nu=\\\\frac{4}{\\\\pi^2 ab}\\nAnalytic Continuation\\nIn 9.1.35 to 9.1.38, m is an integer.\\n9.1.35  J_\\\\nu(ze^{m\\\\pi i})=e^{m\\\\nu\\\\pi i}J_\\\\nu(z)\\n9.1.36  Y_\\\\nu(ze^{m\\\\pi i})=e^{-m\\\\nu\\\\pi i}Y_\\\\nu(z)+2i\\\\sin(m\\\\nu\\\\pi)\\\\cot(\\\\nu\\\\pi)J_\\\\nu(z)\\n9.1.37  \\\\sin(\\\\nu\\\\pi)H^{(1)}_\\\\nu(ze^{m\\\\pi i})=-\\\\sin\\\\{(m-1)\\\\nu\\\\pi\\\\}H^{(1)}_\\\\nu(z)-e^{-\\\\nu\\\\pi i}\\\\sin(m\\\\nu\\\\pi)H^{(2)}_\\\\nu(z)\\n9.1.38  \\\\sin(\\\\nu\\\\pi)H^{(2)}_\\\\nu(ze^{m\\\\pi i})=\\\\sin\\\\{(m+1)\\\\nu\\\\pi\\\\}H^{(2)}_\\\\nu(z)+e^{\\\\nu\\\\pi i}\\\\sin(m\\\\nu\\\\pi)H^{(1)}_\\\\nu(z)\\n9.1.39  H^{(1)}_\\\\nu(ze^{\\\\pi i})=-e^{-\\\\nu\\\\pi i}H^{(2)}_\\\\nu(z)\\n        H^{(2)}_\\\\nu(ze^{-\\\\pi i})=-e^{\\\\nu\\\\pi i}H^{(1)}_\\\\nu(z)\\n9.1.40  J_\\\\nu(\\\\bar z)=\\\\overline{J_\\\\nu(z)} \\\\qquad Y_\\\\nu(\\\\bar z)=\\\\overline{Y_\\\\nu(z)}\\n        H^{(1)}_\\\\nu(\\\\bar z)=\\\\overline{H^{(2)}_\\\\nu(z)} \\\\qquad H^{(2)}_\\\\nu(\\\\bar z)=\\\\overline{H^{(1)}_\\\\nu(z)} \\\\quad (\\\\nu\\\\ \\\\mathrm{real})\\nGenerating Function and Associated Series\\n9.1.41  e^{\\\\frac{1}{2}z(t-1/t)}=\\\\sum_{k=-\\\\infty}^{\\\\infty}t^k J_k(z) \\\\quad (t\\\\neq 0)\\n9.1.42  \\\\cos(z\\\\sin\\\\theta)=J_0(z)+2\\\\sum_{k=1}^{\\\\infty}J_{2k}(z)\\\\cos(2k\\\\theta)\\n9.1.43  \\\\sin(z\\\\sin\\\\theta)=2\\\\sum_{k=0}^{\\\\infty}J_{2k+1}(z)\\\\sin\\\\{(2k+1)\\\\theta\\\\}\\n9.1.44  \\\\cos(z\\\\cos\\\\theta)=J_0(z)+2\\\\sum_{k=1}^{\\\\infty}(-)^k J_{2k}(z)\\\\cos(2k\\\\theta)\\n9.1.45  \\\\sin(z\\\\cos\\\\theta)=2\\\\sum_{k=0}^{\\\\infty}(-)^k J_{2k+1}(z)\\\\cos\\\\{(2k+1)\\\\theta\\\\}\\n9.1.46  1=J_0(z)+2J_2(z)+2J_4(z)+2J_6(z)+\\\\dots\\n9.1.47  \\\\cos z=J_0(z)-2J_2(z)+2J_4(z)-2J_6(z)+\\\\dots\\n9.1.48  \\\\sin z=2J_1(z)-2J_3(z)+2J_5(z)-\\\\dots\"}, {\"page_id\": \"as_p0362\", \"printed_page\": 362, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0362.png\", \"note\": \"Two-column dense-formula page (Ch.9 Bessel), 9.1.49-9.1.71: other differential equations, differential equations for products, upper bounds, derivatives with respect to order, hypergeometric expressions, and the Legendre-function connection. Draft existed (1588 chars, badly truncated) and was corrected/completed against the scan.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.362)\\nOther Differential Equations\\n9.1.49  w''+\\\\left(\\\\lambda^2-\\\\frac{\\\\nu^2-\\\\frac14}{z^2}\\\\right)w=0, \\\\qquad w=z^{\\\\frac12}\\\\mathscr{C}_\\\\nu(\\\\lambda z)\\n9.1.50  w''+\\\\left(\\\\frac{\\\\lambda^2}{4z}-\\\\frac{\\\\nu^2-1}{4z^2}\\\\right)w=0, \\\\qquad w=z^{\\\\frac12}\\\\mathscr{C}_\\\\nu(\\\\lambda z^{\\\\frac12})\\n9.1.51  w''+\\\\lambda^2 z^{p-2}w=0, \\\\qquad w=z^{\\\\frac12}\\\\mathscr{C}_{1/p}(2\\\\lambda z^{\\\\frac12 p}/p)\\n9.1.52  w''-\\\\frac{2\\\\nu-1}{z}w'+\\\\lambda^2 w=0, \\\\qquad w=z^\\\\nu\\\\mathscr{C}_\\\\nu(\\\\lambda z)\\n9.1.53  z^2w''+(1-2p)zw'+(\\\\lambda^2q^2z^{2q}+p^2-\\\\nu^2q^2)w=0, \\\\qquad w=z^p\\\\mathscr{C}_\\\\nu(\\\\lambda z^q)\\n9.1.54  w''+(\\\\lambda^2e^{2z}-\\\\nu^2)w=0, \\\\qquad w=\\\\mathscr{C}_\\\\nu(\\\\lambda e^z)\\n9.1.55  z^2(z^2-\\\\nu^2)w''+z(z^2-3\\\\nu^2)w'+\\\\{(z^2-\\\\nu^2)^2-(z^2+\\\\nu^2)\\\\}w=0, \\\\qquad w=\\\\mathscr{C}'_\\\\nu(z)\\n9.1.56  w^{(2n)}=(-)^n\\\\lambda^{2n}z^{-n}w, \\\\qquad w=z^{\\\\frac12 n}\\\\mathscr{C}_n(2\\\\lambda\\\\alpha z^{\\\\frac12})\\nwhere \\\\alpha is any of the 2n roots of unity.\\nDifferential Equations for Products\\nIn the following \\\\vartheta\\\\equiv z\\\\frac{d}{dz} and \\\\mathscr{C}_\\\\nu(z), \\\\mathscr{D}_\\\\mu(z) are any cylinder functions of orders \\\\nu, \\\\mu respectively.\\n9.1.57  \\\\{\\\\vartheta^4-2(\\\\nu^2+\\\\mu^2)\\\\vartheta^2+(\\\\nu^2-\\\\mu^2)^2\\\\}w+4z^2(\\\\vartheta+1)(\\\\vartheta+2)w=0, \\\\qquad w=\\\\mathscr{C}_\\\\nu(z)\\\\mathscr{D}_\\\\mu(z)\\n9.1.58  \\\\vartheta(\\\\vartheta^2-4\\\\nu^2)w+4z^2(\\\\vartheta+1)w=0, \\\\qquad w=\\\\mathscr{C}_\\\\nu(z)\\\\mathscr{D}_\\\\nu(z)\\n9.1.59  z^3w'''+z(4z^2+1-4\\\\nu^2)w'+(4\\\\nu^2-1)w=0, \\\\qquad w=z\\\\mathscr{C}_\\\\nu(z)\\\\mathscr{D}_\\\\nu(z)\\nUpper Bounds\\n9.1.60  |J_\\\\nu(x)|\\\\le 1\\\\ (\\\\nu\\\\ge 0),\\\\ |J_\\\\nu(x)|\\\\le 1/\\\\sqrt2 \\\\quad (\\\\nu\\\\ge 1)\\n9.1.61  0<J_\\\\nu(\\\\nu)<\\\\frac{2^{\\\\frac13}}{3^{\\\\frac13}\\\\Gamma(\\\\frac23)\\\\nu^{\\\\frac13}} \\\\quad (\\\\nu>0)\\n9.1.62  |J_\\\\nu(z)|\\\\le\\\\frac{|\\\\frac12 z|^\\\\nu e^{|\\\\mathscr{I}z|}}{\\\\Gamma(\\\\nu+1)} \\\\quad (\\\\nu\\\\ge-\\\\tfrac12)\\n9.1.63  |J_n(nz)|\\\\le\\\\left|\\\\frac{z^n\\\\exp\\\\{n\\\\sqrt{(1-z^2)}\\\\}}{\\\\{1+\\\\sqrt{(1-z^2)}\\\\}^n}\\\\right|\\nDerivatives With Respect to Order\\n9.1.64  \\\\frac{\\\\partial}{\\\\partial\\\\nu}J_\\\\nu(z)=J_\\\\nu(z)\\\\ln(\\\\tfrac12 z)-(\\\\tfrac12 z)^\\\\nu\\\\sum_{k=0}^{\\\\infty}(-)^k\\\\frac{\\\\psi(\\\\nu+k+1)}{\\\\Gamma(\\\\nu+k+1)}\\\\frac{(\\\\frac14 z^2)^k}{k!}\\n9.1.65  \\\\frac{\\\\partial}{\\\\partial\\\\nu}Y_\\\\nu(z)=\\\\cot(\\\\nu\\\\pi)\\\\{\\\\frac{\\\\partial}{\\\\partial\\\\nu}J_\\\\nu(z)-\\\\pi Y_\\\\nu(z)\\\\}-\\\\csc(\\\\nu\\\\pi)\\\\frac{\\\\partial}{\\\\partial\\\\nu}J_{-\\\\nu}(z)-\\\\pi J_\\\\nu(z) \\\\quad (\\\\nu\\\\neq 0,\\\\pm1,\\\\pm2,\\\\dots)\\n9.1.66  \\\\left[\\\\frac{\\\\partial}{\\\\partial\\\\nu}J_\\\\nu(z)\\\\right]_{\\\\nu=n}=\\\\frac{\\\\pi}{2}Y_n(z)+\\\\frac{n!(\\\\frac12 z)^{-n}}{2}\\\\sum_{k=0}^{n-1}\\\\frac{(\\\\frac12 z)^kJ_k(z)}{(n-k)k!}\\n9.1.67  \\\\left[\\\\frac{\\\\partial}{\\\\partial\\\\nu}Y_\\\\nu(z)\\\\right]_{\\\\nu=n}=-\\\\frac{\\\\pi}{2}J_n(z)+\\\\frac{n!(\\\\frac12 z)^{-n}}{2}\\\\sum_{k=0}^{n-1}\\\\frac{(\\\\frac12 z)^kY_k(z)}{(n-k)k!}\\n9.1.68  \\\\left[\\\\frac{\\\\partial}{\\\\partial\\\\nu}J_\\\\nu(z)\\\\right]_{\\\\nu=0}=\\\\frac{\\\\pi}{2}Y_0(z), \\\\left[\\\\frac{\\\\partial}{\\\\partial\\\\nu}Y_\\\\nu(z)\\\\right]_{\\\\nu=0}=-\\\\frac{\\\\pi}{2}J_0(z)\\nExpressions in Terms of Hypergeometric Functions\\n9.1.69  J_\\\\nu(z)=\\\\frac{(\\\\frac12 z)^\\\\nu}{\\\\Gamma(\\\\nu+1)}{}_0F_1(\\\\nu+1;-\\\\tfrac14 z^2)=\\\\frac{(\\\\frac12 z)^\\\\nu e^{-iz}}{\\\\Gamma(\\\\nu+1)}M(\\\\nu+\\\\tfrac12,2\\\\nu+1,2iz)\\n9.1.70  J_\\\\nu(z)=\\\\frac{(\\\\frac12 z)^\\\\nu}{\\\\Gamma(\\\\nu+1)}\\\\lim F\\\\left(\\\\lambda,\\\\mu;\\\\nu+1;-\\\\frac{z^2}{4\\\\lambda\\\\mu}\\\\right)\\nas \\\\lambda, \\\\mu\\\\to\\\\infty through real or complex values; z, \\\\nu being fixed.\\n({}_0F_1 is the generalized hypergeometric function. For M(a,b,z) and F(a,b;c;z) see chapters 13 and 15.)\\nConnection With Legendre Functions\\nIf \\\\mu and x are fixed and \\\\nu\\\\to\\\\infty through real positive values\\n9.1.71  \\\\lim\\\\{\\\\nu^\\\\mu P_\\\\nu^{-\\\\mu}(\\\\cos\\\\tfrac{x}{\\\\nu})\\\\}=J_\\\\mu(x) \\\\quad (x>0)\"}, {\"page_id\": \"as_p0366\", \"printed_page\": 366, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"prose+formula\", \"image\": \"grading_kit/heldout_pages/as_p0366.png\", \"note\": \"Two-column formula page with connecting prose (Ch.9 Bessel), 9.3.3-9.3.17: uniform asymptotic expansions, Airy-function forms, and Debye's asymptotic expansions with the u_k / v_k polynomial coefficients. A&S prints the long numeric coefficients with thin-space digit grouping (e.g. 3 69603); kept as \\\\, per the convention of the original gold pages. Draft existed (3214 chars) and was corrected against the scan.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.366)\\n9.3.3   J_\\\\nu(\\\\nu\\\\sec\\\\beta)=\\\\sqrt{2/(\\\\pi\\\\nu\\\\tan\\\\beta)}\\\\{\\\\cos(\\\\nu\\\\tan\\\\beta-\\\\nu\\\\beta-\\\\tfrac14\\\\pi)+O(\\\\nu^{-1})\\\\} \\\\quad (0<\\\\beta<\\\\tfrac12\\\\pi)\\n        Y_\\\\nu(\\\\nu\\\\sec\\\\beta)=\\\\sqrt{2/(\\\\pi\\\\nu\\\\tan\\\\beta)}\\\\{\\\\sin(\\\\nu\\\\tan\\\\beta-\\\\nu\\\\beta-\\\\tfrac14\\\\pi)+O(\\\\nu^{-1})\\\\} \\\\quad (0<\\\\beta<\\\\tfrac12\\\\pi)\\n9.3.4   J_\\\\nu(\\\\nu+z\\\\nu^{\\\\frac13})=2^{\\\\frac13}\\\\nu^{-\\\\frac13}\\\\mathrm{Ai}(-2^{\\\\frac13}z)+O(\\\\nu^{-1})\\n        Y_\\\\nu(\\\\nu+z\\\\nu^{\\\\frac13})=-2^{\\\\frac13}\\\\nu^{-\\\\frac13}\\\\mathrm{Bi}(-2^{\\\\frac13}z)+O(\\\\nu^{-1})\\n9.3.5   J_\\\\nu(\\\\nu)\\\\sim\\\\frac{2^{\\\\frac13}}{3^{\\\\frac23}\\\\Gamma(\\\\frac23)}\\\\frac{1}{\\\\nu^{\\\\frac13}}\\n        Y_\\\\nu(\\\\nu)\\\\sim-\\\\frac{2^{\\\\frac13}}{3^{\\\\frac16}\\\\Gamma(\\\\frac23)}\\\\frac{1}{\\\\nu^{\\\\frac13}}\\n9.3.6   J_\\\\nu(\\\\nu z)=\\\\left(\\\\frac{4\\\\zeta}{1-z^2}\\\\right)^{\\\\frac14}\\\\{\\\\frac{\\\\mathrm{Ai}(\\\\nu^{\\\\frac23}\\\\zeta)}{\\\\nu^{\\\\frac13}}+\\\\frac{\\\\exp(-\\\\frac23\\\\nu\\\\zeta^{\\\\frac32})}{1+\\\\nu^{\\\\frac16}|\\\\zeta|^{\\\\frac14}}O\\\\left(\\\\frac{1}{\\\\nu^{\\\\frac43}}\\\\right)\\\\} \\\\quad (|\\\\arg z|<\\\\pi)\\n        Y_\\\\nu(\\\\nu z)=-\\\\left(\\\\frac{4\\\\zeta}{1-z^2}\\\\right)^{\\\\frac14}\\\\{\\\\frac{\\\\mathrm{Bi}(\\\\nu^{\\\\frac23}\\\\zeta)}{\\\\nu^{\\\\frac13}}+\\\\frac{\\\\exp|\\\\mathscr{R}(\\\\frac23\\\\nu\\\\zeta^{\\\\frac32})|}{1+\\\\nu^{\\\\frac16}|\\\\zeta|^{\\\\frac14}}O\\\\left(\\\\frac{1}{\\\\nu^{\\\\frac43}}\\\\right)\\\\} \\\\quad (|\\\\arg z|<\\\\pi)\\nIn the last two equations \\\\zeta is given by 9.3.38 and 9.3.39 below.\\nDebye's Asymptotic Expansions\\n(i) If \\\\alpha is fixed and positive and \\\\nu is large and positive\\n9.3.7   J_\\\\nu(\\\\nu\\\\,\\\\mathrm{sech}\\\\,\\\\alpha)\\\\sim\\\\frac{e^{\\\\nu(\\\\tanh\\\\alpha-\\\\alpha)}}{\\\\sqrt{2\\\\pi\\\\nu\\\\tanh\\\\alpha}}\\\\{1+\\\\sum_{k=1}^{\\\\infty}\\\\frac{u_k(\\\\coth\\\\alpha)}{\\\\nu^k}\\\\}\\n9.3.8   Y_\\\\nu(\\\\nu\\\\,\\\\mathrm{sech}\\\\,\\\\alpha)\\\\sim-\\\\frac{e^{\\\\nu(\\\\alpha-\\\\tanh\\\\alpha)}}{\\\\sqrt{\\\\frac12\\\\pi\\\\nu\\\\tanh\\\\alpha}}\\\\{1+\\\\sum_{k=1}^{\\\\infty}(-)^k\\\\frac{u_k(\\\\coth\\\\alpha)}{\\\\nu^k}\\\\}\\nwhere\\n9.3.9   u_0(t)=1\\n        u_1(t)=(3t-5t^3)/24\\n        u_2(t)=(81t^2-462t^4+385t^6)/1152\\n        u_3(t)=(30375t^3-3\\\\,69603t^5+7\\\\,65765t^7-4\\\\,25425t^9)/4\\\\,14720\\n        u_4(t)=(44\\\\,65125t^4-941\\\\,21676t^6+3499\\\\,22430t^8-4461\\\\,85740t^{10}+1859\\\\,10725t^{12})/398\\\\,13120\\nFor u_5(t) and u_6(t) see [9.4] or [9.21].\\n9.3.10  u_{k+1}(t)=\\\\tfrac12 t^2(1-t^2)u'_k(t)+\\\\frac18\\\\int_0^t(1-5t^2)u_k(t)dt \\\\quad (k=0,1,\\\\dots)\\nAlso\\n9.3.11  J'_\\\\nu(\\\\nu\\\\,\\\\mathrm{sech}\\\\,\\\\alpha)\\\\sim\\\\sqrt{\\\\frac{\\\\sinh 2\\\\alpha}{4\\\\pi\\\\nu}}e^{\\\\nu(\\\\tanh\\\\alpha-\\\\alpha)}\\\\{1+\\\\sum_{k=1}^{\\\\infty}\\\\frac{v_k(\\\\coth\\\\alpha)}{\\\\nu^k}\\\\}\\n9.3.12  Y'_\\\\nu(\\\\nu\\\\,\\\\mathrm{sech}\\\\,\\\\alpha)\\\\sim\\\\sqrt{\\\\frac{\\\\sinh 2\\\\alpha}{\\\\pi\\\\nu}}e^{\\\\nu(\\\\alpha-\\\\tanh\\\\alpha)}\\\\{1+\\\\sum_{k=1}^{\\\\infty}(-)^k\\\\frac{v_k(\\\\coth\\\\alpha)}{\\\\nu^k}\\\\}\\nwhere\\n9.3.13  v_0(t)=1\\n        v_1(t)=(-9t+7t^3)/24\\n        v_2(t)=(-135t^2+594t^4-455t^6)/1152\\n        v_3(t)=(-42525t^3+4\\\\,51737t^5-8\\\\,83575t^7+4\\\\,75475t^9)/4\\\\,14720\\n9.3.14  v_k(t)=u_k(t)+t(t^2-1)\\\\{\\\\tfrac12 u_{k-1}(t)+tu'_{k-1}(t)\\\\} \\\\quad (k=1,2,\\\\dots)\\n(ii) If \\\\beta is fixed, 0<\\\\beta<\\\\tfrac12\\\\pi and \\\\nu is large and positive\\n9.3.15  J_\\\\nu(\\\\nu\\\\sec\\\\beta)=\\\\sqrt{2/(\\\\pi\\\\nu\\\\tan\\\\beta)}\\\\{L(\\\\nu,\\\\beta)\\\\cos\\\\Psi+M(\\\\nu,\\\\beta)\\\\sin\\\\Psi\\\\}\\n9.3.16  Y_\\\\nu(\\\\nu\\\\sec\\\\beta)=\\\\sqrt{2/(\\\\pi\\\\nu\\\\tan\\\\beta)}\\\\{L(\\\\nu,\\\\beta)\\\\sin\\\\Psi-M(\\\\nu,\\\\beta)\\\\cos\\\\Psi\\\\}\\nwhere \\\\Psi=\\\\nu(\\\\tan\\\\beta-\\\\beta)-\\\\tfrac14\\\\pi\\n9.3.17  L(\\\\nu,\\\\beta)\\\\sim\\\\sum_{k=0}^{\\\\infty}\\\\frac{u_{2k}(i\\\\cot\\\\beta)}{\\\\nu^{2k}}=1-\\\\frac{81\\\\cot^2\\\\beta+462\\\\cot^4\\\\beta+385\\\\cot^6\\\\beta}{1152\\\\nu^2}+\\\\dots\"}, {\"page_id\": \"as_p0367\", \"printed_page\": 367, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"prose+formula\", \"image\": \"grading_kit/heldout_pages/as_p0367.png\", \"note\": \"Two-column formula page with connecting prose (Ch.9 Bessel), 9.3.18-9.3.30: Debye expansions for the derivatives, then asymptotic expansions in the transition regions with the f_k / g_k / h_k / l_k rational-coefficient polynomials. Draft existed (1774 chars, truncated) and was corrected/completed against the scan.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.367)\\n9.3.18  M(\\\\nu,\\\\beta)\\\\sim-i\\\\sum_{k=0}^{\\\\infty}\\\\frac{u_{2k+1}(i\\\\cot\\\\beta)}{\\\\nu^{2k+1}}=\\\\frac{3\\\\cot\\\\beta+5\\\\cot^3\\\\beta}{24\\\\nu}-\\\\dots\\nAlso\\n9.3.19  J'_\\\\nu(\\\\nu\\\\sec\\\\beta)=\\\\sqrt{(\\\\sin 2\\\\beta)/(\\\\pi\\\\nu)}\\\\{-N(\\\\nu,\\\\beta)\\\\sin\\\\Psi-O(\\\\nu,\\\\beta)\\\\cos\\\\Psi\\\\}\\n9.3.20  Y'_\\\\nu(\\\\nu\\\\sec\\\\beta)=\\\\sqrt{(\\\\sin 2\\\\beta)/(\\\\pi\\\\nu)}\\\\{N(\\\\nu,\\\\beta)\\\\cos\\\\Psi-O(\\\\nu,\\\\beta)\\\\sin\\\\Psi\\\\}\\nwhere\\n9.3.21  N(\\\\nu,\\\\beta)\\\\sim\\\\sum_{k=0}^{\\\\infty}\\\\frac{v_{2k}(i\\\\cot\\\\beta)}{\\\\nu^{2k}}=1+\\\\frac{135\\\\cot^2\\\\beta+594\\\\cot^4\\\\beta+455\\\\cot^6\\\\beta}{1152\\\\nu^2}-\\\\dots\\n9.3.22  O(\\\\nu,\\\\beta)\\\\sim i\\\\sum_{k=0}^{\\\\infty}\\\\frac{v_{2k+1}(i\\\\cot\\\\beta)}{\\\\nu^{2k+1}}=\\\\frac{9\\\\cot\\\\beta+7\\\\cot^3\\\\beta}{24\\\\nu}-\\\\dots\\nAsymptotic Expansions in the Transition Regions\\nWhen z is fixed, |\\\\nu| is large and |\\\\arg\\\\nu|<\\\\tfrac12\\\\pi\\n9.3.23  J_\\\\nu(\\\\nu+z\\\\nu^{1/3})\\\\sim\\\\frac{2^{1/3}}{\\\\nu^{1/3}}\\\\mathrm{Ai}(-2^{1/3}z)\\\\{1+\\\\sum_{k=1}^{\\\\infty}\\\\frac{f_k(z)}{\\\\nu^{2k/3}}\\\\}+\\\\frac{2^{2/3}}{\\\\nu}\\\\mathrm{Ai}'(-2^{1/3}z)\\\\sum_{k=0}^{\\\\infty}\\\\frac{g_k(z)}{\\\\nu^{2k/3}}\\n9.3.24  Y_\\\\nu(\\\\nu+z\\\\nu^{1/3})\\\\sim-\\\\frac{2^{1/3}}{\\\\nu^{1/3}}\\\\mathrm{Bi}(-2^{1/3}z)\\\\{1+\\\\sum_{k=1}^{\\\\infty}\\\\frac{f_k(z)}{\\\\nu^{2k/3}}\\\\}-\\\\frac{2^{2/3}}{\\\\nu}\\\\mathrm{Bi}'(-2^{1/3}z)\\\\sum_{k=0}^{\\\\infty}\\\\frac{g_k(z)}{\\\\nu^{2k/3}}\\nwhere\\n9.3.25  f_1(z)=-\\\\frac{1}{5}z\\n        f_2(z)=-\\\\frac{9}{100}z^5+\\\\frac{3}{35}z^2\\n        f_3(z)=\\\\frac{957}{7000}z^6-\\\\frac{173}{3150}z^3-\\\\frac{1}{225}\\n        f_4(z)=\\\\frac{27}{20000}z^{10}-\\\\frac{23573}{147000}z^7+\\\\frac{5903}{138600}z^4+\\\\frac{947}{346500}z\\n9.3.26  g_0(z)=\\\\frac{3}{10}z^2\\n        g_1(z)=-\\\\frac{17}{70}z^3+\\\\frac{1}{70}\\n        g_2(z)=-\\\\frac{9}{1000}z^7+\\\\frac{611}{3150}z^4-\\\\frac{37}{3150}z\\n        g_3(z)=\\\\frac{549}{28000}z^8-\\\\frac{110767}{693000}z^5+\\\\frac{79}{12375}z^2\\nThe corresponding expansions for H^{(1)}_\\\\nu(\\\\nu+z\\\\nu^{1/3}) and H^{(2)}_\\\\nu(\\\\nu+z\\\\nu^{1/3}) are obtained by use of 9.1.3 and 9.1.4; they are valid for -\\\\tfrac12\\\\pi<\\\\arg\\\\nu<\\\\tfrac32\\\\pi and -\\\\tfrac32\\\\pi<\\\\arg\\\\nu<\\\\tfrac12\\\\pi, respectively.\\n9.3.27  J'_\\\\nu(\\\\nu+z\\\\nu^{1/3})\\\\sim-\\\\frac{2^{2/3}}{\\\\nu^{2/3}}\\\\mathrm{Ai}'(-2^{1/3}z)\\\\{1+\\\\sum_{k=1}^{\\\\infty}\\\\frac{h_k(z)}{\\\\nu^{2k/3}}\\\\}+\\\\frac{2^{1/3}}{\\\\nu^{4/3}}\\\\mathrm{Ai}(-2^{1/3}z)\\\\sum_{k=0}^{\\\\infty}\\\\frac{l_k(z)}{\\\\nu^{2k/3}}\\n9.3.28  Y'_\\\\nu(\\\\nu+z\\\\nu^{1/3})\\\\sim\\\\frac{2^{2/3}}{\\\\nu^{2/3}}\\\\mathrm{Bi}'(-2^{1/3}z)\\\\{1+\\\\sum_{k=1}^{\\\\infty}\\\\frac{h_k(z)}{\\\\nu^{2k/3}}\\\\}-\\\\frac{2^{1/3}}{\\\\nu^{4/3}}\\\\mathrm{Bi}(-2^{1/3}z)\\\\sum_{k=0}^{\\\\infty}\\\\frac{l_k(z)}{\\\\nu^{2k/3}}\\nwhere\\n9.3.29  h_1(z)=-\\\\frac{4}{5}z\\n        h_2(z)=-\\\\frac{9}{100}z^5+\\\\frac{57}{70}z^2\\n        h_3(z)=\\\\frac{699}{3500}z^6-\\\\frac{2617}{3150}z^3+\\\\frac{23}{3150}\\n        h_4(z)=\\\\frac{27}{20000}z^{10}-\\\\frac{46631}{147000}z^7+\\\\frac{3889}{4620}z^4-\\\\frac{1159}{115500}z\\n9.3.30  l_0(z)=\\\\frac{3}{5}z^3-\\\\frac{1}{5}\\n        l_1(z)=-\\\\frac{131}{140}z^4+\\\\frac{1}{5}z\\n        l_2(z)=-\\\\frac{9}{500}z^8+\\\\frac{5437}{4500}z^5-\\\\frac{593}{3150}z^2\\n        l_3(z)=\\\\frac{369}{7000}z^9-\\\\frac{999443}{693000}z^6+\\\\frac{31727}{173250}z^3+\\\\frac{947}{346500}\"}, {\"page_id\": \"as_p0368\", \"printed_page\": 368, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"prose+formula\", \"image\": \"grading_kit/heldout_pages/as_p0368.png\", \"note\": \"Two-column formula page with connecting prose (Ch.9 Bessel), 9.3.31-9.3.42: expansions at z=nu with their numeric alpha/beta/gamma/delta coefficients, then the uniform asymptotic (Airy-type) expansions and their a_k/b_k coefficients. Draft existed (4105 chars) and was corrected against the scan.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.368)\\n9.3.31  J_\\\\nu(\\\\nu)\\\\sim\\\\frac{a}{\\\\nu^{1/3}}\\\\{1+\\\\sum_{k=1}^{\\\\infty}\\\\frac{\\\\alpha_k}{\\\\nu^{2k}}\\\\}-\\\\frac{b}{\\\\nu^{5/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{\\\\beta_k}{\\\\nu^{2k}}\\n9.3.32  Y_\\\\nu(\\\\nu)\\\\sim-\\\\frac{3^{1/2}a}{\\\\nu^{1/3}}\\\\{1+\\\\sum_{k=1}^{\\\\infty}\\\\frac{\\\\alpha_k}{\\\\nu^{2k}}\\\\}-\\\\frac{3^{1/2}b}{\\\\nu^{5/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{\\\\beta_k}{\\\\nu^{2k}}\\n9.3.33  J'_\\\\nu(\\\\nu)\\\\sim\\\\frac{b}{\\\\nu^{2/3}}\\\\{1+\\\\sum_{k=1}^{\\\\infty}\\\\frac{\\\\gamma_k}{\\\\nu^{2k}}\\\\}-\\\\frac{a}{\\\\nu^{4/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{\\\\delta_k}{\\\\nu^{2k}}\\n9.3.34  Y'_\\\\nu(\\\\nu)\\\\sim\\\\frac{3^{1/2}b}{\\\\nu^{2/3}}\\\\{1+\\\\sum_{k=1}^{\\\\infty}\\\\frac{\\\\gamma_k}{\\\\nu^{2k}}\\\\}+\\\\frac{3^{1/2}a}{\\\\nu^{4/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{\\\\delta_k}{\\\\nu^{2k}}\\nwhere\\na=\\\\frac{2^{1/3}}{3^{2/3}\\\\Gamma(\\\\frac23)}=.44730\\\\,73184, \\\\qquad 3^{\\\\frac12}a=.77475\\\\,90021\\nb=\\\\frac{2^{2/3}}{3^{1/3}\\\\Gamma(\\\\frac13)}=.41085\\\\,01939, \\\\qquad 3^{\\\\frac12}b=.71161\\\\,34101\\n\\\\alpha_0=1, \\\\qquad \\\\alpha_1=-\\\\frac{1}{225}=-.004\\\\dot{4},\\n\\\\alpha_2=.00069\\\\,3735\\\\dots, \\\\qquad \\\\alpha_3=-.00035\\\\,38\\\\dots\\n\\\\beta_0=\\\\frac{1}{70}=.01428\\\\,57143\\\\dots,\\n\\\\beta_1=-\\\\frac{1213}{10\\\\,23750}=-.00118\\\\,48596\\\\dots,\\n\\\\beta_2=.00043\\\\,78\\\\dots, \\\\qquad \\\\beta_3=-.00038\\\\dots\\n\\\\gamma_0=1, \\\\qquad \\\\gamma_1=\\\\frac{23}{3150}=.00730\\\\,15873\\\\dots,\\n\\\\gamma_2=-.00093\\\\,7300\\\\dots, \\\\qquad \\\\gamma_3=.00044\\\\,40\\\\dots\\n\\\\delta_0=\\\\frac{1}{5}, \\\\qquad \\\\delta_1=-\\\\frac{947}{3\\\\,46500}=-.00273\\\\,30447\\\\dots,\\n\\\\delta_2=.00060\\\\,47\\\\dots, \\\\qquad \\\\delta_3=-.00038\\\\dots\\nUniform Asymptotic Expansions\\nThese are more powerful than the previous expansions of this section, save for 9.3.31 and 9.3.32, but their coefficients are more complicated. They reduce to 9.3.31 and 9.3.32 when the argument equals the order.\\n9.3.35  J_\\\\nu(\\\\nu z)\\\\sim\\\\left(\\\\frac{4\\\\zeta}{1-z^2}\\\\right)^{1/4}\\\\{\\\\frac{\\\\mathrm{Ai}(\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{1/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{a_k(\\\\zeta)}{\\\\nu^{2k}}+\\\\frac{\\\\mathrm{Ai}'(\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{5/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{b_k(\\\\zeta)}{\\\\nu^{2k}}\\\\}\\n9.3.36  Y_\\\\nu(\\\\nu z)\\\\sim-\\\\left(\\\\frac{4\\\\zeta}{1-z^2}\\\\right)^{1/4}\\\\{\\\\frac{\\\\mathrm{Bi}(\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{1/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{a_k(\\\\zeta)}{\\\\nu^{2k}}+\\\\frac{\\\\mathrm{Bi}'(\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{5/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{b_k(\\\\zeta)}{\\\\nu^{2k}}\\\\}\\n9.3.37  H^{(1)}_\\\\nu(\\\\nu z)\\\\sim 2e^{-\\\\pi i/3}\\\\left(\\\\frac{4\\\\zeta}{1-z^2}\\\\right)^{1/4}\\\\{\\\\frac{\\\\mathrm{Ai}(e^{2\\\\pi i/3}\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{1/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{a_k(\\\\zeta)}{\\\\nu^{2k}}+\\\\frac{e^{2\\\\pi i/3}\\\\mathrm{Ai}'(e^{2\\\\pi i/3}\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{5/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{b_k(\\\\zeta)}{\\\\nu^{2k}}\\\\}\\nWhen \\\\nu\\\\to+\\\\infty, these expansions hold uniformly with respect to z in the sector |\\\\arg z|\\\\le\\\\pi-\\\\epsilon, where \\\\epsilon is an arbitrary positive number. The corresponding expansion for H^{(2)}_\\\\nu(\\\\nu z) is obtained by changing the sign of i in 9.3.37.\\nHere\\n9.3.38  \\\\frac{2}{3}\\\\zeta^{3/2}=\\\\int_z^1\\\\frac{\\\\sqrt{1-t^2}}{t}dt=\\\\ln\\\\frac{1+\\\\sqrt{1-z^2}}{z}-\\\\sqrt{1-z^2}\\nequivalently,\\n9.3.39  \\\\frac{2}{3}(-\\\\zeta)^{3/2}=\\\\int_1^z\\\\frac{\\\\sqrt{t^2-1}}{t}dt=\\\\sqrt{z^2-1}-\\\\arccos\\\\left(\\\\frac{1}{z}\\\\right)\\nthe branches being chosen so that \\\\zeta is real when z is positive. The coefficients are given by\\n9.3.40  a_k(\\\\zeta)=\\\\sum_{s=0}^{2k}\\\\mu_s\\\\zeta^{-3s/2}u_{2k-s}\\\\{(1-z^2)^{-\\\\frac12}\\\\}\\n        b_k(\\\\zeta)=-\\\\zeta^{-\\\\frac12}\\\\sum_{s=0}^{2k+1}\\\\lambda_s\\\\zeta^{-3s/2}u_{2k-s+1}\\\\{(1-z^2)^{-\\\\frac12}\\\\}\\nwhere u_k is given by 9.3.9 and 9.3.10, \\\\lambda_0=\\\\mu_0=1 and\\n9.3.41  \\\\lambda_s=\\\\frac{(2s+1)(2s+3)\\\\dots(6s-1)}{s!(144)^s}, \\\\qquad \\\\mu_s=-\\\\frac{6s+1}{6s-1}\\\\lambda_s\\nThus a_0(\\\\zeta)=1,\\n9.3.42  b_0(\\\\zeta)=-\\\\frac{5}{48\\\\zeta^2}+\\\\frac{1}{\\\\zeta^{\\\\frac12}}\\\\{\\\\frac{5}{24(1-z^2)^{3/2}}-\\\\frac{1}{8(1-z^2)^{\\\\frac12}}\\\\}\\n        =-\\\\frac{5}{48\\\\zeta^2}+\\\\frac{1}{(-\\\\zeta)^{\\\\frac12}}\\\\{\\\\frac{5}{24(z^2-1)^{3/2}}+\\\\frac{1}{8(z^2-1)^{\\\\frac12}}\\\\}\\nTables of the early coefficients are given below. For more extensive tables of the coefficients and for bounds on the remainder terms in 9.3.35 and 9.3.36 see [9.38].\"}, {\"page_id\": \"as_p0369\", \"printed_page\": 369, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"prose+formula\", \"image\": \"grading_kit/heldout_pages/as_p0369.png\", \"note\": \"BLANK-SLATE page: the repaired reader logged repetition-degeneration here, so there was no draft to correct -- transcribed directly from the scan. Mixed page (Ch.9 Bessel): 9.3.43-9.3.46 uniform expansions of the derivatives, two compact 11-row coefficient tables (transcribed in full, being small), the large-|zeta| asymptotic forms, and section 9.4 polynomial approximations 9.4.1-9.4.3 with the attribution footnote.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.369)\\nUniform Expansions of the Derivatives\\nWith the conditions of the preceding subsection\\n9.3.43  J'_\\\\nu(\\\\nu z)\\\\sim-\\\\frac{2}{z}\\\\left(\\\\frac{1-z^2}{4\\\\zeta}\\\\right)^{\\\\frac12}\\\\{\\\\frac{\\\\mathrm{Ai}(\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{4/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{c_k(\\\\zeta)}{\\\\nu^{2k}}+\\\\frac{\\\\mathrm{Ai}'(\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{2/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{d_k(\\\\zeta)}{\\\\nu^{2k}}\\\\}\\n9.3.44  Y'_\\\\nu(\\\\nu z)\\\\sim\\\\frac{2}{z}\\\\left(\\\\frac{1-z^2}{4\\\\zeta}\\\\right)^{\\\\frac12}\\\\{\\\\frac{\\\\mathrm{Bi}(\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{4/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{c_k(\\\\zeta)}{\\\\nu^{2k}}+\\\\frac{\\\\mathrm{Bi}'(\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{2/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{d_k(\\\\zeta)}{\\\\nu^{2k}}\\\\}\\n9.3.45  H^{(1)'}_\\\\nu(\\\\nu z)\\\\sim\\\\frac{4e^{2\\\\pi i/3}}{z}\\\\left(\\\\frac{1-z^2}{4\\\\zeta}\\\\right)^{\\\\frac12}\\\\{\\\\frac{\\\\mathrm{Ai}(e^{2\\\\pi i/3}\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{4/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{c_k(\\\\zeta)}{\\\\nu^{2k}}+\\\\frac{e^{2\\\\pi i/3}\\\\mathrm{Ai}'(e^{2\\\\pi i/3}\\\\nu^{2/3}\\\\zeta)}{\\\\nu^{2/3}}\\\\sum_{k=0}^{\\\\infty}\\\\frac{d_k(\\\\zeta)}{\\\\nu^{2k}}\\\\}\\nwhere\\n9.3.46  c_k(\\\\zeta)=-\\\\zeta^{\\\\frac12}\\\\sum_{s=0}^{2k+1}\\\\mu_s\\\\zeta^{-3s/2}v_{2k-s+1}\\\\{(1-z^2)^{-\\\\frac12}\\\\}\\n        d_k(\\\\zeta)=\\\\sum_{s=0}^{2k}\\\\lambda_s\\\\zeta^{-3s/2}v_{2k-s}\\\\{(1-z^2)^{-\\\\frac12}\\\\}\\nand v_k is given by 9.3.13 and 9.3.14. For bounds on the remainder terms in 9.3.43 and 9.3.44 see [9.38].\\nTable (positive argument). Columns: \\\\zeta | b_0(\\\\zeta) | a_1(\\\\zeta) | c_0(\\\\zeta) | d_1(\\\\zeta)\\n0   0.0180  -0.004  0.1587  0.007\\n1   .0278   -.004   .1785   .009\\n2   .0351   -.001   .1862   .007\\n3   .0366   +.002   .1927   .005\\n4   .0352   .003    .2031   .004\\n5   .0331   .004    .2155   .003\\n6   .0311   .004    .2284   .003\\n7   .0294   .004    .2413   .003\\n8   .0278   .004    .2539   .003\\n9   .0265   .004    .2662   .003\\n10  .0253   .004    .2781   .003\\nTable (negative argument). Columns: -\\\\zeta | b_0(\\\\zeta) | a_1(\\\\zeta) | c_0(\\\\zeta) | d_1(\\\\zeta)\\n0   0.0180  -0.004  0.1587  0.007\\n1   .0109   -.003   .1323   .004\\n2   .0067   -.002   .1087   .002\\n3   .0044   -.001   .0903   .001\\n4   .0031   -.001   .0764   .001\\n5   .0022   -.000   .0658   .000\\n6   .0017   -.000   .0576   .000\\n7   .0013   -.000   .0511   .000\\n8   .0011   -.000   .0459   .000\\n9   .0009   -.000   .0415   .000\\n10  .0007   -.000   .0379   .000\\nFor \\\\zeta>10 use\\nb_0(\\\\zeta)\\\\sim\\\\frac{1}{12}\\\\zeta^{-\\\\frac12}-.104\\\\zeta^{-2}, \\\\qquad a_1(\\\\zeta)=.003,\\nc_0(\\\\zeta)\\\\sim\\\\frac{1}{12}\\\\zeta^{\\\\frac12}+.146\\\\zeta^{-1}, \\\\qquad d_1(\\\\zeta)=.003.\\nFor \\\\zeta<-10 use\\nb_0(\\\\zeta)\\\\sim\\\\frac{1}{12}\\\\zeta^{-2}, \\\\qquad a_1(\\\\zeta)=.000,\\nc_0(\\\\zeta)\\\\sim-\\\\frac{5}{12}\\\\zeta^{-1}-1.33(-\\\\zeta)^{-5/2}, \\\\qquad d_1(\\\\zeta)=.000.\\nMaximum values of higher coefficients:\\n|b_1(\\\\zeta)|=.003, \\\\qquad |a_2(\\\\zeta)|=.0008, \\\\qquad |d_2(\\\\zeta)|=.001\\n|c_1(\\\\zeta)|=.008\\\\ (\\\\zeta<10), \\\\qquad c_1(\\\\zeta)\\\\sim-.003\\\\zeta^{\\\\frac12} as \\\\zeta\\\\to+\\\\infty.\\n9.4. Polynomial Approximations\\n9.4.1  -3\\\\le x\\\\le 3\\nJ_0(x)=1-2.24999\\\\,97(x/3)^2+1.26562\\\\,08(x/3)^4-.31638\\\\,66(x/3)^6+.04444\\\\,79(x/3)^8-.00394\\\\,44(x/3)^{10}+.00021\\\\,00(x/3)^{12}+\\\\epsilon\\n|\\\\epsilon|<5\\\\times 10^{-8}\\n9.4.2  0<x\\\\le 3\\nY_0(x)=(2/\\\\pi)\\\\ln(\\\\tfrac12 x)J_0(x)+.36746\\\\,691+.60559\\\\,366(x/3)^2-.74350\\\\,384(x/3)^4+.25300\\\\,117(x/3)^6-.04261\\\\,214(x/3)^8+.00427\\\\,916(x/3)^{10}-.00024\\\\,846(x/3)^{12}+\\\\epsilon\\n|\\\\epsilon|<1.4\\\\times 10^{-8}\\n9.4.3  3\\\\le x<\\\\infty\\nJ_0(x)=x^{-\\\\frac12}f_0\\\\cos\\\\theta_0 \\\\qquad Y_0(x)=x^{-\\\\frac12}f_0\\\\sin\\\\theta_0\\nf_0=.79788\\\\,456-.00000\\\\,077(3/x)-.00552\\\\,740(3/x)^2-.00009\\\\,512(3/x)^3+.00137\\\\,237(3/x)^4-.00072\\\\,805(3/x)^5+.00014\\\\,476(3/x)^6+\\\\epsilon\\n|\\\\epsilon|<1.6\\\\times 10^{-8}\\nFootnote 2: Equations 9.4.1 to 9.4.6 and 9.8.1 to 9.8.8 are taken from E. E. Allen, Analytical approximations, Math. Tables Aids Comp. 8, 240-241 (1954), and Polynomial approximations to some modified Bessel functions, Math. Tables Aids Comp. 10, 162-164 (1956) (with permission). They were checked at the National Physical Laboratory by systematic tabulation; new bounds for the errors, \\\\epsilon, given here were obtained as a result.\"}, {\"page_id\": \"as_p0376\", \"printed_page\": 376, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0376.png\", \"note\": \"Two-column dense-formula page (Ch.9, modified Bessel functions I and K), 9.6.16-9.6.40: integral representations, recurrence relations, derivative formulas, analytic continuation, and the generating function with associated series. Draft existed (3728 chars) and was corrected against the scan.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.376)\\nIntegral Representations\\n9.6.16  I_0(z)=\\\\frac{1}{\\\\pi}\\\\int_0^\\\\pi e^{\\\\pm z\\\\cos\\\\theta}d\\\\theta=\\\\frac{1}{\\\\pi}\\\\int_0^\\\\pi\\\\cosh(z\\\\cos\\\\theta)d\\\\theta\\n9.6.17  K_0(z)=-\\\\frac{1}{\\\\pi}\\\\int_0^\\\\pi e^{\\\\pm z\\\\cos\\\\theta}\\\\{\\\\gamma+\\\\ln(2z\\\\sin^2\\\\theta)\\\\}d\\\\theta\\n9.6.18  I_\\\\nu(z)=\\\\frac{(\\\\frac12 z)^\\\\nu}{\\\\pi^{\\\\frac12}\\\\Gamma(\\\\nu+\\\\frac12)}\\\\int_0^\\\\pi e^{\\\\pm z\\\\cos\\\\theta}\\\\sin^{2\\\\nu}\\\\theta\\\\,d\\\\theta=\\\\frac{(\\\\frac12 z)^\\\\nu}{\\\\pi^{\\\\frac12}\\\\Gamma(\\\\nu+\\\\frac12)}\\\\int_{-1}^1(1-t^2)^{\\\\nu-\\\\frac12}e^{\\\\pm zt}dt \\\\quad (\\\\mathscr{R}\\\\nu>-\\\\tfrac12)\\n9.6.19  I_n(z)=\\\\frac{1}{\\\\pi}\\\\int_0^\\\\pi e^{z\\\\cos\\\\theta}\\\\cos(n\\\\theta)d\\\\theta\\n9.6.20  I_\\\\nu(z)=\\\\frac{1}{\\\\pi}\\\\int_0^\\\\pi e^{z\\\\cos\\\\theta}\\\\cos(\\\\nu\\\\theta)d\\\\theta-\\\\frac{\\\\sin(\\\\nu\\\\pi)}{\\\\pi}\\\\int_0^\\\\infty e^{-z\\\\cosh t-\\\\nu t}dt \\\\quad (|\\\\arg z|<\\\\tfrac12\\\\pi)\\n9.6.21  K_0(x)=\\\\int_0^\\\\infty\\\\cos(x\\\\sinh t)dt=\\\\int_0^\\\\infty\\\\frac{\\\\cos(xt)}{\\\\sqrt{t^2+1}}dt \\\\quad (x>0)\\n9.6.22  K_\\\\nu(x)=\\\\sec(\\\\tfrac12\\\\nu\\\\pi)\\\\int_0^\\\\infty\\\\cos(x\\\\sinh t)\\\\cosh(\\\\nu t)dt=\\\\csc(\\\\tfrac12\\\\nu\\\\pi)\\\\int_0^\\\\infty\\\\sin(x\\\\sinh t)\\\\sinh(\\\\nu t)dt \\\\quad (|\\\\mathscr{R}\\\\nu|<1, x>0)\\n9.6.23  K_\\\\nu(z)=\\\\frac{\\\\pi^{\\\\frac12}(\\\\frac12 z)^\\\\nu}{\\\\Gamma(\\\\nu+\\\\frac12)}\\\\int_0^\\\\infty e^{-z\\\\cosh t}\\\\sinh^{2\\\\nu}t\\\\,dt=\\\\frac{\\\\pi^{\\\\frac12}(\\\\frac12 z)^\\\\nu}{\\\\Gamma(\\\\nu+\\\\frac12)}\\\\int_1^\\\\infty e^{-zt}(t^2-1)^{\\\\nu-\\\\frac12}dt \\\\quad (\\\\mathscr{R}\\\\nu>-\\\\tfrac12, |\\\\arg z|<\\\\tfrac12\\\\pi)\\n9.6.24  K_\\\\nu(z)=\\\\int_0^\\\\infty e^{-z\\\\cosh t}\\\\cosh(\\\\nu t)dt \\\\quad (|\\\\arg z|<\\\\tfrac12\\\\pi)\\n9.6.25  K_\\\\nu(xz)=\\\\frac{\\\\Gamma(\\\\nu+\\\\frac12)(2z)^\\\\nu}{\\\\pi^{\\\\frac12}x^\\\\nu}\\\\int_0^\\\\infty\\\\frac{\\\\cos(xt)dt}{(t^2+z^2)^{\\\\nu+\\\\frac12}} \\\\quad (\\\\mathscr{R}\\\\nu\\\\ge-\\\\tfrac12, x>0, |\\\\arg z|<\\\\tfrac12\\\\pi)\\nRecurrence Relations\\n9.6.26  \\\\mathscr{Z}_{\\\\nu-1}(z)-\\\\mathscr{Z}_{\\\\nu+1}(z)=\\\\frac{2\\\\nu}{z}\\\\mathscr{Z}_\\\\nu(z)\\n        \\\\mathscr{Z}'_\\\\nu(z)=\\\\mathscr{Z}_{\\\\nu-1}(z)-\\\\frac{\\\\nu}{z}\\\\mathscr{Z}_\\\\nu(z)\\n        \\\\mathscr{Z}_{\\\\nu-1}(z)+\\\\mathscr{Z}_{\\\\nu+1}(z)=2\\\\mathscr{Z}'_\\\\nu(z)\\n        \\\\mathscr{Z}'_\\\\nu(z)=\\\\mathscr{Z}_{\\\\nu+1}(z)+\\\\frac{\\\\nu}{z}\\\\mathscr{Z}_\\\\nu(z)\\n\\\\mathscr{Z}_\\\\nu denotes I_\\\\nu, e^{\\\\nu\\\\pi i}K_\\\\nu or any linear combination of these functions, the coefficients in which are independent of z and \\\\nu.\\n9.6.27  I'_0(z)=I_1(z), \\\\qquad K'_0(z)=-K_1(z)\\nFormulas for Derivatives\\n9.6.28  \\\\left(\\\\frac{1}{z}\\\\frac{d}{dz}\\\\right)^k\\\\{z^\\\\nu\\\\mathscr{Z}_\\\\nu(z)\\\\}=z^{\\\\nu-k}\\\\mathscr{Z}_{\\\\nu-k}(z)\\n        \\\\left(\\\\frac{1}{z}\\\\frac{d}{dz}\\\\right)^k\\\\{z^{-\\\\nu}\\\\mathscr{Z}_\\\\nu(z)\\\\}=z^{-\\\\nu-k}\\\\mathscr{Z}_{\\\\nu+k}(z) \\\\quad (k=0,1,2,\\\\dots)\\n9.6.29  \\\\mathscr{Z}^{(k)}_\\\\nu(z)=\\\\frac{1}{2^k}\\\\{\\\\mathscr{Z}_{\\\\nu-k}(z)+\\\\binom{k}{1}\\\\mathscr{Z}_{\\\\nu-k+2}(z)+\\\\binom{k}{2}\\\\mathscr{Z}_{\\\\nu-k+4}(z)+\\\\dots+\\\\mathscr{Z}_{\\\\nu+k}(z)\\\\} \\\\quad (k=0,1,2,\\\\dots)\\nAnalytic Continuation\\n9.6.30  I_\\\\nu(ze^{m\\\\pi i})=e^{m\\\\nu\\\\pi i}I_\\\\nu(z) \\\\quad (m\\\\ \\\\mathrm{an\\\\ integer})\\n9.6.31  K_\\\\nu(ze^{m\\\\pi i})=e^{-m\\\\nu\\\\pi i}K_\\\\nu(z)-\\\\pi i\\\\sin(m\\\\nu\\\\pi)\\\\csc(\\\\nu\\\\pi)I_\\\\nu(z) \\\\quad (m\\\\ \\\\mathrm{an\\\\ integer})\\n9.6.32  I_\\\\nu(\\\\bar z)=\\\\overline{I_\\\\nu(z)}, \\\\qquad K_\\\\nu(\\\\bar z)=\\\\overline{K_\\\\nu(z)} \\\\quad (\\\\nu\\\\ \\\\mathrm{real})\\nGenerating Function and Associated Series\\n9.6.33  e^{\\\\frac12 z(t+1/t)}=\\\\sum_{k=-\\\\infty}^{\\\\infty}t^kI_k(z) \\\\quad (t\\\\neq 0)\\n9.6.34  e^{z\\\\cos\\\\theta}=I_0(z)+2\\\\sum_{k=1}^{\\\\infty}I_k(z)\\\\cos(k\\\\theta)\\n9.6.35  e^{z\\\\sin\\\\theta}=I_0(z)+2\\\\sum_{k=0}^{\\\\infty}(-)^kI_{2k+1}(z)\\\\sin\\\\{(2k+1)\\\\theta\\\\}+2\\\\sum_{k=1}^{\\\\infty}(-)^kI_{2k}(z)\\\\cos(2k\\\\theta)\\n9.6.36  1=I_0(z)-2I_2(z)+2I_4(z)-2I_6(z)+\\\\dots\\n9.6.37  e^z=I_0(z)+2I_1(z)+2I_2(z)+2I_3(z)+\\\\dots\\n9.6.38  e^{-z}=I_0(z)-2I_1(z)+2I_2(z)-2I_3(z)+\\\\dots\\n9.6.39  \\\\cosh z=I_0(z)+2I_2(z)+2I_4(z)+2I_6(z)+\\\\dots\\n9.6.40  \\\\sinh z=2I_1(z)+2I_3(z)+2I_5(z)+\\\\dots\"}, {\"page_id\": \"as_p0379\", \"printed_page\": 379, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0379.png\", \"note\": \"Two-column dense-formula page (Ch.9 Bessel), 9.8.5-9.9.11: the last polynomial approximations for K_0/K_1, then the start of the Kelvin functions section 9.9 (definitions of ber/bei/ker/kei, their differential equations, relations between solutions, and ascending series). Draft existed (2989 chars) and was corrected against the scan.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.379)\\n9.8.5  0<x\\\\le 2\\nK_0(x)=-\\\\ln(x/2)I_0(x)-.57721\\\\,566+.42278\\\\,420(x/2)^2+.23069\\\\,756(x/2)^4+.03488\\\\,590(x/2)^6+.00262\\\\,698(x/2)^8+.00010\\\\,750(x/2)^{10}+.00000\\\\,740(x/2)^{12}+\\\\epsilon\\n|\\\\epsilon|<1\\\\times 10^{-8}\\n9.8.6  2\\\\le x<\\\\infty\\nx^{\\\\frac12}e^xK_0(x)=1.25331\\\\,414-.07832\\\\,358(2/x)+.02189\\\\,568(2/x)^2-.01062\\\\,446(2/x)^3+.00587\\\\,872(2/x)^4-.00251\\\\,540(2/x)^5+.00053\\\\,208(2/x)^6+\\\\epsilon\\n|\\\\epsilon|<1.9\\\\times 10^{-7}\\n9.8.7  0<x\\\\le 2\\nxK_1(x)=x\\\\ln(x/2)I_1(x)+1+.15443\\\\,144(x/2)^2-.67278\\\\,579(x/2)^4-.18156\\\\,897(x/2)^6-.01919\\\\,402(x/2)^8-.00110\\\\,404(x/2)^{10}-.00004\\\\,686(x/2)^{12}+\\\\epsilon\\n|\\\\epsilon|<8\\\\times 10^{-9}\\n9.8.8  2\\\\le x<\\\\infty\\nx^{\\\\frac12}e^xK_1(x)=1.25331\\\\,414+.23498\\\\,619(2/x)-.03655\\\\,620(2/x)^2+.01504\\\\,268(2/x)^3-.00780\\\\,353(2/x)^4+.00325\\\\,614(2/x)^5-.00068\\\\,245(2/x)^6+\\\\epsilon\\n|\\\\epsilon|<2.2\\\\times 10^{-7}\\nFor expansions of I_0(x), K_0(x), I_1(x), and K_1(x) in series of Chebyshev polynomials for the ranges 0\\\\le x\\\\le 8 and 0\\\\le 8/x\\\\le 1, see [9.37].\\nKelvin Functions\\n9.9. Definitions and Properties\\nIn this and the following section \\\\nu is real, x is real and non-negative, and n is again a positive integer or zero.\\nDefinitions\\n9.9.1  \\\\mathrm{ber}_\\\\nu x+i\\\\,\\\\mathrm{bei}_\\\\nu x=J_\\\\nu(xe^{3\\\\pi i/4})=e^{\\\\nu\\\\pi i}J_\\\\nu(xe^{-\\\\pi i/4})=e^{\\\\frac12\\\\nu\\\\pi i}I_\\\\nu(xe^{\\\\pi i/4})=e^{3\\\\nu\\\\pi i/2}I_\\\\nu(xe^{-3\\\\pi i/4})\\n9.9.2  \\\\mathrm{ker}_\\\\nu x+i\\\\,\\\\mathrm{kei}_\\\\nu x=e^{-\\\\frac12\\\\nu\\\\pi i}K_\\\\nu(xe^{\\\\pi i/4})=\\\\tfrac12\\\\pi iH^{(1)}_\\\\nu(xe^{3\\\\pi i/4})=-\\\\tfrac12\\\\pi ie^{-\\\\nu\\\\pi i}H^{(2)}_\\\\nu(xe^{-\\\\pi i/4})\\nWhen \\\\nu=0, suffices are usually suppressed.\\nDifferential Equations\\n9.9.3  x^2w''+xw'-(ix^2+\\\\nu^2)w=0,\\n       w=\\\\mathrm{ber}_\\\\nu x+i\\\\,\\\\mathrm{bei}_\\\\nu x, \\\\quad \\\\mathrm{ber}_{-\\\\nu} x+i\\\\,\\\\mathrm{bei}_{-\\\\nu} x, \\\\quad \\\\mathrm{ker}_\\\\nu x+i\\\\,\\\\mathrm{kei}_\\\\nu x, \\\\quad \\\\mathrm{ker}_{-\\\\nu} x+i\\\\,\\\\mathrm{kei}_{-\\\\nu} x\\n9.9.4  x^4w^{iv}+2x^3w'''-(1+2\\\\nu^2)(x^2w''-xw')+(\\\\nu^4-4\\\\nu^2+x^4)w=0,\\n       w=\\\\mathrm{ber}_{\\\\pm\\\\nu} x,\\\\ \\\\mathrm{bei}_{\\\\pm\\\\nu} x,\\\\ \\\\mathrm{ker}_{\\\\pm\\\\nu} x,\\\\ \\\\mathrm{kei}_{\\\\pm\\\\nu} x\\nRelations Between Solutions\\n9.9.5  \\\\mathrm{ber}_{-\\\\nu} x=\\\\cos(\\\\nu\\\\pi)\\\\,\\\\mathrm{ber}_\\\\nu x+\\\\sin(\\\\nu\\\\pi)\\\\,\\\\mathrm{bei}_\\\\nu x+(2/\\\\pi)\\\\sin(\\\\nu\\\\pi)\\\\,\\\\mathrm{ker}_\\\\nu x\\n       \\\\mathrm{bei}_{-\\\\nu} x=-\\\\sin(\\\\nu\\\\pi)\\\\,\\\\mathrm{ber}_\\\\nu x+\\\\cos(\\\\nu\\\\pi)\\\\,\\\\mathrm{bei}_\\\\nu x+(2/\\\\pi)\\\\sin(\\\\nu\\\\pi)\\\\,\\\\mathrm{kei}_\\\\nu x\\n9.9.6  \\\\mathrm{ker}_{-\\\\nu} x=\\\\cos(\\\\nu\\\\pi)\\\\,\\\\mathrm{ker}_\\\\nu x-\\\\sin(\\\\nu\\\\pi)\\\\,\\\\mathrm{kei}_\\\\nu x\\n       \\\\mathrm{kei}_{-\\\\nu} x=\\\\sin(\\\\nu\\\\pi)\\\\,\\\\mathrm{ker}_\\\\nu x+\\\\cos(\\\\nu\\\\pi)\\\\,\\\\mathrm{kei}_\\\\nu x\\n9.9.7  \\\\mathrm{ber}_{-n} x=(-)^n\\\\,\\\\mathrm{ber}_n x, \\\\quad \\\\mathrm{bei}_{-n} x=(-)^n\\\\,\\\\mathrm{bei}_n x\\n9.9.8  \\\\mathrm{ker}_{-n} x=(-)^n\\\\,\\\\mathrm{ker}_n x, \\\\quad \\\\mathrm{kei}_{-n} x=(-)^n\\\\,\\\\mathrm{kei}_n x\\nAscending Series\\n9.9.9  \\\\mathrm{ber}_\\\\nu x=(\\\\tfrac12 x)^\\\\nu\\\\sum_{k=0}^{\\\\infty}\\\\frac{\\\\cos\\\\{(\\\\frac34\\\\nu+\\\\frac12 k)\\\\pi\\\\}}{k!\\\\Gamma(\\\\nu+k+1)}(\\\\tfrac14 x^2)^k\\n       \\\\mathrm{bei}_\\\\nu x=(\\\\tfrac12 x)^\\\\nu\\\\sum_{k=0}^{\\\\infty}\\\\frac{\\\\sin\\\\{(\\\\frac34\\\\nu+\\\\frac12 k)\\\\pi\\\\}}{k!\\\\Gamma(\\\\nu+k+1)}(\\\\tfrac14 x^2)^k\\n9.9.10 \\\\mathrm{ber}\\\\,x=1-\\\\frac{(\\\\frac14 x^2)^2}{(2!)^2}+\\\\frac{(\\\\frac14 x^2)^4}{(4!)^2}-\\\\dots\\n       \\\\mathrm{bei}\\\\,x=\\\\tfrac14 x^2-\\\\frac{(\\\\frac14 x^2)^3}{(3!)^2}+\\\\frac{(\\\\frac14 x^2)^5}{(5!)^2}-\\\\dots\\n9.9.11 \\\\mathrm{ker}_n x=\\\\tfrac12(\\\\tfrac12 x)^{-n}\\\\sum_{k=0}^{n-1}\\\\cos\\\\{(\\\\tfrac34 n+\\\\tfrac12 k)\\\\pi\\\\}\\\\frac{(n-k-1)!}{k!}(\\\\tfrac14 x^2)^k-\\\\ln(\\\\tfrac12 x)\\\\,\\\\mathrm{ber}_n x+\\\\tfrac14\\\\pi\\\\,\\\\mathrm{bei}_n x+\\\\tfrac12(\\\\tfrac12 x)^n\\\\sum_{k=0}^{\\\\infty}\\\\cos\\\\{(\\\\tfrac34 n+\\\\tfrac12 k)\\\\pi\\\\}\\\\frac{\\\\{\\\\psi(k+1)+\\\\psi(n+k+1)\\\\}}{k!(n+k)!}(\\\\tfrac14 x^2)^k\"}, {\"page_id\": \"as_p0383\", \"printed_page\": 383, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"prose+formula\", \"image\": \"grading_kit/heldout_pages/as_p0383.png\", \"note\": \"Two-column formula page with connecting prose (Ch.9, Kelvin functions), 9.10.21-9.10.36: asymptotic expansions of modulus and phase, of cross-products, and of the large zeros. Draft existed (2049 chars, truncated) and was corrected/completed against the scan.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.383)\\nAsymptotic Expansions of Modulus and Phase\\nWhen \\\\nu is fixed, x is large and \\\\mu=4\\\\nu^2\\n9.10.21  M_\\\\nu=\\\\frac{e^{x/\\\\sqrt2}}{\\\\sqrt{2\\\\pi x}}\\\\{1-\\\\frac{\\\\mu-1}{8\\\\sqrt2}\\\\frac{1}{x}+\\\\frac{(\\\\mu-1)^2}{256}\\\\frac{1}{x^2}-\\\\frac{(\\\\mu-1)(\\\\mu^2+14\\\\mu-399)}{6144\\\\sqrt2}\\\\frac{1}{x^3}+O\\\\left(\\\\frac{1}{x^4}\\\\right)\\\\}\\n9.10.22  \\\\ln M_\\\\nu=\\\\frac{x}{\\\\sqrt2}-\\\\tfrac12\\\\ln(2\\\\pi x)-\\\\frac{\\\\mu-1}{8\\\\sqrt2}\\\\frac{1}{x}-\\\\frac{(\\\\mu-1)(\\\\mu-25)}{384\\\\sqrt2}\\\\frac{1}{x^3}-\\\\frac{(\\\\mu-1)(\\\\mu-13)}{128}\\\\frac{1}{x^4}+O\\\\left(\\\\frac{1}{x^5}\\\\right)\\n9.10.23  \\\\theta_\\\\nu=\\\\frac{x}{\\\\sqrt2}+\\\\left(\\\\frac12\\\\nu-\\\\frac18\\\\right)\\\\pi+\\\\frac{\\\\mu-1}{8\\\\sqrt2}\\\\frac{1}{x}+\\\\frac{\\\\mu-1}{16}\\\\frac{1}{x^2}-\\\\frac{(\\\\mu-1)(\\\\mu-25)}{384\\\\sqrt2}\\\\frac{1}{x^3}+O\\\\left(\\\\frac{1}{x^5}\\\\right)\\n9.10.24  N_\\\\nu=\\\\sqrt{\\\\frac{\\\\pi}{2x}}e^{-x/\\\\sqrt2}\\\\{1+\\\\frac{\\\\mu-1}{8\\\\sqrt2}\\\\frac{1}{x}+\\\\frac{(\\\\mu-1)^2}{256}\\\\frac{1}{x^2}+\\\\frac{(\\\\mu-1)(\\\\mu^2+14\\\\mu-399)}{6144\\\\sqrt2}\\\\frac{1}{x^3}+O\\\\left(\\\\frac{1}{x^4}\\\\right)\\\\}\\n9.10.25  \\\\ln N_\\\\nu=-\\\\frac{x}{\\\\sqrt2}+\\\\tfrac12\\\\ln\\\\left(\\\\frac{\\\\pi}{2x}\\\\right)+\\\\frac{\\\\mu-1}{8\\\\sqrt2}\\\\frac{1}{x}+\\\\frac{(\\\\mu-1)(\\\\mu-25)}{384\\\\sqrt2}\\\\frac{1}{x^3}-\\\\frac{(\\\\mu-1)(\\\\mu-13)}{128}\\\\frac{1}{x^4}+O\\\\left(\\\\frac{1}{x^5}\\\\right)\\n9.10.26  \\\\phi_\\\\nu=-\\\\frac{x}{\\\\sqrt2}-\\\\left(\\\\frac12\\\\nu+\\\\frac18\\\\right)\\\\pi-\\\\frac{\\\\mu-1}{8\\\\sqrt2}\\\\frac{1}{x}+\\\\frac{\\\\mu-1}{16}\\\\frac{1}{x^2}+\\\\frac{(\\\\mu-1)(\\\\mu-25)}{384\\\\sqrt2}\\\\frac{1}{x^3}+O\\\\left(\\\\frac{1}{x^5}\\\\right)\\nAsymptotic Expansions of Cross-Products\\nIf x is large\\n9.10.27  \\\\mathrm{ber}^2 x+\\\\mathrm{bei}^2 x\\\\sim\\\\frac{e^{x\\\\sqrt2}}{2\\\\pi x}\\\\left(1+\\\\frac{1}{4\\\\sqrt2}\\\\frac{1}{x}+\\\\frac{1}{64}\\\\frac{1}{x^2}-\\\\frac{33}{256\\\\sqrt2}\\\\frac{1}{x^3}-\\\\frac{1797}{8192}\\\\frac{1}{x^4}+\\\\dots\\\\right)\\n9.10.28  \\\\mathrm{ber}\\\\,x\\\\,\\\\mathrm{bei}'\\\\,x-\\\\mathrm{ber}'\\\\,x\\\\,\\\\mathrm{bei}\\\\,x\\\\sim\\\\frac{e^{x\\\\sqrt2}}{2\\\\pi x}\\\\left(\\\\frac{1}{\\\\sqrt2}+\\\\frac18\\\\frac{1}{x}+\\\\frac{9}{64\\\\sqrt2}\\\\frac{1}{x^2}+\\\\frac{39}{512}\\\\frac{1}{x^3}+\\\\frac{75}{8192\\\\sqrt2}\\\\frac{1}{x^4}+\\\\dots\\\\right)\\n9.10.29  \\\\mathrm{ber}\\\\,x\\\\,\\\\mathrm{ber}'\\\\,x+\\\\mathrm{bei}\\\\,x\\\\,\\\\mathrm{bei}'\\\\,x\\\\sim\\\\frac{e^{x\\\\sqrt2}}{2\\\\pi x}\\\\left(\\\\frac{1}{\\\\sqrt2}-\\\\frac38\\\\frac{1}{x}-\\\\frac{15}{64\\\\sqrt2}\\\\frac{1}{x^2}-\\\\frac{45}{512}\\\\frac{1}{x^3}+\\\\frac{315}{8192\\\\sqrt2}\\\\frac{1}{x^4}+\\\\dots\\\\right)\\n9.10.30  \\\\mathrm{ber}'^2 x+\\\\mathrm{bei}'^2 x\\\\sim\\\\frac{e^{x\\\\sqrt2}}{2\\\\pi x}\\\\left(1-\\\\frac{3}{4\\\\sqrt2}\\\\frac{1}{x}+\\\\frac{9}{64}\\\\frac{1}{x^2}+\\\\frac{75}{256\\\\sqrt2}\\\\frac{1}{x^3}+\\\\frac{2475}{8192}\\\\frac{1}{x^4}+\\\\dots\\\\right)\\n9.10.31  \\\\mathrm{ker}^2 x+\\\\mathrm{kei}^2 x\\\\sim\\\\frac{\\\\pi}{2x}e^{-x\\\\sqrt2}\\\\left(1-\\\\frac{1}{4\\\\sqrt2}\\\\frac{1}{x}+\\\\frac{1}{64}\\\\frac{1}{x^2}+\\\\frac{33}{256\\\\sqrt2}\\\\frac{1}{x^3}-\\\\frac{1797}{8192}\\\\frac{1}{x^4}+\\\\dots\\\\right)\\n9.10.32  \\\\mathrm{ker}\\\\,x\\\\,\\\\mathrm{kei}'\\\\,x-\\\\mathrm{ker}'\\\\,x\\\\,\\\\mathrm{kei}\\\\,x\\\\sim-\\\\frac{\\\\pi}{2x}e^{-x\\\\sqrt2}\\\\left(\\\\frac{1}{\\\\sqrt2}-\\\\frac18\\\\frac{1}{x}+\\\\frac{9}{64\\\\sqrt2}\\\\frac{1}{x^2}-\\\\frac{39}{512}\\\\frac{1}{x^3}+\\\\frac{75}{8192\\\\sqrt2}\\\\frac{1}{x^4}+\\\\dots\\\\right)\\n9.10.33  \\\\mathrm{ker}\\\\,x\\\\,\\\\mathrm{ker}'\\\\,x+\\\\mathrm{kei}\\\\,x\\\\,\\\\mathrm{kei}'\\\\,x\\\\sim-\\\\frac{\\\\pi}{2x}e^{-x\\\\sqrt2}\\\\left(\\\\frac{1}{\\\\sqrt2}+\\\\frac38\\\\frac{1}{x}-\\\\frac{15}{64\\\\sqrt2}\\\\frac{1}{x^2}+\\\\frac{45}{512}\\\\frac{1}{x^3}+\\\\frac{315}{8192\\\\sqrt2}\\\\frac{1}{x^4}+\\\\dots\\\\right)\\n9.10.34  \\\\mathrm{ker}'^2 x+\\\\mathrm{kei}'^2 x\\\\sim\\\\frac{\\\\pi}{2x}e^{-x\\\\sqrt2}\\\\left(1+\\\\frac{3}{4\\\\sqrt2}\\\\frac{1}{x}+\\\\frac{9}{64}\\\\frac{1}{x^2}-\\\\frac{75}{256\\\\sqrt2}\\\\frac{1}{x^3}+\\\\frac{2475}{8192}\\\\frac{1}{x^4}+\\\\dots\\\\right)\\nAsymptotic Expansions of Large Zeros\\nLet\\n9.10.35  f(\\\\delta)=\\\\frac{\\\\mu-1}{16\\\\delta}+\\\\frac{\\\\mu-1}{32\\\\delta^2}+\\\\frac{(\\\\mu-1)(5\\\\mu+19)}{1536\\\\delta^3}+\\\\frac{3(\\\\mu-1)^2}{512\\\\delta^4}+\\\\dots\\nwhere \\\\mu=4\\\\nu^2. Then if s is a large positive integer\\n9.10.36  Zeros of \\\\mathrm{ber}_\\\\nu x\\\\sim\\\\sqrt2\\\\{\\\\delta-f(\\\\delta)\\\\}, \\\\qquad \\\\delta=(s-\\\\tfrac12\\\\nu-\\\\tfrac38)\\\\pi\\n         Zeros of \\\\mathrm{bei}_\\\\nu x\\\\sim\\\\sqrt2\\\\{\\\\delta-f(\\\\delta)\\\\}, \\\\qquad \\\\delta=(s-\\\\tfrac12\\\\nu+\\\\tfrac18)\\\\pi\\n         Zeros of \\\\mathrm{ker}_\\\\nu x\\\\sim\\\\sqrt2\\\\{\\\\delta+f(-\\\\delta)\\\\}, \\\\qquad \\\\delta=(s-\\\\tfrac12\\\\nu-\\\\tfrac58)\\\\pi\\n         Zeros of \\\\mathrm{kei}_\\\\nu x\\\\sim\\\\sqrt2\\\\{\\\\delta+f(-\\\\delta)\\\\}, \\\\qquad \\\\delta=(s-\\\\tfrac12\\\\nu-\\\\tfrac18)\\\\pi\"}, {\"page_id\": \"as_p0385\", \"printed_page\": 385, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"prose+formula\", \"image\": \"grading_kit/heldout_pages/as_p0385.png\", \"note\": \"Mixed page (Ch.9 Bessel): 9.11.8-9.11.14 polynomial approximations for the Kelvin functions with complex-coefficient expansions, then the start of the Numerical Methods section 9.12 -- worked Example 1 (backward recurrence for J_n(1.55)) with its small 10-row trial-value table, transcribed in full. Draft existed (1011 chars, heavily truncated) and was corrected/completed against the scan.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.385)\\n9.11.8  0<x\\\\le 8\\n\\\\mathrm{kei}'\\\\,x=-\\\\ln(\\\\tfrac12 x)\\\\,\\\\mathrm{bei}'\\\\,x-x^{-1}\\\\,\\\\mathrm{bei}\\\\,x-\\\\tfrac14\\\\pi\\\\,\\\\mathrm{ber}'\\\\,x+x[.21139\\\\,217-13.39858\\\\,846(x/8)^4+19.41182\\\\,758(x/8)^8-4.65950\\\\,823(x/8)^{12}+.33049\\\\,424(x/8)^{16}-.00926\\\\,707(x/8)^{20}+.00011\\\\,997(x/8)^{24}]+\\\\epsilon\\n|\\\\epsilon|<7\\\\times 10^{-8}\\n9.11.9  8\\\\le x<\\\\infty\\n\\\\mathrm{ker}\\\\,x+i\\\\,\\\\mathrm{kei}\\\\,x=f(x)(1+\\\\epsilon_1)\\nf(x)=\\\\sqrt{\\\\frac{\\\\pi}{2x}}\\\\exp\\\\left[-\\\\frac{1+i}{\\\\sqrt2}x+\\\\theta(-x)\\\\right]\\n|\\\\epsilon_1|<1\\\\times 10^{-7}\\n9.11.10  8\\\\le x<\\\\infty\\n\\\\mathrm{ber}\\\\,x+i\\\\,\\\\mathrm{bei}\\\\,x-\\\\frac{i}{\\\\pi}(\\\\mathrm{ker}\\\\,x+i\\\\,\\\\mathrm{kei}\\\\,x)=g(x)(1+\\\\epsilon_2)\\ng(x)=\\\\frac{1}{\\\\sqrt{2\\\\pi x}}\\\\exp\\\\left[\\\\frac{1+i}{\\\\sqrt2}x+\\\\theta(x)\\\\right]\\n|\\\\epsilon_2|<3\\\\times 10^{-7}\\nwhere\\n9.11.11  \\\\theta(x)=(.00000\\\\,00-.39269\\\\,91i)+(.01104\\\\,86-.01104\\\\,85i)(8/x)+(.00000\\\\,00-.00097\\\\,65i)(8/x)^2+(-.00009\\\\,06-.00009\\\\,01i)(8/x)^3+(-.00002\\\\,52+.00000\\\\,00i)(8/x)^4+(-.00000\\\\,34+.00000\\\\,51i)(8/x)^5+(.00000\\\\,06+.00000\\\\,19i)(8/x)^6\\n9.11.12  8\\\\le x<\\\\infty\\n\\\\mathrm{ker}'\\\\,x+i\\\\,\\\\mathrm{kei}'\\\\,x=-f(x)\\\\phi(-x)(1+\\\\epsilon_3)\\n|\\\\epsilon_3|<2\\\\times 10^{-7}\\n9.11.13  8\\\\le x<\\\\infty\\n\\\\mathrm{ber}'\\\\,x+i\\\\,\\\\mathrm{bei}'\\\\,x-\\\\frac{i}{\\\\pi}(\\\\mathrm{ker}'\\\\,x+i\\\\,\\\\mathrm{kei}'\\\\,x)=g(x)\\\\phi(x)(1+\\\\epsilon_4)\\n|\\\\epsilon_4|<3\\\\times 10^{-7}\\nwhere\\n9.11.14  \\\\phi(x)=(.70710\\\\,68+.70710\\\\,68i)+(-.06250\\\\,01-.00000\\\\,01i)(8/x)+(-.00138\\\\,13+.00138\\\\,11i)(8/x)^2+(.00000\\\\,05+.00024\\\\,52i)(8/x)^3+(.00003\\\\,46+.00003\\\\,38i)(8/x)^4+(.00001\\\\,17-.00000\\\\,24i)(8/x)^5+(.00000\\\\,16-.00000\\\\,32i)(8/x)^6\\nNumerical Methods\\n9.12. Use and Extension of the Tables\\nExample 1. To evaluate J_n(1.55), n=0, 1, 2, \\\\dots, each to 5 decimals.\\nThe recurrence relation\\nJ_{n-1}(x)+J_{n+1}(x)=(2n/x)J_n(x)\\ncan be used to compute J_0(x), J_1(x), J_2(x), \\\\dots, successively provided that n<x, otherwise severe accumulation of rounding errors will occur. Since, however, J_n(x) is a decreasing function of n when n>x, recurrence can always be carried out in the direction of decreasing n.\\nInspection of Table 9.2 shows that J_n(1.55) vanishes to 5 decimals when n>7. Taking arbitrary values zero for J_9 and unity for J_8, we compute by recurrence the entries in the second column of the following table, rounding off to the nearest integer at each step.\\nColumns: n | Trial values | J_n(1.55)\\n9   0        .00000\\n8   1        .00000\\n7   10       .00003\\n6   89       .00028\\n5   679      .00211\\n4   4292     .01331\\n3   21473    .06661\\n2   78829    .24453\\n1   181957   .56442\\n0   155954   .48376\\nWe normalize the results by use of the equation 9.1.46, namely\\nJ_0(x)+2J_2(x)+2J_4(x)+\\\\dots=1\\nThis yields the normalization factor\\n1/322376=.00000\\\\,31019\\\\,7\"}, {\"page_id\": \"as_p0423\", \"printed_page\": 423, \"chapter_id\": \"ch09_bessel\", \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0423.png\", \"note\": \"Dense numeric-table page (Ch.9, Table 9.9, modified Bessel functions orders 3-9) -- seven data columns, the widest table in this batch. Gold = structural headers + sampled rows + the source attribution at the foot, per the as_p0242/as_p0243 convention. Entries use A&S's bracketed power-of-ten notation, e.g. (-4)1.3680 means 1.3680e-4. Draft existed (1585 chars) and was corrected against the scan.\", \"text\": \"BESSEL FUNCTIONS OF INTEGER ORDER (p.423)\\nTable 9.9  MODIFIED BESSEL FUNCTIONS-ORDERS 3-9\\nColumns: x | e^{-x}I_3(x) | e^{-x}I_4(x) | e^{-x}I_5(x) | e^{-x}I_6(x) | e^{-x}I_7(x) | e^{-x}I_8(x) | e^{-x}I_9(x)\\nSample rows:\\n0.0   0.0000       0.0000       0.0000       0.0000       0.0000        0.0000        0.0000\\n0.2   (-4)1.3680   (-6)3.4182   (-8)6.8341   (-8)1.1388   (-11)1.6265   (-13)2.0328   (-15)2.2585\\n1.0   (-3)8.1553   (-3)1.0069   (-5)9.9866   (-6)8.2731   (-7)5.8832    (-8)3.6643    (-9)2.0301\\n2.0   (-2)2.8791   (-3)6.8654   (-3)1.3298   (-4)2.1656   (-5)3.0402    (-6)3.7487    (-7)4.1199\\n3.0   (-2)4.7783   (-2)1.6216   (-3)4.5409   (-3)1.0796   (-4)2.2265    (-5)4.0512    (-6)6.5905\\n4.0   (-2)6.1124   (-2)2.5940   (-3)9.2443   (-3)2.8291   (-4)7.5698    (-4)1.7968    (-5)3.8284\\n5.0   (-2)6.9611   (-2)3.4419   (-2)1.4540   (-3)5.3384   (-3)1.7282    (-4)4.9939    (-4)1.3015\\n6.0   (-2)7.4736   (-2)4.1238   (-2)1.9752   (-3)8.3181   (-3)3.1156    (-3)1.0484    (-4)3.1978\\n7.0   (-2)7.7670   (-2)4.6509   (-2)2.4516   (-2)1.1486   (-3)4.8261    (-3)1.8337    (-4)6.3475\\n8.0   (-2)7.9194   (-2)5.0500   (-2)2.8694   (-2)1.4633   (-3)6.7449    (-3)2.8292    (-3)1.0866\\n9.0   (-2)7.9808   (-2)5.3482   (-2)3.2269   (-2)1.7627   (-3)8.7663    (-3)3.9907    (-3)1.6716\\n10.0  (-2)7.9830   (-2)5.5683   (-2)3.5284   (-2)2.0398   (-2)1.0806    (-3)5.2694    (-3)2.3753\\n12.0  (-2)7.8848   (-2)5.8425   (-2)3.9898   (-2)2.5176   (-2)1.4722    (-3)8.0010    (-3)4.0537\\n15.0  (-2)7.6236   (-2)6.0022   (-2)4.4225   (-2)3.0538   (-2)1.9794    (-2)1.2064    (-3)6.9260\\n17.5  (-2)7.3761   (-2)6.0119   (-2)4.6278   (-2)3.3675   (-2)2.3187    (-2)1.5125    (-3)9.3584\\n20.0  (-2)7.1300   (-2)5.9640   (-2)4.7444   (-2)3.5917   (-2)2.5894    (-2)1.7792    (-2)1.1661\\nSource note (foot of page): Compiled from British Association for the Advancement of Science, Bessel functions, Part II. Functions of positive integer order, Mathematical Tables, vol. X (Cambridge Univ. Press, Cambridge, England, 1952) (with permission).\\n(The full table runs x=0.0(0.2)10.0 then 10.0(0.5)20.0 across the seven columns above; rows were sampled at unit intervals, not transcribed exhaustively.)\"}, {\"page_id\": \"as_p0600\", \"printed_page\": 600, \"chapter_id\": \"ch17_elliptic_integrals\", \"region_type\": \"formula\", \"image\": \"grading_kit/heldout_pages/as_p0600.png\", \"note\": \"Two-column formula page (Ch.17 elliptic integrals), 17.7.15-17.7.25 plus the start of Numerical Methods 17.8 Example 1. Carries Figure 17.11 (plot of the elliptic integral of the third kind) -- caption and axis/curve labels transcribed, the curve itself is not reducible to text. Draft existed (2220 chars) and was corrected against the scan.\", \"text\": \"ELLIPTIC INTEGRALS (p.600)\\nFIGURE 17.11. Elliptic integral of the third kind \\\\Pi(n; \\\\varphi\\\\backslash\\\\alpha).\\n(Curve labels: n=.8,\\\\varphi=90^\\\\circ; n=.7,\\\\varphi=90^\\\\circ; n=0,\\\\varphi=90^\\\\circ; n=1,\\\\varphi=45^\\\\circ; n=0,\\\\varphi=45^\\\\circ; n=0,1,\\\\varphi=15^\\\\circ. Vertical axis \\\\Pi(n;\\\\varphi\\\\backslash\\\\alpha) from .2 to 3.5; horizontal axis \\\\alpha from 0^\\\\circ to 90^\\\\circ.)\\nCase (iv) Circular Case n<0\\nThe case n<0 can be reduced to the case \\\\sin^2\\\\alpha<N<1 by writing\\n17.7.15  N=(\\\\sin^2\\\\alpha-n)(1-n)^{-1}\\n         p_2=[-n(1-n)^{-1}(\\\\sin^2\\\\alpha-n)]^{\\\\frac12}\\n17.7.16  [(1-n)(1-n^{-1}\\\\sin^2\\\\alpha)]^{\\\\frac12}\\\\Pi(n; \\\\varphi\\\\backslash\\\\alpha)=[(1-N)(1-N^{-1}\\\\sin^2\\\\alpha)]^{\\\\frac12}\\\\Pi(N; \\\\varphi\\\\backslash\\\\alpha)+p_2^{-1}\\\\sin^2\\\\alpha F(\\\\varphi\\\\backslash\\\\alpha)+\\\\arctan[\\\\tfrac12 p_2\\\\sin 2\\\\varphi/\\\\Delta(\\\\varphi)]\\n17.7.17  \\\\Pi(n\\\\backslash\\\\alpha)=(-n\\\\cos^2\\\\alpha)(1-n)^{-1}(\\\\sin^2\\\\alpha-n)^{-1}\\\\Pi(N\\\\backslash\\\\alpha)+\\\\sin^2\\\\alpha(\\\\sin^2\\\\alpha-n)^{-1}K(\\\\alpha)\\nSpecial Cases\\n17.7.18  n=0\\n\\\\Pi(0; \\\\varphi\\\\backslash\\\\alpha)=F(\\\\varphi\\\\backslash\\\\alpha)\\n17.7.19  n=0, \\\\alpha=0\\n\\\\Pi(0; \\\\varphi\\\\backslash 0)=\\\\varphi\\n17.7.20  \\\\alpha=0\\n\\\\Pi(n; \\\\varphi\\\\backslash 0)=(1-n)^{-\\\\frac12}\\\\mathrm{arctanh}[(1-n)^{\\\\frac12}\\\\tan\\\\varphi], \\\\quad n<1\\n         =(n-1)^{-\\\\frac12}\\\\arctan[(n-1)^{\\\\frac12}\\\\tan\\\\varphi], \\\\quad n>1\\n         =\\\\tan\\\\varphi \\\\quad n=1\\n17.7.21  \\\\alpha=\\\\pi/2\\n\\\\Pi(n; \\\\varphi\\\\backslash\\\\pi/2)=(1-n)^{-1}[\\\\ln(\\\\tan\\\\varphi+\\\\sec\\\\varphi)-\\\\tfrac12 n^{\\\\frac12}\\\\ln(1+n^{\\\\frac12}\\\\sin\\\\varphi)(1-n^{\\\\frac12}\\\\sin\\\\varphi)^{-1}] \\\\quad n\\\\neq 1\\n17.7.22  n=\\\\pm\\\\sin\\\\alpha\\n(1\\\\mp\\\\sin\\\\alpha)\\\\{2\\\\Pi(\\\\pm\\\\sin\\\\alpha; \\\\varphi\\\\backslash\\\\alpha)-F(\\\\varphi\\\\backslash\\\\alpha)\\\\}=\\\\arctan[(1\\\\mp\\\\sin\\\\alpha)\\\\tan\\\\varphi/\\\\Delta(\\\\varphi)]\\n17.7.23  n=1\\\\pm\\\\cos\\\\alpha\\n2\\\\cos\\\\alpha\\\\Pi(1\\\\pm\\\\cos\\\\alpha; \\\\varphi\\\\backslash\\\\alpha)=\\\\pm\\\\tfrac12\\\\ln[(1+\\\\tan\\\\varphi\\\\cdot\\\\Delta(\\\\varphi))(1-\\\\tan\\\\varphi\\\\cdot\\\\Delta(\\\\varphi))^{-1}]+\\\\tfrac12\\\\ln[(\\\\Delta(\\\\varphi)+\\\\cos\\\\alpha\\\\cdot\\\\tan\\\\varphi)(\\\\Delta(\\\\varphi)-\\\\cos\\\\alpha\\\\tan\\\\varphi)^{-1}]\\\\mp(1\\\\mp\\\\cos\\\\alpha)F(\\\\varphi\\\\backslash\\\\alpha)\\n17.7.24  n=\\\\sin^2\\\\alpha\\n\\\\Pi(\\\\sin^2\\\\alpha; \\\\varphi\\\\backslash\\\\alpha)=\\\\sec^2\\\\alpha E(\\\\varphi\\\\backslash\\\\alpha)-(\\\\tan^2\\\\alpha\\\\sin 2\\\\varphi)/(2\\\\Delta(\\\\varphi))\\n17.7.25  n=1\\n\\\\Pi(1; \\\\varphi\\\\backslash\\\\alpha)=F(\\\\varphi\\\\backslash\\\\alpha)-\\\\sec^2\\\\alpha E(\\\\varphi\\\\backslash\\\\alpha)+\\\\sec^2\\\\alpha\\\\tan\\\\varphi\\\\Delta(\\\\varphi)\\nNumerical Methods\\n17.8. Use and Extension of the Tables\\nExample 1. Reduce to canonical form \\\\int y^{-1}dx, where\\ny^2=-3x^4+34x^3-119x^2+172x-90\\nBy inspection or by solving an equation of the fourth degree we find that\\ny^2=Q_1Q_2 where Q_1=3x^2-10x+9, Q_2=-x^2+8x-10\\nFirst Method\\nQ_1-\\\\lambda Q_2=(3+\\\\lambda)x^2-(10+8\\\\lambda)x+9+10\\\\lambda is a perfect square if the discriminant\\n(10+8\\\\lambda)^2-4(3+\\\\lambda)(9+10\\\\lambda)=0; i.e., if \\\\lambda=-\\\\frac{2}{3} or \\\\frac{1}{2}\\nand then\\nQ_1+\\\\frac{2}{3}Q_2=\\\\frac{7}{3}(x-1)^2, Q_1-\\\\frac{1}{2}Q_2=\\\\frac{7}{2}(x-2)^2\\nSolving for Q_1 and Q_2 we get\\nQ_1=(x-1)^2+2(x-2)^2, Q_2=2(x-1)^2-3(x-2)^2\\nThe substitution t=(x-1)/(x-2) then gives\\n\\\\int y^{-1}dx=\\\\pm\\\\int[(t^2+2)(2t^2-3)]^{-\\\\frac12}dt\"}, {\"page_id\": \"as_p0619\", \"printed_page\": 619, \"chapter_id\": \"ch17_elliptic_integrals\", \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0619.png\", \"note\": \"Dense numeric-table page (Ch.17, Table 17.7, Jacobian zeta function). Gold = structural headers + the two defining relations + sampled rows + the supplementary odd-alpha block and source attribution at the foot, per the as_p0242/as_p0243 convention. Draft existed (3438 chars) and was corrected against the scan.\", \"text\": \"ELLIPTIC INTEGRALS (p.619)\\nTable 17.7  JACOBIAN ZETA FUNCTION Z(\\\\varphi\\\\backslash\\\\alpha)\\nDefining relations:\\nK(\\\\alpha)Z(\\\\varphi\\\\backslash\\\\alpha)=K(\\\\alpha)E(\\\\varphi\\\\backslash\\\\alpha)-E(\\\\alpha)F(\\\\varphi\\\\backslash\\\\alpha)\\nK(90^\\\\circ)Z(\\\\varphi\\\\backslash\\\\alpha)=K(90^\\\\circ)Z(u|1)=K(90^\\\\circ)\\\\tanh u=\\\\infty for all u\\nColumns: \\\\alpha\\\\backslash\\\\varphi | 0^\\\\circ | 5^\\\\circ | 10^\\\\circ | 15^\\\\circ | 20^\\\\circ | 25^\\\\circ | 30^\\\\circ\\nSample rows:\\n0   0  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000\\n10  0  0.002080  0.004098  0.005992  0.007706  0.009188  0.010393\\n20  0  0.008357  0.016470  0.024105  0.031035  0.037055  0.041981\\n30  0  0.018962  0.037403  0.054811  0.070696  0.084599  0.096103\\n40  0  0.034205  0.067540  0.099145  0.128185  0.153860  0.175418\\n50  0  0.054771  0.108280  0.159273  0.206513  0.248789  0.284929\\n60  0  0.082227  0.162776  0.239971  0.312138  0.377610  0.434726\\n70  0  0.120612  0.239097  0.353322  0.461145  0.560402  0.648900\\n80  0  0.183967  0.365230  0.541075  0.708771  0.865556  1.008608\\n88  0  0.325753  0.647691  0.962000  1.264856  1.552420  1.820811\\n90  \\\\infty  \\\\infty  \\\\infty  \\\\infty  \\\\infty  \\\\infty  \\\\infty\\nSupplementary block for odd \\\\alpha (printed below the main table):\\n5   0  0.000519  0.001023  0.001496  0.001923  0.002292  0.002592\\n25  0  0.013105  0.025838  0.037836  0.048754  0.058271  0.066098\\n45  0  0.043755  0.086448  0.127026  0.164459  0.197748  0.225942\\n65  0  0.099601  0.197305  0.291216  0.379430  0.460039  0.531121\\n85  0  0.245478  0.487761  0.723644  0.949910  1.163313  1.360551\\nSee Example 16.\\nSource note (foot of page): Compiled from P. F. Byrd and M. D. Friedman, Handbook of elliptic integrals for engineers and physicists, Springer-Verlag, Berlin, Germany, 1954 (with permission).\\n(The main table runs \\\\alpha=0(2)90 in 46 rows across the seven columns above, followed by a 9-row block for \\\\alpha=5(10)85; rows were sampled at 10-degree intervals, not transcribed exhaustively.)\"}, {\"page_id\": \"as_p0621\", \"printed_page\": 621, \"chapter_id\": \"ch17_elliptic_integrals\", \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0621.png\", \"note\": \"Dense numeric-table page (Ch.17, Table 17.7 continued -- the phi=65..90 degree half of the Jacobian zeta function, continuing directly from as_p0619's phi=0..30 columns). Gold = structural headers + the two defining relations + sampled rows + the supplementary odd-alpha block, per the as_p0242/as_p0243 convention. Draft existed (2474 chars) and was corrected against the scan.\", \"text\": \"ELLIPTIC INTEGRALS (p.621)\\nTable 17.7  JACOBIAN ZETA FUNCTION Z(\\\\varphi\\\\backslash\\\\alpha)\\nDefining relations:\\nK(\\\\alpha)Z(\\\\varphi\\\\backslash\\\\alpha)=K(\\\\alpha)E(\\\\varphi\\\\backslash\\\\alpha)-E(\\\\alpha)F(\\\\varphi\\\\backslash\\\\alpha)\\nK(90^\\\\circ)Z(\\\\varphi\\\\backslash\\\\alpha)=K(90^\\\\circ)Z(u|1)=K(90^\\\\circ)\\\\tanh u=\\\\infty for all u\\nColumns: \\\\alpha\\\\backslash\\\\varphi | 65^\\\\circ | 70^\\\\circ | 75^\\\\circ | 80^\\\\circ | 85^\\\\circ | 90^\\\\circ\\nSample rows:\\n0   0.000000  0.000000  0.000000  0.000000  0.000000  0\\n10  0.009233  0.007751  0.006032  0.004127  0.002096  0\\n20  0.037803  0.031783  0.024763  0.016959  0.008617  0\\n30  0.088594  0.074696  0.058332  0.040018  0.020354  0\\n40  0.167527  0.141905  0.111254  0.076554  0.039011  0\\n50  0.286045  0.244154  0.192704  0.133299  0.068157  0\\n60  0.467411  0.404143  0.322854  0.225584  0.116121  0\\n70  0.765385  0.677086  0.554038  0.395917  0.207230  0\\n80  1.345674  1.240571  1.069839  0.814374  0.453784  0\\n88  2.790834  2.721008  2.555104  2.241393  1.628299  0\\n90  \\\\infty  \\\\infty  \\\\infty  \\\\infty  \\\\infty  \\\\infty\\nSupplementary block for odd \\\\alpha (printed below the main table):\\n5   0.002295  0.001926  0.001498  0.001025  0.000520  0\\n25  0.060141  0.050625  0.039483  0.027060  0.013755  0\\n45  0.220781  0.187640  0.147536  0.101748  0.051923  0\\n65  0.596098  0.520463  0.419877  0.295957  0.153297  0\\n85  1.962673  1.866624  1.686113  1.380465  0.860811  0\\n(The main table runs \\\\alpha=0(2)90 in 46 rows across the six columns above, followed by a 9-row block for \\\\alpha=5(10)85; rows were sampled at 10-degree intervals, not transcribed exhaustively.)\"}, {\"page_id\": \"as_p0622\", \"printed_page\": 622, \"chapter_id\": \"ch17_elliptic_integrals\", \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0622.png\", \"note\": \"BLANK-SLATE page: the repaired reader produced empty output, so there was no draft to correct -- transcribed directly from the scan. Dense numeric-table page (Ch.17, Table 17.8, Heuman's lambda function). Gold = structural headers + the defining relation + sampled rows + the interpolation-difference scale factors and source attribution at the foot, per the as_p0242/as_p0243 convention.\", \"text\": \"ELLIPTIC INTEGRALS (p.622)\\nTable 17.8  HEUMAN'S LAMBDA FUNCTION \\\\Lambda_0(\\\\varphi\\\\backslash\\\\alpha)\\nDefining relation:\\n\\\\Lambda_0(\\\\varphi\\\\backslash\\\\alpha)=\\\\frac{F(\\\\varphi\\\\backslash 90^\\\\circ-\\\\alpha)}{K'(\\\\alpha)}+\\\\frac{2}{\\\\pi}K(\\\\alpha)Z(\\\\varphi\\\\backslash 90^\\\\circ-\\\\alpha)=\\\\frac{2}{\\\\pi}\\\\{K(\\\\alpha)E(\\\\varphi\\\\backslash 90^\\\\circ-\\\\alpha)-[K(\\\\alpha)-E(\\\\alpha)]F(\\\\varphi\\\\backslash 90^\\\\circ-\\\\alpha)\\\\}\\nColumns: \\\\alpha\\\\backslash\\\\varphi | 0^\\\\circ | 5^\\\\circ | 10^\\\\circ | 15^\\\\circ | 20^\\\\circ | 25^\\\\circ | 30^\\\\circ\\nSample rows:\\n0   0  0.087156  0.173648  0.258819  0.342020  0.422618  0.500000\\n10  0  0.086495  0.172332  0.256858  0.339430  0.419419  0.496219\\n20  0  0.084549  0.168458  0.251092  0.331827  0.410054  0.485184\\n30  0  0.081425  0.162247  0.241870  0.319707  0.395191  0.467777\\n40  0  0.077307  0.154073  0.229767  0.303869  0.375880  0.445330\\n50  0  0.072455  0.144464  0.215587  0.285399  0.353500  0.419519\\n60  0  0.067226  0.134126  0.200380  0.265684  0.329751  0.392328\\n70  0  0.062100  0.124009  0.185540  0.246517  0.306778  0.366180\\n80  0  0.057773  0.115479  0.173054  0.230436  0.287571  0.344410\\n88  0  0.055698  0.111392  0.167078  0.222751  0.278408  0.334046\\n90  0  0.055556  0.111111  0.166667  0.222222  0.277778  0.333333\\nInterpolation-difference scale factors printed under the last row: [(-5)2/5], [(-5)5/5], [(-5)7/6], [(-5)9/6], [(-4)1/6], [(-4)1/6]\\nSupplementary block for odd \\\\alpha (printed below the main table):\\n5   0  0.086990  0.173318  0.258327  0.341370  0.421815  0.499050\\n25  0  0.083124  0.165625  0.246882  0.326288  0.403252  0.477203\\n45  0  0.074953  0.149408  0.222878  0.294884  0.364976  0.432729\\n65  0  0.064614  0.128968  0.192809  0.255897  0.318009  0.378946\\n85  0  0.056256  0.112490  0.168682  0.224814  0.280867  0.336826\\nSource note (foot of page): Compiled from C. Heuman, Tables of complete elliptic integrals, J. Math. Phys. 20, 127-206, 1941 (with permission).\\n(The main table runs \\\\alpha=0(2)90 in 46 rows across the seven columns above, followed by a 9-row block for \\\\alpha=5(10)85; rows were sampled at 10-degree intervals, not transcribed exhaustively.)\"}, {\"page_id\": \"as_p0625\", \"printed_page\": 625, \"chapter_id\": \"ch17_elliptic_integrals\", \"region_type\": \"table\", \"image\": \"grading_kit/heldout_pages/as_p0625.png\", \"note\": \"Dense numeric-table page (Ch.17, Table 17.9, elliptic integral of the third kind) -- a doubly-indexed table (n and alpha both vary down the page). Gold = structural headers + the defining integral + sampled rows covering each n block + the interpolation-difference scale factors at the foot, per the as_p0242/as_p0243 convention. Draft existed (3474 chars) and was corrected against the scan.\", \"text\": \"ELLIPTIC INTEGRALS (p.625)\\nTable 17.9  ELLIPTIC INTEGRAL OF THE THIRD KIND \\\\Pi(n; \\\\varphi\\\\backslash\\\\alpha)\\nDefining integral:\\n\\\\Pi(n; \\\\varphi\\\\backslash\\\\alpha)=\\\\int_0^\\\\varphi(1-n\\\\sin^2\\\\theta)^{-1}[1-\\\\sin^2\\\\alpha\\\\sin^2\\\\theta]^{-\\\\frac12}d\\\\theta\\nColumns: n | \\\\alpha\\\\backslash\\\\varphi | 0^\\\\circ | 15^\\\\circ | 30^\\\\circ | 45^\\\\circ | 60^\\\\circ | 75^\\\\circ | 90^\\\\circ\\nSample rows (one per n block, plus the \\\\alpha=90 terminator of each):\\n0.0  0   0  0.26180  0.52360  0.78540  1.04720  1.30900  1.57080\\n0.0  45  0  0.26330  0.53562  0.82602  1.14243  1.48788  1.85407\\n0.0  90  0  0.26484  0.54931  0.88137  1.31696  2.02759  \\\\infty\\n0.1  0   0  0.26239  0.52820  0.80013  1.07949  1.36560  1.65576\\n0.1  90  0  0.26545  0.55431  0.89939  1.36454  2.14201  \\\\infty\\n0.2  0   0  0.26299  0.53294  0.81586  1.11534  1.43078  1.75620\\n0.2  90  0  0.26606  0.55948  0.91867  1.41777  2.27604  \\\\infty\\n0.3  0   0  0.26359  0.53784  0.83271  1.15551  1.50701  1.87746\\n0.3  90  0  0.26667  0.56483  0.93938  1.47789  2.43581  \\\\infty\\n0.4  0   0  0.26420  0.54291  0.85084  1.20098  1.59794  2.02789\\n0.4  90  0  0.26729  0.57035  0.96171  1.54653  2.63052  \\\\infty\\n0.5  0   0  0.26481  0.54814  0.87042  1.25310  1.70919  2.22144\\n0.5  90  0  0.26792  0.57606  0.98591  1.62599  2.87468  \\\\infty\\n0.6  0   0  0.26543  0.55357  0.89167  1.31379  1.85002  2.48365\\n0.6  90  0  0.26855  0.58198  1.01225  1.71951  3.19278  \\\\infty\\nInterpolation-difference scale factors printed under the last row: [(-5)5/4], [(-4)4/6], [(-3)2/7], [(-3)7/7]\\nSee Examples 15-20.\\n(The full table steps n=0.0(0.1)0.6 with \\\\alpha=0(15)90 inside each n block, 49 rows in all across the seven columns above; rows were sampled at the head and tail of each n block, not transcribed exhaustively.)\"}]"

labels = json.loads(LABELS_JSON)
print(f"loaded {len(labels)} gold TEST-page labels")
assert len(labels) == 39, f"expected 39 gold pages, got {len(labels)}"


def _pick_device() -> str:
    """See plan_a3.md Step 3's RESULT: torch.cuda.is_available() alone isn't enough --
    a GPU can be present but incompatible with the installed PyTorch build. Try a real op."""
    if not torch.cuda.is_available():
        return "cpu"
    try:
        _p = torch.zeros(1, device="cuda")
        _ = _p + 1
        return "cuda"
    except RuntimeError as exc:
        print(f"CUDA reports available but a real op failed ({exc}) -- using CPU")
        return "cpu"


device = _pick_device()
print(f"device: {device}")

# --- Find the mounted index dataset -- recursive, no assumed mount depth. Step 3 learned
# the hard way that /kaggle/input/<slug>/ is NOT a safe assumption on this Kaggle layout
# (real one seen: /kaggle/input/datasets/<user>/<slug>/).
print("locating mounted index dataset...")
faiss_files = sorted(Path("/kaggle/input").rglob("faiss.index"))
if not faiss_files:
    zips = sorted(Path("/kaggle/input").rglob("*.zip"))
    assert zips, "no faiss.index and no .zip found anywhere under /kaggle/input/"
    extract_dir = Path("/kaggle/working/index_extracted")
    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f"extracting {zips[0]} to {extract_dir}")
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(extract_dir)
    faiss_files = sorted(extract_dir.rglob("faiss.index"))
assert faiss_files, "still no faiss.index found after checking for a zip"
index_dir = faiss_files[0].parent
print(f"index dir: {index_dir}")
for p in sorted(index_dir.iterdir()):
    print(" ", p.name, p.stat().st_size)

# --- Load the index + chunk sidecar (mirrors index/store.py's own load(), standalone here
# since this notebook doesn't clone the repo). faiss isn't on Kaggle's base image by
# default. Deliberately NOT this repo's own pin (faiss-cpu>=1.8,<1.9) -- that range
# predates numpy 2.x support and its prebuilt extension is compiled against numpy 1.x's
# C-ABI (exactly the reason pyproject.toml pins numpy<2 locally). Kaggle's base image
# already has numpy 2.0.2 pre-installed. Two real failures here, not one: first,
# --no-deps was needed because a plain install let pip downgrade that numpy to satisfy
# the old faiss-cpu pin, which broke scipy (already imported, expecting numpy 2.x) --
# ModuleNotFoundError: No module named 'numpy.strings'. Then, even with --no-deps kept,
# the OLD faiss-cpu wheel itself crashed against numpy 2.0.2 at import
# ("compiled using NumPy 1.x cannot be run in NumPy 2.0.2"). Fix: let pip pick whatever
# current faiss-cpu it wants (no upper bound), which supports numpy 2.x, so nothing
# needs downgrading in either direction. This version constraint is deliberately looser
# here than the repo's own pin -- this notebook never touches the real index-build
# pipeline pyproject.toml governs, it only reads an already-built index back.
import subprocess  # noqa: E402
import sys  # noqa: E402

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], check=True)
import faiss  # noqa: E402

faiss_index = faiss.read_index(str(index_dir / "faiss.index"))
chunks = []
with open(index_dir / "chunks.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))
meta = json.loads((index_dir / "index_meta.json").read_text(encoding="utf-8"))
print(f"index: {faiss_index.ntotal} vectors, {len(chunks)} chunks, meta={meta}")
assert faiss_index.ntotal == len(chunks), "index/sidecar count mismatch"

# --- Models: BGE-M3 for dense retrieval, bge-reranker-v2-m3 for the rerank arm.
from sentence_transformers import CrossEncoder, SentenceTransformer  # noqa: E402

t0 = time.time()
encoder = SentenceTransformer(meta.get("embed_model", "BAAI/bge-m3"), device=device)
print(f"encoder loaded in {time.time()-t0:.1f}s")
t0 = time.time()
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device)
print(f"reranker loaded in {time.time()-t0:.1f}s")

K_VALUES = [1, 5, 10]
K_CANDIDATES = 40  # matches cfg.retrieve.k_max -- widest net either arm ever needs


def _query_for(label: dict) -> str:
    """One query per gold page, formed from the page's own content -- its first
    non-empty line, deterministic and reproducible, no cherry-picking."""
    first_line = next(ln.strip() for ln in label["text"].splitlines() if ln.strip())
    return first_line


results_dense = {k: 0 for k in K_VALUES}
results_reranked = {k: 0 for k in K_VALUES}
per_page_rows = []

for label in labels:
    query = _query_for(label)
    target_page = label["page_id"]

    qvec = encoder.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)
    scores, ids = faiss_index.search(qvec, K_CANDIDATES)
    candidates = [chunks[i] for i in ids[0] if i >= 0]

    dense_ranks = [i for i, c in enumerate(candidates, start=1) if target_page in c["page_ids"]]
    dense_rank = dense_ranks[0] if dense_ranks else None

    pairs = [(query, c["text"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)
    reranked = [
        c for _, c in sorted(zip(rerank_scores, candidates, strict=True), key=lambda x: -x[0])
    ]
    reranked_ranks = [
        i for i, c in enumerate(reranked, start=1) if target_page in c["page_ids"]
    ]
    reranked_rank = reranked_ranks[0] if reranked_ranks else None

    for k in K_VALUES:
        if dense_rank is not None and dense_rank <= k:
            results_dense[k] += 1
        if reranked_rank is not None and reranked_rank <= k:
            results_reranked[k] += 1

    per_page_rows.append(
        {
            "page_id": target_page,
            "query": query,
            "dense_rank": dense_rank,
            "reranked_rank": reranked_rank,
        }
    )
    print(f"{target_page}: dense_rank={dense_rank}  reranked_rank={reranked_rank}  query={query!r}")

n = len(labels)
print("\n=== recall@k over 39 gold TEST pages ===")
print(f"{'k':>4} | {'dense':>8} | {'reranked':>8}")
for k in K_VALUES:
    print(f"{k:>4} | {results_dense[k]/n:>8.3f} | {results_reranked[k]/n:>8.3f}")

report = {
    "n_queries": n,
    "recall_at_k_dense": {str(k): results_dense[k] / n for k in K_VALUES},
    "recall_at_k_reranked": {str(k): results_reranked[k] / n for k in K_VALUES},
    "raw_counts_dense": results_dense,
    "raw_counts_reranked": results_reranked,
    "per_page": per_page_rows,
}
out_path = Path("/kaggle/working/a3_retrieval_probe.json")
out_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"\nsaved {out_path}")


loaded 39 gold TEST-page labels
device: cuda
locating mounted index dataset...
index dir: /kaggle/input/datasets/anuronmaitro/mathscholar-index
  chunks.jsonl 1711864
  embed_cache.npz 13459580
  faiss.index 14512173
  image_embed_cache.npz 2163716
  index_meta.json 230
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.0 MB/s eta 0:00:00
index: 3543 vectors, 3543 chunks, meta={'n_chunks': 3543, 'embedding_dim': 1024, 'index_type': 'faiss:flat', 'index_size_bytes': 16224037, 'pages_covered': 987, 'chapters_covered': 29, 'embed_model': 'BAAI/bge-m3', 'built_at': '2026-08-14T11:01:35Z'}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

reranker loaded in 25.0s
as_p0243: dense_rank=None  reranked_rank=None  query='EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.243)'
as_p0255: dense_rank=1  reranked_rank=1  query='6. Gamma Function and Related Functions. Mathematical Properties.'
as_p0360: dense_rank=None  reranked_rank=None  query='BESSEL FUNCTIONS OF INTEGER ORDER (p.360)'
as_p0229: dense_rank=4  reranked_rank=1  query='EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.229)'
as_p0230: dense_rank=3  reranked_rank=1  query='EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.230)'
as_p0232: dense_rank=1  reranked_rank=1  query='EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.232)'
as_p0234: dense_rank=None  reranked_rank=None  query='EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.234)'
as_p0242: dense_rank=None  reranked_rank=None  query='EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.242)'
as_p0247: dense_rank=24  reranked_rank=10  query='EXPONENTIAL INTEGRAL AND RELATED FUNCTIONS (p.247)'
as_p0256: dense_rank=36  reranked_rank=3  q